In [1]:
!pip install -q langchain langchain-anthropic langchain-google-genai langchain-community faiss-cpu python-docx pydantic pymupdf
# =====================================================
# IMPORTS
# =====================================================
import os
import re
import json
import time
import unicodedata
import hashlib
from collections import Counter, defaultdict
from typing import List, Optional

import pandas as pd
import fitz
from docx import Document
from pydantic import BaseModel, Field

from langchain_anthropic import ChatAnthropic
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document as LCDocument

from google.colab import drive, userdata
drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_1298/1379595441.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Mounted at /content/drive


In [2]:
# =====================================================
# CONFIGURAÇÕES
# =====================================================
BASE_PATH = "/content/drive/MyDrive/Scripts/Analise de municipio"

ARQUIVO_PLANILHA = f"{BASE_PATH}/indicadores.xlsx"

COLUNA_MUNICIPIO = "municipio"
COLUNA_POPULACAO = "populacao total estimada do municipio"

# Recomendado: manter as chaves no Secrets do Colab.
# ANTHROPIC_API_KEY: geração, revisão e Web Search do Claude.
# API: chave do Gemini mantida SOMENTE para embeddings/FAISS.
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
GEMINI_API_KEY = userdata.get("API")

# Modelo principal: geração do diagnóstico e revisão final.
MODELO = "claude-sonnet-4-6"

llm = ChatAnthropic(
    model=MODELO,
    api_key=ANTHROPIC_API_KEY,
    max_tokens=4096,
)

# Claude Web Search — usado SOMENTE DEPOIS da geração do rascunho,
# para validação pontual de cada dimensão. A planilha continua sendo a
# fonte principal do diagnóstico e dos níveis de maturidade; a web não
# recalcula nem substitui esses indicadores.
MODELO_PESQUISA_WEB = MODELO
llm_pesquisa_base = ChatAnthropic(
    model=MODELO_PESQUISA_WEB,
    api_key=ANTHROPIC_API_KEY,
    max_tokens=4096,
)

# Até 5 pesquisas do servidor por dimensão. O prompt limita a uma consulta
# específica por ponto, permitindo validar no máximo 5 pontos prioritários.
llm_pesquisa_web = llm_pesquisa_base.bind_tools([
    {"type": "web_search_20250305", "name": "web_search", "max_uses": 5}
])

# Evita rajadas sucessivas de chamadas entre dimensões.
PAUSA_ENTRE_PESQUISAS_WEB = 10

# ---- Embeddings (para classificação semântica de indicadores e RAG sobre os chunks da Carta) ----
# O Gemini permanece apenas aqui. Os índices FAISS existentes continuam
# compatíveis enquanto o mesmo modelo de embedding for mantido.
MODELO_EMBEDDING = "models/gemini-embedding-001"

embeddings = GoogleGenerativeAIEmbeddings(
    model=MODELO_EMBEDDING,
    google_api_key=GEMINI_API_KEY,
)

# Índices FAISS são persistidos no Drive: gerados uma única vez e reaproveitados
# nas próximas execuções (evita reprocessar embeddings sempre).
FAISS_INDEX_TOPICOS_PATH = f"{BASE_PATH}/faiss_index_topicos"
FAISS_INDEX_CHUNKS_PATH = f"{BASE_PATH}/faiss_index_chunks"

# NOTA: a leitura da Carta original (VersoResumidadaCarta.docx via
# ler_carta_como_base_conceitual) foi substituída pelos chunks temáticos
# pré-extraídos da Carta Brasileira para Cidades Inteligentes (ver célula
# "CHUNKS TEMÁTICOS DA CARTA" abaixo). Não é mais necessário CAMINHO_CARTA.


# =====================================================
# RAG DA CARTA COMPLETA — ÍNDICE SEPARADO DO ÍNDICE ANTIGO
# =====================================================
# "completa": usa o novo FAISS granular.
# "antigo": volta exatamente à lógica antiga dos 20 chunks.
MODO_RAG_CARTA = "completa"

FAISS_CARTA_COMPLETA_PATH = f"{BASE_PATH}/faiss_carta_completa_v1"
FAISS_CARTA_COMPLETA_MANIFEST_PATH = f"{BASE_PATH}/faiss_carta_completa_v1_manifest.json"

# Opcional para reprocessar a Edição Revisada diretamente no Colab no futuro.
ARQUIVO_CARTA_COMPLETA = f"{BASE_PATH}/carta_brasileira_cidades_inteligentes_edicao_revisada.pdf"
REPROCESSAR_CARTA_DO_PDF = False

MAX_INDICADORES_RAG_POR_DIMENSAO = 5
K_CANDIDATOS_RAG_POR_INDICADOR = 8
MAX_CHUNKS_CARTA_COMPLETA = 8
PESO_MAXIMO_MATURIDADE_RAG = 0.30
BONUS_DIMENSAO_RAG = 0.05
BONUS_TOPICO_RAG = 0.04
BONUS_COBERTURA_MULTIPLA_RAG = 0.02

# Não herdar o 0.75 do índice de tópicos. Calibrar empiricamente.
LIMIAR_SEMANTICO_CARTA_COMPLETA = None
AUDITAR_RAG_CARTA_COMPLETA = True

# Poucas âncoras críticas, se desejado.
ANCORAS_DETERMINISTICAS_CARTA = {}
VERSAO_PIPELINE_CARTA_COMPLETA = "carta-revisada-posicional-v1"


In [3]:
# =====================================================
# FUNÇÕES AUXILIARES
# =====================================================
def classificar_porte(populacao):
    if populacao <= 50000:
        return "pequeno porte"
    elif populacao <= 300000:
        return "médio porte"
    else:
        return "grande porte"


def _normalizar(texto):
    """Remove acentos e caixa para facilitar comparação de palavras-chave."""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = texto.encode("ascii", errors="ignore").decode("utf-8")
    return texto.lower()


In [4]:
# =====================================================
# CHUNKS TEMÁTICOS DA CARTA (pré-extraídos do PDF oficial)
# =====================================================
# Os chunks abaixo foram extraídos e classificados a partir do texto real
# da "Carta Brasileira para Cidades Inteligentes" (versão oficial em PDF,
# 180 páginas, seção "2.5 Objetivos estratégicos e recomendações").
#
# Processo de extração (feito uma única vez, fora deste notebook):
# 1. Texto extraído com pdftotext -layout, isolando a coluna de corpo de
#    texto e descartando a legenda lateral de atores (GF, GE, GM, ...).
# 2. A edição revisada possui 163 recomendações numeradas. O conjunto
#    antigo abaixo permanece congelado por compatibilidade e teste A/B.
#    Ele NÃO é sobrescrito pelo novo índice granular da Carta completa.
# 3. Cada recomendação foi classificada, por palavras-chave de título e
#    corpo, no tópico de maior aderência dentro da estrutura de dimensões
#    abaixo. Recomendações sobre o mesmo tema foram reunidas em um único
#    chunk (sem depender da posição no documento), respeitando ~250–500
#    tokens por chunk e sem sobreposição (overlap) entre chunks.
#
# IMPORTANTE: nem todos os 30 tópicos planejados têm conteúdo dedicado na
# Carta (ela é organizada por 8 objetivos transversais de transformação
# digital, não por setores como saúde, transporte ou saneamento). Os 10
# tópicos abaixo ficaram sem chunk correspondente e são tratados via
# fallback no prompt (ver função gerar_analise_dimensao):
TOPICOS_SEM_CHUNK_NA_CARTA = [
    "economica_transporte",
    "economica_vias_publicas",
    "sociocultural_cultura_e_esporte",
    "sociocultural_saude",
    "sociocultural_seguranca_publica",
    "sociocultural_defesa_civil",
    "meio_ambiente_agua_e_saneamento",
    "meio_ambiente_residuos_solidos",
    "meio_ambiente_areas_verdes",
    "institucional_servicos_publicos_digitais",
]

_CHUNKS_JSON = r"""{"geral_conceito_brasileiro_de_cidades_inteligentes": {"chunk_id": "geral_conceito_brasileiro_de_cidades_inteligentes", "dimensao": "Geral", "topico": "Conceito Brasileiro de Cidades Inteligentes", "texto": "Medidas para o alcance da visão de futuro: Elaborar ou revisar normas, políticas, programas e estratégias para adequá-los à visão de futuro da cidade, conforme estabelecido nos instrumentos de planejamento municipal (exemplos: Plano Diretor PD, Plano Plurianual PPA, Lei de Diretrizes Orçamentárias LDO, Lei Orçamentária Anual LOA). Essa adequação irá garantir que os projetos urbanos, inclusive iniciativas de cidades inteligentes, contribuam para realizar a visão de futuro. [Ver recomendação 1.2.4]\n\nPlanejamento para “cidades inteligentes”: Considerar as determinações do Plano Diretor (ver Estatuto da Cidade) ao elaborar estratégias e planos municipais para a transformação digital. Da mesma forma, considerar as determinações do Plano de Desenvolvimento Urbano Integrado (Estatuto da Metrópole), caso exista. Alinhar o planejamento para “cidades inteligentes” com as recomendações desta Carta e seus desdobramentos em termos de normas, diretrizes e padrões. Exemplos de planos municipais para a transformação digital: Plano Diretor de Cidades Inteligentes e Plano Diretor de Tecnologias de Informação e Comunicação–TICs. 2.5.5. Conectividade digital e integração de equipamentos públicos: Fortalecer iniciativas que integrem instituições e equipamentos públicos de ensino e pesquisa. Para isso, formar parcerias entre instituições de modo a prover redes de infraestrutura digital. Ampliar o modelo de Redes Comunitárias de Ensino e Pesquisa para instituições e equipamentos públicos que atendam outras finalidades.\n\nLaboratórios de experimentação urbana: Incentivar o surgimento de soluções urbanas inovadoras, criando espaços colaborativos transdisciplinares (que possibilitam a cooperação entre diferentes disciplinas e saberes) para cidades inteligentes. Essas ações devem considerar a visão ampla da transformação digital nas cidades. Para garantir que as soluções sejam realizáveis, deve-se focar em pesquisa e experimentação em ambientes reais. Para isso, articular instituições de ensino e pesquisa e outros setores envolvidos na produção de conhecimento, com apoio institucional e jurídico da Administração Pública Municipal. Integrar esses Laboratórios ao Observatório da Transformação Digital nas cidades e a outros fóruns oficiais relacionados à transformação digital [ver recomendação 8.2].", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "geral_diversidade_territorial_e_reducao_de_desigualdades": {"chunk_id": "geral_diversidade_territorial_e_reducao_de_desigualdades", "dimensao": "Geral", "topico": "Diversidade Territorial e Redução de Desigualdades", "texto": "Tipologias urbanas: Estabelecer tipologias (categorias) de território que apoiem a compreensão do urbano no Brasil. Esse trabalho deve ser feito no processo de formulação da Política Nacional de Desenvolvimento Urbano (PNDU). Deve compreender o território a partir de diferentes níveis: municipal, supramunicipal (agrupamento de municípios) e regional. As tipologias também devem se adequar à diversidade territorial do país. O objetivo é orientar agendas, programas e iniciativas para o desenvolvimento urbano sustentável, inclusive de cidades inteligentes, nos três níveis (municipal, supramunicipal e regional). 1.2.2. Instrumentos e metodologias para a diversidade territorial: Desenvolver e adaptar instrumentos e metodologias de informação, planejamento, gestão e governança para o desenvolvimento urbano sustentável, considerando diferentes graus de complexidade. Esses instrumentos e metodologias devem ser adequados às tipologias (categorias de territórios) da Política Nacional de Desenvolvimento (PNDU). Devem considerar a diversidade territorial das cidades brasileiras. Devem ser fáceis de implementar, considerando diferentes capacidades presentes no nível local.\n\nVisão de futuro da cidade: Construir a visão de futuro da cidade de forma participativa e inclusiva. Estabelecer essa visão em instrumentos de planejamento municipal (exemplos: Plano Diretor PD, Plano Plurianual PPA, Lei de Diretrizes Orçamentárias LDO, Lei Orçamentária Anual LOA). Na construção da visão de futuro, considerar a perspectiva e os impactos específicos da transformação digital no território da cidade. Considerar também o contexto regional e as características locais nos aspectos econômico-financeiro, sociocultural, urbano-ambiental e político-institucional. Refletir a visão em metas, com etapas, atividades e prazos associados.\n\nArticulação setorial no território: Desenvolver estratégias para que as políticas, planos e programas de desenvolvimento urbano e de setores afins sejam integradas no território, em todos os níveis de governo. As estratégias devem enfatizar as áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs).", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "geral_transformacao_digital_adaptada_a_capacidade_municipal": {"chunk_id": "geral_transformacao_digital_adaptada_a_capacidade_municipal", "dimensao": "Geral", "topico": "Transformação Digital Adaptada à Capacidade Municipal", "texto": "Diálogo com órgãos de controle: Estabelecer fóruns regulares de diálogo entre: (1) instituições públicas que formulam e implementam políticas públicas; (2) órgãos de controle dos poderes executivo, legislativo e judiciário; (3) Ministério Público; (4) setores envolvidos; (5) organizações da sociedade civil. Esses fóruns devem ter caráter estratégico na tarefa de construir conjuntamente caminhos e suporte à tomada de decisões sobre a transformação digital nas cidades. O objetivo é assegurar a boa condução das políticas sobre o tema da transformação digital nas cidades, em todos os níveis de governo.\n\nServiços urbanos disruptivos: Estruturar espaços de gestão e governança e usar metodologias ágeis para garantir: (1) a tomada de decisão informada por evidências; e (2) a regulação de soluções urbanas em momento adequado. Exemplos de soluções que demandam essas ações: soluções que usam mecanismos ou tecnologias disruptivas (que causam ruptura com padrões e modelos existentes); soluções que geram bases de dados com informações pessoais ou de interesse público; e soluções que usam ou interferem em espaços públicos urbanos (calçadas, praças, sistema viário, soluções de transporte motorizado ou não motorizado, serviços de entrega) etc. entar o desenvolvimento econômico local no contexto da transformadigital Governo Governo Governo Cooperação Cooperação Federal Estadual Municipal Intragovernamental Intragovernamental Vertical Horizontal Agência Empresas Empresas de Setor Privado Reguladora Concecionárias de Telecomunicações Serviços Públicos Instituições de Instituições Organizações da Ensino Financeiras Sociedade Civil e Pesquisa de Fomento OMENDAÇÕES:\n\nLinhas de pesquisa: Incentivar linhas de pesquisa e bolsas de fomento que favoreçam projetos transdisciplinares. O objetivo é produzir conhecimento científico de ponta e de forma contínua sobre a transformação digital nas cidades e seus impactos. 8.5.2. “Ciberinfraestrutura” para geração de conhecimento sobre desenvolvimento urbano sustentável: Apoiar projetos de pesquisa, desenvolvimento e inovação que precisem de “ciberinfraestrutura” (infraestrutura de sistemas operacionais, gestão e processamento de dados, instrumentos avançados e ambientes de visualização) de grande porte. Para tal apoio, devem-se realizar investimentos de longo prazo e articular iniciativas desse tipo de infraestrutura.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_agua_e_esgoto": {"chunk_id": "economica_agua_e_esgoto", "dimensao": "Econômica", "topico": "Água e Esgoto", "texto": "Estratégias setoriais para transformação digital: Elaborar estratégias setoriais para a transformação digital nas cidades, nas áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs). As estratégias devem: (1) ser elaboradas com base em metodologia única que permita sua consolidação em uma estratégia global; (2) ser elaboradas de forma alinhada com esta Carta; (3) ser desenvolvidas pelos respectivos setores, com apoio da Comunidade da Carta. Os objetivos são: (a) identificar, organizar e endereçar demandas específicas de cada setor; e (b) permitir uma visão global que evite sobreposições e otimize esforços no território.\n\nGoverno Digital: Formular e implementar estratégias estaduais e municipais de governo digital que sejam adequadas a cada realidade. O objetivo é tornar a Administração Pública mais acessível e mais eficiente ao prover serviços, como indica a Estratégia de Governo Digital e a Estratégia Brasileira para a Transformação Digital. 3.6.1. Ampliar o acesso a serviços públicos e direitos sociais por meio de TICs: Usar tecnologias de informação e comunicação (TICs) para promover o direito à cidade e para ampliar os direitos sociais. Focar em áreas urbanas com carências de serviços públicos e em pessoas e grupos sociais vulneráveis. Para realizar esses direitos, as TICs devem ajudar a simplificar o acesso a serviços de saúde, educação, moradia, transporte, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), telecomunicações (inclusive serviços de internet), lazer e cultura.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_residuos_solidos": {"chunk_id": "economica_residuos_solidos", "dimensao": "Econômica", "topico": "Resíduos Sólidos", "texto": "Logística reversa de produtos eletrônicos: Acelerar e dar transparência à estruturação e à implementação de sistemas de logística reversa (coletar e devolver resíduos sólidos ao setor empresarial ou descartá-los corretamente). Esses sistemas devem incluir, por exemplo, fábricas, importadoras, distribuidoras e comércios de produtos eletroeletrônicos e seus componentes. As empresas devem oferecer às pessoas consumidoras dos itens a possibilidade de devolver os resíduos, sem usar serviços públicos de limpeza urbana ou manejo de resíduos sólidos (Política Nacional de Resíduos Sólidos, Art. 33).", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_conectividade": {"chunk_id": "economica_conectividade", "dimensao": "Econômica", "topico": "Conectividade", "texto": "Planejamento na escala de projetos urbanos: Desenvolver, consolidar e disseminar metodologias para elaborar projetos na escala intermediária da cidade (regiões, conjuntos de bairros ou outro agrupamento de áreas que seja menor que o território municipal). O objetivo é implementar processos de renovação urbana, de estruturação urbana ou de expansão urbana. Usar os projetos como oportunidades para distribuir infraestruturas para inclusão digital no espaço urbano. Na elaboração desses projetos, observar os princípios de desenho universal (que viabiliza o uso por todas as pessoas) e as normas de acessibilidade (Estatuto da Pessoa com Deficiência, Art. 55). 1.5.3. Gestão e governança para o desenvolvimento urbano sustentável: [ver Objetivos Estratégicos 3 e 4]. ernet de qualidade para todas as pessoas overno Governo Cooperação Cooperação stadual Municipal Intragovernamental Intragovernamental Vertical Horizontal esas Empresas de Setor Privado onárias de Telecomunicações s Públicos ituições Organizações da nceiras Sociedade Civil omento à internet: Reconhecer e tornar efetivo o dinet por todas as pessoas (Marco Civil da In4o). Para isso, desenvolver e implantar políticas, infraestrutura. Incluir nessas ações projee suporte para redes de telecomunicações, estação dos serviços de telecomunicações bém outros aspectos relacionados à inclus devem ser feitas respeitando as diretrizes União Federal e Agências Reguladoras. ital para todas as pessoas: Viabilizar a ão da infraestrutura para inclusão digital carecem dessa infraestrutura e em áreconectividade. Manter a infraestrutura arantir a inclusão digital em todas as cinte. Nessas ações, enfatizar os núcleas localidades afastadas. Respeitar as prioridades definidas nas políticas nacionais de desenvolvimento regional, de desenvolvimento urbano e de telecomunicações.\n\nEditais de faixas de frequência: Prever contrapartidas para ampliação da infraestrutura para inclusão digital nos editais de faixas de frequência de serviços de telecomunicações. Priorizar o atendimento de áreas que carecem de infraestrutura de qualidade e o atendimento a todas as cidades e comunidades do país. Os municípios devem acompanhar e viabilizar as implantações decorrentes de leilão de faixas de frequência.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_inovacao": {"chunk_id": "economica_inovacao", "dimensao": "Econômica", "topico": "Inovação", "texto": "Centros de gestão integrada: Implantar centros de informações integradas e protocolos públicos para apoiar a tomada de decisões em tempo real. Priorizar a gestão de emergências e a resposta a desastres. Centros articulados com instituições de Ensino e Pesquisa e com o ecossistema de inovação local. O objetivo dessa articulação é produzir conhecimento e construir respostas para problemas públicos. Para essa finalidade, disponibilizar dados coletados pela infraestrutura digital urbana e de registros administrativos anonimizados. Articular os recursos e meios dos Centros de gestão integrada com os dos laboratórios de experimentação urbana.\n\nConstrução de ambientes para inovação: Promover processos de governança e gestão urbana que sejam interinstitucionais (com cooperação entre diferentes instituições) e colaborativos. O objetivo é construir ambientes político-jurídico-institucionais que sejam: (1) favoráveis à inovação; e (2) adaptados ao contexto territorial e ao nível de atuação das instituições.\n\nPolíticas de inovação: Estimular e integrar fóruns de inovação no setor público que sejam interfederativos (agrupando diferentes entes da federação com interesse compartilhado União, Estados, Municípios e Distrito Federal) e abertos à participação ampla de pessoas, instituições e setores interessados. O objetivo é trocar experiências, construir estratégias, políticas e programas, e formular propostas de aperfeiçoamento legislativo e de mecanismos jurídicos. Essas propostas devem reduzir os obstáculos burocráticos à inovação no setor público, incluindo as relações dos governos com a sociedade e a realização de negócios e contratos com empresas de inovação.\n\nProgramas de fomento à inovação: Promover processos de formação e programas de fomento à inovação e ao desenvolvimento tecnológico. Os objetivos são: (1) orientar ações nos setores público e privado; e (2) apoiar o desenvolvimento urbano e a transformação digital sustentáveis, conforme as necessidades e prioridades locais e regionais. 4.4. Capacidades na administração pública para a transformação digital: Desenvolver capacidades e competências na Administração Pública que sejam voltadas à atuação no contexto da transformação digital e seus desdobramentos territoriais. Implementar e fortalecer programas de desenvolvimento institucional em todos os níveis de governo.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_gestao_urbana": {"chunk_id": "economica_gestao_urbana", "dimensao": "Econômica", "topico": "Gestão Urbana", "texto": "TICs para o diagnóstico e a gestão urbana: Usar ferramentas de geoprocessamento (processamento de dados com localização geográfica) para entender melhor os fenômenos urbanos e para aperfeiçoar a capacidade de gestão dos governos locais. Incorporar nessas ações mecanismos inovadores da ciência de dados. Exemplos: (1) Inteligência Artificial (AI); e (2) análise de grandes quantidades de dados anonimizados (sem elementos que identifiquem as pessoas), conhecidos como Big Data. Respeitar a Lei Geral de Proteção de Dados Pessoais (LGPD). [Ver recomendação 3.2.] 1.5.1.2. Sistema nacional de informações para o desenvolvimento urbano: Identificar, sistematizar e disponibilizar dados e informações públicas que sejam relevantes para o desenvolvimento urbano sustentável. Esses dados e informações devem ser elaborados para formular, implementar e monitorar a Política Nacional de Desenvolvimento Urbano (PNDU). Essas ações têm duas finalidades: (1) apoiar a implementação de iniciativas locais pelos entes federados (União, Estados, Distrito Federal e Municípios) e órgãos interfederativos (que representam mais de um ente federado); e (2) atender ao Art. 16-A do Estatuto da Metrópole. [ver recomendação 3.9.]\n\nPlanejamento urbano interfederativo: Apoiar processos de planejamento urbano integrado e intersetorial (com cooperação entre as diferentes áreas de política pública) nas seguintes realidades: (1) regiões metropolitanas, (2) municípios conurbados (municípios com zonas urbanas unidas) e (3) municípios que apresentem relações de interdependência porque compartilham funções públicas de interesse comum. Esses processos de planejamento devem ser integrados de duas formas: pela elaboração de Planos de Desenvolvimento Urbano Integrado (PDUIs) ou pela elaboração conjunta e simultânea de Planos Diretores municipais (PDs). Ao elaborar os planos, é necessário articular dados, ferramentas, estratégias e as abordagens setoriais que façam parte dos planos municipais específicos.\n\nGestão territorial integrada: Usar sistemas de planejamento integrado e de gestão territorial integrada, com base em plataformas interoperáveis (que trabalham em conjunto para a troca eficaz de informações) de dados georreferenciados (plataformas que possibilitem a troca eficaz de dados com localização geográfica), em todos os níveis de governo. Os sistemas devem ser adequados às diferentes escalas das políticas públicas e respeitar a proteção de dados pessoais. Também devem atender às especificidades, demandas e capacidades locais, nos casos de sistemas municipais.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_servicos_online": {"chunk_id": "economica_servicos_online", "dimensao": "Econômica", "topico": "Serviços Online", "texto": "Identidade digital: Adotar e apoiar a implementação da “identidade digital ao cidadão”, conforme consta da Estratégia de Governo Digital.\n\nCooperação interfederativa em governo digital: Promover o intercâmbio de informações em governo digital. Implementar medidas conjuntas de natureza colaborativa por arranjos de cooperação entre governos. Exemplo: adesão voluntária à Rede Nacional de Governo Digital – Rede Gov.br (Decreto 10.332/20, Art. 7o). O objetivo é otimizar recursos e tempo. 4.2. Atuação em rede e plataformas colaborativas Estado-Sociedade: Mobilizar saberes de diferentes segmentos da sociedade, pessoas e instituições, para construir soluções criativas para problemas urbanos contemporâneos com mais agilidade.\n\nPagamentos digitais de serviços públicos: Facilitar o uso de meios de pagamentos digitais para serviços públicos, desenvolvendo e compartilhando ferramentas que estejam alinhadas com a Plataforma de Cidadania Digital. Adotar o PIX (pagamento instantâneo do Banco Central) como forma de pagamento para serviços públicos. As ações devem ocorrer em todos os níveis de governo e em cooperação interfederativa (entre União, Estados, Municípios e Distrito Federal).\n\nCompetitividade em serviços digitais urbanos: Buscar formas de garantir competitividade aos ecossistemas (conjunto e relações de pessoas e instituições que desenvolvem tecnologia e inovam) de serviços digitais urbanos. Para isso, devem-se usar práticas que evitem monopólios e promovam a escolha livre dos usuários. As ações devem estar alinhadas com a Declaração de Direitos de Liberdade Econômica.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "economica_dados_abertos": {"chunk_id": "economica_dados_abertos", "dimensao": "Econômica", "topico": "Dados Abertos", "texto": "Políticas de dados abertos: Implementar políticas de dados abertos em todos os níveis de governo. Usar experiências e recursos já disponíveis e em operação, tais como: Portal Brasileiro de Dados Abertos, Infraestrutura Nacional de Dados Abertos (INDA) e Infraestrutura Nacional de Dados Espaciais (INDE). Usar as políticas de dados abertos para cumprir o princípio da transparência na administração pública e a Lei de Acesso à Informação (LAI). Usar os modelos e recomendações produzidos pela Parceria para Governo Aberto (OGP Open Government Partnership).\n\nDados geoespaciais: Fortalecer a Infraestrutura Nacional de Dados Espaciais (INDE) como plataforma que facilita o intercâmbio de dados geoespaciais (dados espaciais com localização geográfica). Estabelecer a Política Nacional de Geoinformação (PNGeo) e consolidar um vocabulário uniforme e específico em sistemas de informação geográfica urbana. 3.5.3. Padronização para elaboração de cadastros territoriais: Articular iniciativas governamentais que elaboram, ou contribuem para elaborar, cadastros imobiliários. Essa articulação deve ter como foco uniformizar conceitos, nomenclaturas, métodos e meios de implementação. Isso irá otimizar esforços e garantir a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) de dados.\n\nRegulação da propriedade de dados: Definir com precisão os direitos sobre a propriedade e as condições para usar dados em contratos públicos e na atuação pública de caráter regulatório. O mesmo deve ocorrer em iniciativas interinstitucionais que impliquem na geração e no compartilhamento de dados, incluindo as iniciativas público-privadas. Priorizar a abertura e uso dos dados em políticas públicas. Em todos os casos mencionados, respeitar o princípio da função social da propriedade, conforme consta do artigo constitucional sobre ordem econômica. (Art. 170 da Constituição Federal).\n\nPlataformas públicas de compartilhamento de dados: Disponibilizar dados abertos e informações públicas em linguagem inclusiva, de forma organizada, compreensível e, sempre que possível, georreferenciados (com localização geográfica). As plataformas de visualização de dados e informações devem ser fáceis de usar por pessoas não-especialistas. Deste modo, as plataformas devem ser programadas em código aberto e com base em softwares livres. Os objetivos são: (1) possibilitar o uso dos dados e das informações pelo ecossistema de inovação local; (2) produzir conhecimento e soluções de interesse público; (3) promover a colaboração para aprimorar dados e análises geradas; e (4) reduzir a dependência de recursos para contratação e manutenção de licenças de softwares.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "sociocultural_educacao": {"chunk_id": "sociocultural_educacao", "dimensao": "Sociocultural", "topico": "Educação", "texto": "Formação e mercado profissional: Estimular a formação profissional na área de TICs (exemplos: programadores, cientistas de dados), por meio de ensino profissionalizante e de nível superior. Fomentar mercado de trabalho para alocação e retenção das pessoas formadas por meio da articulação de estratégias locais que respondam a demandas das cidades, apoiadas pela rede de Institutos Nacionais de Ciência e Tecnologia (INCT).\n\nTransformação digital e educação urbana: Promover ações de comunicação pública inclusiva e acessível que sejam voltadas ao desenvolvimento urbano e à transformação digital sustentáveis. Abordar grandes transformações globais (ex. mudança do clima). O objetivo dessas ações é sensibilizar e ampliar a consciência da sociedade sobre os impactos desses processos.\n\nCidade educadora: Usar a cidade como suporte para a educação urbana. Para isso, deve-se incentivar que as pessoas e instituições deem valor aos recursos naturais, as áreas verdes e espaços públicos, equipamentos e mobiliário urbano. Também deve-se informar o público sobre a história e o significado dos lugares. Essas ações devem ser associadas ao uso de ferramentas de mapeamento colaborativo que levantem e registrem aspectos subjetivos relacionados a espaços urbanos.\n\nLetramento digital nos currículos escolares: Observar, cumprir e ampliar as propostas contidas na Base Nacional Comum Curricular (BNCC) para integrar a cultura digital nos currículos escolares.\n\nCultura digital na comunidade escolar: Estimular processos de capacitação e aprendizagem em tecnologias digitais para toda a comunidade escolar. Desenvolver ações de educação especificas para o letramento digital de pessoas educadoras capacitando-as para atuar como multiplicadoras da inclusão digital. O objetivo é ampliar, agilizar e facilitar o letramento digital desde a infância até a fase adulta.\n\nRecursos digitais na educação formal: Promover o aparelhamento tecnológico das instituições de ensino por meio de laboratórios, equipamentos, programas, ferramentas, softwares e outros recursos digitais.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "sociocultural_inclusao_digital": {"chunk_id": "sociocultural_inclusao_digital", "dimensao": "Sociocultural", "topico": "Inclusão Digital", "texto": "Informações sobre exclusão digital: Entender melhor os fatores associados à exclusão digital. Exemplos: (1) compreender quais são as condições de conectividade dos grupos vulneráveis; e (2) compreender quais são as condições de conexão em cada localização. Para isso, usar dados georreferenciados (com localização geográfica) separados por critérios como renda, raça, gênero, escolaridade e idade. Incluir análises específicas para as pessoas com deficiência. O uso e tratamento dos dados deve respeitar a legislação sobre proteção de dados pessoais (LGPD). 1.2. Visão de território para o desenvolvimento urbano sustentável:\n\nEnfrentamento da exclusão digital: Promover soluções para os diferentes fatores de exclusão digital nas estratégias de universalização e democratização do acesso à internet e a tecnologias digitais. Essas ações devem estar alinhadas com a Estratégia Brasileira de Transformação Digital, para ajudar a alcançar suas metas.\n\nInclusão digital de pessoas com deficiência: Criar e usar soluções, elaborar e difundir normas e procedimentos para ampliar a acessibilidade da pessoa com deficiência à computação e à internet. Realizar essas ações também na oferta de serviços públicos digitais e outras iniciativas de governo digital (Estatuto da Pessoa com Deficiência, Art. 78). Estimular o desenvolvimento de soluções técnicas previstas no Plano Nacional de Internet das Coisas (Decreto 9.854/2019). 2.4.2.Inclusão digital na perspectiva de gênero: Cumprir as metas nacionais para garantir a igualdade de gênero nas seguintes situações: (1) no acesso, nas habilidades de uso e na produção de tecnologias da informação e comunicação; (2) no acesso e na produção do conhecimento científico; e (3) no acesso e na produção de informação, conteúdos de comunicação e mídias (Agenda 2030, ODS 5, 5.b).\n\nLetramento digital: [ver Objetivo Estratégico 7]\n\nOtimização e melhoria de processos administrativos: Estabelecer sistema de processo administrativo eletrônico. Aderir preferencialmente à infraestrutura pública colaborativa do Processo Eletrônico Nacional (PEN) e suas ações, como o Sistema Eletrônico de Informações – SEI. O objetivo é diminuir custos e tornar a tramitação (o andamento) de documentos públicos mais rápida, transparente e acessível. 3.6.3. Serviços analógicos e medidas de transição para o digital: Manter e melhorar procedimentos analógicos e presenciais quando ofertar serviços públicos digitais. Essas ações também devem ser feitas ao implementar medidas de transição, especialmente quando for um serviço essencial. Considerar a grande quantidade de fatores de exclusão digital.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "sociocultural_inclusao_e_equidade": {"chunk_id": "sociocultural_inclusao_e_equidade", "dimensao": "Sociocultural", "topico": "Inclusão e Equidade", "texto": "Integração de dados para a política urbana: Promover a constante integração de setores e instituições para o intercâmbio de dados, como os dados fiscais, de serviços urbanos e de registros imobiliários. Essa integração permitirá entender melhor o uso e a ocupação do solo urbano. Essas ações irão viabilizar a aplicação de instrumentos de política urbana, como o Imposto Predial e Territorial Urbano (IPTU) progressivo no tempo e o Parcelamento, Edificação e Utilização Compulsório (PEUC). 1.5.1.4. Mapeamento de áreas verdes urbanas e serviços ecossistêmicos: Apoiar os municípios e órgãos interfederativos (que representam mais de um ente federado União, Estados, Distrito Federal e Municípios) a mapear as suas áreas verdes urbanas. Essa ação contribuirá com a meta 11.7 do Objetivo de Desenvolvimento Sustentável 11 da Agenda 2030 da ONU. Além das áreas verdes urbanas, apoiar municípios e órgãos interfederativos a mapear, atribuir valor financeiro e gerir de forma responsável seus recursos naturais e serviços ecossistêmicos. Para isso, disponibilizar sistema e metodologia de cadastro que sejam unificados em âmbito nacional.\n\nRede digital para colaboração urbana: Estimular a formação de uma rede para o desenvolvimento urbano sustentável. A rede deve ser multinível (atuar nos níveis nacionais, regionais, estaduais e locais), interinstitucional (cooperação entre diferentes instituições) e intersetorial (com cooperação entre as diferentes áreas de política pública). A rede deve oferecer recursos digitais e inclusivos para realizar trabalhos colaborativos, incluindo a implementação e a retroalimentação desta Carta Brasileira para Cidades Inteligentes. 4.2.2. Rede de assistência técnica remota para ações no território: Expandir e adaptar o modelo da assistência técnica remota baseada em recursos digitais que foi implementado de forma pioneira pela Rede Universitária de Telemedicina. Essa rede de assistência técnica remota deve apoiar órgãos oficiais interfederativos (que agrupam diferentes entes da federação com interesse compartilhado União, Estados, Municípios e Distrito Federal) e municípios para implementar políticas, projetos e ações de desenvolvimento urbano sustentável, incluindo iniciativas de cidades inteligentes. Apoiar principalmente os municípios de menor capacidade institucional.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "sociocultural_participacao_cidada": {"chunk_id": "sociocultural_participacao_cidada", "dimensao": "Sociocultural", "topico": "Participação Cidadã", "texto": "Mapeamentos colaborativos: Ampliar o uso de ferramentas de mapeamento colaborativo na gestão pública como estratégia para mobilizar saberes e engajamento comunitários. Essas ferramentas também são estratégicas no controle social das políticas públicas, especialmente para levantar necessidades habitacionais, bens comuns, ativos urbanos, ambientais e culturais de interesse coletivo. Além disso, contribuem para identificar e gerir conflitos urbanos. Essas ferramentas devem incluir tecnologias assistivas, de forma a possibilitar a participação da pessoa com deficiência ou mobilidade reduzida. Nessas ações, privilegiar o uso de plataformas e ferramentas gratuitas e de código aberto, como o OpenStreetMap. [Ver recomendação 3.9]\n\nGestão democrática das cidades: Estimular o engajamento e a participação pública inclusiva: na elaboração e na revisão do Plano Diretor e de outros instrumentos de planejamento municipal; (1) em aspectos cotidianos de zeladoria e gestão urbana; e (2) na interação governo-pessoas. Esse estímulo deve se dar por meio de mecanismos inovadores e soluções digitais, e com o uso de tecnologias assistivas (com funcionalidade para garantir autonomia, independência, qualidade de vida e inclusão social da pessoa com deficiência ou com mobilidade reduzida). As ações devem estar de acordo com as demandas e necessidades locais e devem ser adequadas às características organizacionais e institucionais do município. Buscar alinhamento com a Estratégia de Governo Digital (Decreto 10.332/2020, objetivo 14.2) e executar a gestão democrática da cidade (Estatuto da Cidade, Capítulo IV).\n\nImpactos locais da transformação digital e controle social: Estimular que os temas do desenvolvimento urbano e da transformação digital sejam discutidos de forma integrada. Para isso, deve-se estimular a articulação institucional de conselhos ou fóruns que debatem sobre esses temas e que atuem no controle social de políticas públicas. Essas instituições devem acompanhar, avaliar e dar suporte à atuação do município sobre os impactos da transformação digital no território. As ações junto aos municípios devem considerar as condições político-institucionais específicas de cada cidade. 8.5. Ciência, tecnologia e inovação para a transformação digital e o desenvolvimento urbano sustentáveis: Mobilizar diferentes setores da sociedade para ampliar a compreensão sobre os impactos da transformação digital nas cidades. Devem ser considerados os impactos sobre os aspectos econômico-financeiro, sociocultural, urbano-ambiental e político-institucional.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "meio_ambiente_qualidade_do_ar_e_emissoes": {"chunk_id": "meio_ambiente_qualidade_do_ar_e_emissoes", "dimensao": "Meio Ambiente", "topico": "Qualidade do Ar e Emissões", "texto": "Transformação digital e meio ambiente: Desenvolver e usar metodologias, dados e indicadores que respondam às mudanças ambientais e climática (aumento da temperatura média global com aumento da ocorrência de eventos climáticos extremos). Atuar nas frentes de adaptação (como prevenção a eventos climáticos extremos – deslizamentos, inundações, secas, erosões etc.) e de mitigação (redução de emissões de carbono).\n\nDecrescimento e economia zero emissões: Incluir perspectivas de decrescimento, descarbonização e outras variáveis inovadoras de sustentabilidade na exploração de novas alternativas de organização social e econômica. Introzudir a redução de desigualdades socioeconomicas e a distribuição de riquezas na discusssão de modelos econômicos verdes, justos e inovadores. O objetivo é lidar com a escassez de recursos naturais e com a precarização do mundo do trabalho.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "meio_ambiente_energia_e_iluminacao_publica": {"chunk_id": "meio_ambiente_energia_e_iluminacao_publica", "dimensao": "Meio Ambiente", "topico": "Energia e Iluminação Pública", "texto": "Eficiência energética e economia circular: Desenvolver projetos, utilizar mecanismos e tecnologias que ampliem a eficiência energética de infraestruturas e edifícios urbanos. Promover processos e desenvolver soluções que incorporem a lógica da economia circular (aproveitamento de resíduos). O objetivo é promover o uso responsável dos recursos naturais e garantir a qualidade de vida das pessoas das atuais e futuras gerações.\n\nProjetos de iluminação pública: Promover a equidade de acesso ao serviço de iluminação pública nas cidades. Nos projetos de expansão e modernização das redes de iluminação pública, priorizar as seguintes áreas: (1) espaços públicos de utilização intensiva; (2) áreas urbanas desservidas; e (3) áreas urbanas inseguras, com índices de violência urbana acima da média da cidade. Essa priorização e as características de cada área devem ser observadas para a definição de padrões luminotécnicos adequados. Implantar projetos de iluminação pública adequados à diversidade dos municípios brasileiros.\n\nSustentabilidade em iluminação pública: Elevar os padrões de eficiência energética em projetos de modernização e expansão da rede de iluminação pública. Nesses projetos, buscar a redução da poluição luminosa (poluição gerada pelo excesso de luz artificial). Promover a gestão eficiente do serviço por meio da adoção de soluções digitais integradas à rede. O objetivo é minimizar impactos da prestação do serviço de iluminação pública no meio ambiente e na saúde humana, assim como melhorar a qualidade de vida das pessoas nas cidades.\n\nAproveitamento da infraestrutura: Considerar a utilização potencial da rede de iluminação pública como infraestrutura de suporte para a oferta de serviços digitais. Buscar esse aproveitamento especialmente nos projetos de modernização e de expansão da rede de iluminação pública. Garantir o compartilhamento em condições justas, razoáveis e não discriminatórias de acesso aos postes de distribuição de energia elétrica. [Ver recomendação 2.6].", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "institucional_governanca_e_planejamento": {"chunk_id": "institucional_governanca_e_planejamento", "dimensao": "Capacidades Institucionais", "topico": "Governança e Planejamento", "texto": "Visão de contexto: Estimular a atuação local com visão de contexto, disponibilizando ferramentas para facilitar que os municípios percebam seus próprios contextos e inserções regionais. O objetivo é qualificar o planejamento e a gestão integrada de suas áreas urbanas, rurais e naturais. Deve haver articulação com outros municípios e demais entes federados (União, Estados, Municípios e Distrito Federal). Essas ações devem estar em linha com a Política Nacional de Desenvolvimento Regional (PNDR) e com a Política Nacional de Desenvolvimento Urbano (PNDU).\n\nIntersetorialidade no planejamento urbano: Construir e consolidar uma visão integrada do planejamento municipal com base nos instrumentos de planejamento setorial. Enfatizar as áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs). Exemplo de instrumentos de tecnologias de informação e comunicação nas cidades: Plano Diretor de Cidades Inteligentes e Plano Diretor de TICs. O objetivo é possibilitar que as iniciativas sejam implementadas de forma coordenada no território, usando mecanismos locais de gestão e governança. Para isso, devem ser incluídos mecanismos de dados e informações.\n\nGovernança intermunicipal de dados: Estabelecer instituições de cooperação intermunicipal (entre municípios) para implantar, gerir e operar bases de dados, sistemas digitais e soluções compartilhadas de tecnologia de informação e comunicação. O objetivo deve ser otimizar recursos e ampliar a sustentabilidade dessas ações. Exemplos de instituições de cooperação intermunicipal (entre municípios): consórcios públicos, instâncias de governança metropolitana e associações de municípios.\n\nAgências reguladoras: Alinhar normas, técnicas e operações relativas a serviços públicos que requeiram a instalação de infraestruturas no espaço urbano. Para isso, estabelecer espaço de governança permanente entre agências reguladoras desses serviços públicos. Os objetivos são: (1) racionalizar a instalação e a manutenção de infraestruturas no espaço urbano, otimizando sua utilização; (2) assegurar a observância das normas urbanísticas locais pelas concessionárias dos serviços regulados.\n\nValorização de servidores públicos inovadores: Estabelecer mecanismos para identificar servidores públicos inovadores em todos os níveis de governo. Oferecer incentivos e oportunidades para o desenvolvimento e uso das potencialidades dos servidores em trabalhos institucionais e no aprimoramento de políticas públicas. 4.5. Adoção de processos inovadores de gestão e governança no nível local:", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "institucional_infraestrutura_de_ti": {"chunk_id": "institucional_infraestrutura_de_ti", "dimensao": "Capacidades Institucionais", "topico": "Infraestrutura de TI", "texto": "Instrumentos ambientais: Introduzir o conceito e desenvolver projetos de infraestrutura verde em áreas urbanas. Sempre que possível, substituir a infraestrutura cinza pela infraestrutura verde. Integrar as perspectivas de serviços ecossistêmicos e de soluções baseadas na natureza nos instrumentos de política urbana. Estimular o desenvolvimento de regiões produtoras de alimentos próximas dos centros urbanos. Utilizar as TICs para estimular padrões responsáveis de produção e consumo e ativação da economia local.\n\nInteroperabilidade: Garantir a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) ao implementar soluções de TICs (Tecnologias de Informação e Comunicação) em governos. Garantir a interoperabilidade também em iniciativas interinstitucionais, inclusive público-privadas. Em todos os casos, respeitar e usar normas, padrões e protocolos públicos oficiais (Programa de Interoperabilidade do Governo Eletrônico e-PING).\n\nContratações governamentais de TICs: Instituir, testar e normatizar novos modelos de governos contratarem Tecnologias de Informação e Comunicação (TICs). Essas ações devem ser feitas de forma conjunta, em cooperação intergovernamental (entre governos). Os novos modelos de contratação devem ter como base o uso de softwares livres e códigos abertos. Assegurar a contratação de instituições, entidades e empresas que tenham: (1) compromisso com os direitos humanos; (2) compromisso com a liberdade de expressão; (3) reputação ilibada; (4) comprovada experiência na área; e (5) responsabilidade e compromisso com a coisa pública. Priorizar a contratação de instituições, entidades e empresas locais. Usar mecanismos de colaboração para compartilhar experiências e boas práticas, tal como acontece na Comunidade de TICs da Plataforma GestGov.\n\nPadrões sustentáveis de produção e consumo: Utilizar as TICs para estimular padrões responsáveis de produção e consumo e ativação da economia local.\n\nCrédito para pequenas empresas de TICs: Facilitar o acesso a condições especiais de crédito por pessoas microempreendedoras individuais e por pequenas empresas de TICs (tecnologias de informação e comunicação). Estabelecer incentivos financeiros e técnicos à operação de pequenos provedores de Internet de forma a garantir a provisão e a sustentabilidade de iniciativas de acesso à internet em parceria com o poder público.\n\nTICs para a redução da pobreza urbana: Usar as tecnologias de informação e comunicação para reduzir a pobreza urbana, contribuindo para a Meta 1.4 do Objetivo de Desenvolvimento Sustentável 1.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "institucional_monitoramento_e_transparencia": {"chunk_id": "institucional_monitoramento_e_transparencia", "dimensao": "Capacidades Institucionais", "topico": "Monitoramento e Transparência", "texto": "Dispositivos digitais no ambiente urbano: Estimular o uso de metodologias, dados e indicadores, digitais ou não, para monitorar e avaliar os impactos ambientais causados por infraestruturas e dispositivos digitais nos ambientes urbanos. Promover o uso responsável de recursos nas soluções de modernização tecnológica de serviços urbanos. O objetivo é reduzir a pegada de carbono na transformação digital das cidades.\n\nTransparência nos algoritmos de empresas de TICs: Incentivar que empresas de tecnologia de informação e comunicação digital tenham padrões elevados de transparência sobre os critérios e pressupostos que usam nos seus algoritmos. Possibilitar e fortalecer processos de auditoria algorítmica e fomentar o uso de softwares de código fonte aberto ou livres. Essas ações contribuem e devem estar alinhadas com o Sistema Nacional para a Transformação Digital.\n\nTransparência orçamentária na Administração Pública: Padronizar dados e informações relativos a contas públicas de todos os poderes e níveis de governo. Garantir a qualidade e a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) desses dados e informações. Incluir mecanismos que permitam a geolocalização de investimentos públicos. Implementar a transparência ativa, adotando portais públicos organizados que facilitem a compreensão e o manuseio dos dados e informações por pessoas não especializadas. Os objetivos são: (1) facilitar o planejamento e a gestão orçamentária, financeira e patrimonial na Administração Pública; (2) permitir a integração de dados e informações; (3) facilitar o controle interno e externo, bem como o controle social das contas públicas. clusivos de governança urbana e fortaleomo gestor de impactos da transformação overno Governo Cooperação Cooperação stadual Municipal Intragovernamental Intragovernamental Vertical Horizontal esas Empresas de Setor Privado onárias de Telecomunicações s Públicos ituições Organizações da nceiras Sociedade Civil omento overnamental: Fortalecer a articulação solidar a governança urbana multinível eis nacional, regional, estadual e local), operação entre diferentes entes da fedeMunicípios e Distrito Federal) e interseentre as diferentes áreas de política púdos governos estaduais e federal no apoio ações e políticas para os contextos os municípios. terministerial: Fortalecer espaço de gocional de âmbito federal para cidades interticipação aberta aos setores interessados. : (1) construir condições para implementar artilhada para cidades inteligentes; e (2) criar condições para a continuidade da plataforma colaborativa da Carta Brasileira para Cidades Inteligentes.", "fonte": "Carta Brasileira para Cidades Inteligentes"}, "institucional_dados_e_seguranca_da_informacao": {"chunk_id": "institucional_dados_e_seguranca_da_informacao", "dimensao": "Capacidades Institucionais", "topico": "Dados e Segurança da Informação", "texto": "Apoio técnico e financeiro para a conectividade: Oferecer soluções para implantar e manter infraestrutura para inclusão digital. Isso deve ser feito por meio de apoio técnico e financeiro ou outros mecanismos de prestação de serviços públicos essenciais. Considerar as capacidades governativas dos municípios brasileiros. Considerar também as condições socioeconômicas e a localização da moradia da população beneficiária. Fomentar e facilitar a articulação dos municípios e de entidades supramunicipais (entidades que atuam sobre um agrupamento de municípios) com operadoras de serviços de telecomunicações. ança de dados e de tecnologias, com vacidade overno Governo Cooperação Cooperação stadual Municipal Intragovernamental Intragovernamental Vertical Horizontal esas Empresas de Setor Privado onárias de Telecomunicações s Públicos ituições Organizações da nceiras Sociedade Civil omento ica: Garantir a segurança cibernética ositivos, sistemas, dados e informações iretrizes, normas e procedimentos que idem a confiabilidade de hardwares, sispositivos de acesso pessoal e ferramentivos). dados pessoais: Garantir a proteção de o completamente à Lei Geral de ProteLGPD). Respeitar a titularidade da pesus próprios dados pessoais, garantindo, itos fundamentais de liberdade, intimiegurar que o compartilhamento de dados incípios de finalidade e transparência. ações, estabelecer normas e procedimensenvolvimento seguro e ético de negócios inovadores baseados em dados. Seguir definições estabelecidas pela Agência Nacional de Proteção de Dados (ANPD).\n\nNormas locais de proteção de dados pessoais: Apoiar os municípios para que adéquem normas e procedimentos à Lei Geral de Proteção de Dados Pessoais (LGPD). Nessa ação, regular de forma prioritária:(1) a regulação do tratamento de dados em serviços públicos essenciais; e (2) os cadastros em serviços digitais. Articular ações junto à Autoridade Nacional de Proteção de Dados (ANPD). O objetivo é garantir a coesão entre as políticas de compartilhamento de dados com aplicação geral e as propostas de cidades inteligentes.", "fonte": "Carta Brasileira para Cidades Inteligentes"}}"""

CHUNKS_CARTA = json.loads(_CHUNKS_JSON)  # chunk_id -> {chunk_id, dimensao, topico, texto, fonte}

print(f"🔹 {len(CHUNKS_CARTA)} chunks temáticos carregados da Carta "
      f"(+ {len(TOPICOS_SEM_CHUNK_NA_CARTA)} tópicos sem conteúdo dedicado, tratados via fallback).")


🔹 20 chunks temáticos carregados da Carta (+ 10 tópicos sem conteúdo dedicado, tratados via fallback).


In [5]:
# =====================================================
# ÍNDICE FAISS DOS CHUNKS DA CARTA (busca semântica / RAG)
# =====================================================
# Permite recuperar os chunks da Carta mais relevantes por similaridade de
# embeddings, em vez de depender apenas do mapeamento fixo tópico -> chunk_id.
# Usado principalmente como fallback para os tópicos listados em
# TOPICOS_SEM_CHUNK_NA_CARTA, que hoje ficam sem nenhum trecho de referência
# (ver selecionar_chunks_dimensao, mais adiante).

def _construir_indice_chunks():
    documentos = [
        LCDocument(
            page_content=f"{c['topico']}: {c['texto']}",
            metadata={
                "chunk_id": c["chunk_id"],
                "dimensao": c["dimensao"],
                "topico": c["topico"],
            },
        )
        for c in CHUNKS_CARTA.values()
    ]
    return FAISS.from_documents(documentos, embeddings)


def carregar_ou_construir_indice_chunks():
    """Carrega o índice FAISS dos chunks do Drive se ele já existir; caso
    contrário, gera os embeddings uma única vez e salva para reuso nas
    próximas execuções (evita custo/tempo de reprocessar sempre)."""
    if os.path.exists(FAISS_INDEX_CHUNKS_PATH):
        return FAISS.load_local(
            FAISS_INDEX_CHUNKS_PATH, embeddings, allow_dangerous_deserialization=True
        )
    indice = _construir_indice_chunks()
    indice.save_local(FAISS_INDEX_CHUNKS_PATH)
    return indice


INDICE_CHUNKS = carregar_ou_construir_indice_chunks()
print(f"🔹 Índice FAISS de chunks pronto ({len(CHUNKS_CARTA)} vetores).")


🔹 Índice FAISS de chunks pronto (20 vetores).


In [6]:
# =====================================================
# ESTRUTURA DE DIMENSÕES E TÓPICOS
# =====================================================
CHUNKS_GERAIS = [
    "conceito_brasileiro_de_cidades_inteligentes",
    "diversidade_territorial_e_reducao_de_desigualdades",
    "transformacao_digital_adaptada_a_capacidade_municipal",
]
CHUNKS_ECONOMICA = [
    "agua_e_esgoto", "residuos_solidos", "transporte", "vias_publicas",
    "conectividade", "inovacao", "gestao_urbana", "servicos_online", "dados_abertos",
]
CHUNKS_SOCIOCULTURAL = [
    "educacao", "cultura_e_esporte", "saude", "seguranca_publica",
    "defesa_civil", "inclusao_digital", "inclusao_e_equidade", "participacao_cidada",
]
CHUNKS_MEIO_AMBIENTE = [
    "agua_e_saneamento", "residuos_solidos", "areas_verdes",
    "qualidade_do_ar_e_emissoes", "energia_e_iluminacao_publica",
]
CHUNKS_CAPACIDADES_INSTITUCIONAIS = [
    "governanca_e_planejamento", "infraestrutura_de_ti", "servicos_publicos_digitais",
    "monitoramento_e_transparencia", "dados_e_seguranca_da_informacao",
]

# dimensão -> (prefixo do chunk_id, lista de tópicos)
DIMENSOES = {
    "Econômica": {"prefixo": "economica", "topicos": CHUNKS_ECONOMICA},
    "Sociocultural": {"prefixo": "sociocultural", "topicos": CHUNKS_SOCIOCULTURAL},
    "Meio Ambiente": {"prefixo": "meio_ambiente", "topicos": CHUNKS_MEIO_AMBIENTE},
    "Capacidades Institucionais": {"prefixo": "institucional", "topicos": CHUNKS_CAPACIDADES_INSTITUCIONAIS},
}

# Palavras-chave por tópico — usadas apenas para classificar indicadores
# (não são trechos da Carta, são termos de busca).
PALAVRAS_CHAVE_TOPICO = {
    "agua_e_esgoto": ["água", "esgoto", "saneamento básico", "abastecimento"],
    "residuos_solidos": ["resíduos sólidos", "lixo", "coleta seletiva", "reciclagem"],
    "transporte": ["transporte público", "mobilidade urbana", "ônibus"],
    "vias_publicas": ["vias públicas", "pavimentação", "trânsito", "mobiliário urbano"],
    "conectividade": ["conectividade", "internet", "banda larga", "wi-fi"],
    "inovacao": ["inovação", "empreendedorismo", "startups"],
    "gestao_urbana": ["gestão urbana", "planejamento urbano", "uso do solo", "plano diretor"],
    "servicos_online": ["serviços online", "serviços digitais", "atendimento digital", "governo digital"],
    "dados_abertos": ["dados abertos", "portal de dados", "transparência de dados"],
    "educacao": ["educação", "escola", "ensino"],
    "cultura_e_esporte": ["cultura", "esporte", "lazer"],
    "saude": ["saúde", "atenção primária", "telessaúde", "telemedicina"],
    "seguranca_publica": ["segurança pública", "violência", "policiamento"],
    "defesa_civil": ["defesa civil", "risco", "desastre"],
    "inclusao_digital": ["inclusão digital", "acesso digital", "letramento digital"],
    "inclusao_e_equidade": ["inclusão", "equidade", "acessibilidade", "grupos vulneráveis"],
    "participacao_cidada": ["participação cidadã", "participação social", "controle social"],
    "agua_e_saneamento": ["água", "saneamento", "recursos hídricos"],
    "areas_verdes": ["áreas verdes", "parques", "arborização"],
    "qualidade_do_ar_e_emissoes": ["qualidade do ar", "emissões", "poluição"],
    "energia_e_iluminacao_publica": ["energia", "iluminação pública", "eficiência energética"],
    "governanca_e_planejamento": ["governança", "planejamento estratégico", "plano diretor"],
    "infraestrutura_de_ti": ["infraestrutura de ti", "tecnologia da informação", "sistemas municipais"],
    "servicos_publicos_digitais": ["serviços públicos digitais", "digitalização"],
    "monitoramento_e_transparencia": ["monitoramento", "transparência", "prestação de contas"],
    "dados_e_seguranca_da_informacao": ["proteção de dados", "segurança da informação", "lgpd"],
}


# =====================================================
# MAPA ENTRE INDICADORES, DIMENSÕES E TÓPICOS
# =====================================================
# Construído automaticamente a partir da estrutura real da planilha
# indicadores.xlsx: 85 indicadores setoriais nas 4 dimensões, cada um
# seguido, na própria planilha, por uma coluna "N M Indicador..." com o
# nível de maturidade (0-7) já calculado para aquele indicador.
# MAPA_NIVEL_MATURIDADE guarda essa relação indicador -> coluna de nível,
# usada para priorizar sugestões pelos indicadores de menor maturidade
# (ver estimar_nivel_indicador, na próxima célula).
#
# 3 indicadores do grupo "Habitação" (dimensão Econômica) não têm tópico
# correspondente na taxonomia dos 30 tópicos e por isso NÃO entram no
# mapa (ficam registrados em INDICADORES_NAO_CLASSIFICADOS quando
# encontrados na planilha): "Percentual de domicílios com população
# vivendo em aglomerados subnormais", "Assentamentos urbanos precários"
# e "Programas e ações habitacionais".
_MAPA_INDICADORES_JSON = r"""{"Índice da população total com atendimento de água": {"dimensao": "Econômica", "topico": "Água e Esgoto", "chunk_id": "economica_agua_e_esgoto"}, "Índice da população total com atendimento de esgoto": {"dimensao": "Econômica", "topico": "Água e Esgoto", "chunk_id": "economica_agua_e_esgoto"}, "Índice da população urbana com atendimento de esgoto": {"dimensao": "Econômica", "topico": "Água e Esgoto", "chunk_id": "economica_agua_e_esgoto"}, "Taxa da população coberta com serviço de coleta de resíduos": {"dimensao": "Econômica", "topico": "Resíduos Sólidos", "chunk_id": "economica_residuos_solidos"}, "Coleta seletiva de resíduos no município": {"dimensao": "Econômica", "topico": "Resíduos Sólidos", "chunk_id": "economica_residuos_solidos"}, "Serviços regulares de transporte de passageiros": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Serviços de compartilhamento de viagens": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Serviço de informações de transporte público em tempo real": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Serviços e soluções inteligentes para mobilidade urbana": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Acessibilidade no transporte público": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Ciclomobilidade na cidade": {"dimensao": "Econômica", "topico": "Transporte", "chunk_id": "economica_transporte"}, "Índice de pavimentação das vias públicas": {"dimensao": "Econômica", "topico": "Vias Públicas", "chunk_id": "economica_vias_publicas"}, "Escala de acesso a banda larga fixa": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Escala de acesso a banda larga móvel": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Cobertura de acesso a banda larga móvel por tecnologias 3G e 4G": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Cobertura de fibra ótica": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Rede de tecnologia interligando os equipamentos e edifícios públicos": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Escala de acesso a banda larga fixa de alta velocidade": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Números de estações rádio base": {"dimensao": "Econômica", "topico": "Conectividade", "chunk_id": "economica_conectividade"}, "Qualificação profissional e intermediação de mão de obra": {"dimensao": "Econômica", "topico": "Inovação", "chunk_id": "economica_inovacao"}, "Inclusão produtiva urbana": {"dimensao": "Econômica", "topico": "Inovação", "chunk_id": "economica_inovacao"}, "Acesso a crédito, microcrédito e seguro": {"dimensao": "Econômica", "topico": "Inovação", "chunk_id": "economica_inovacao"}, "Geração de trabalho e renda no município": {"dimensao": "Econômica", "topico": "Inovação", "chunk_id": "economica_inovacao"}, "Sistema de informação geográfica da prefeitura": {"dimensao": "Econômica", "topico": "Gestão Urbana", "chunk_id": "economica_gestao_urbana"}, "Centros de comando e controle para gestão da cidade": {"dimensao": "Econômica", "topico": "Gestão Urbana", "chunk_id": "economica_gestao_urbana"}, "Plataforma integrada de cidade inteligente": {"dimensao": "Econômica", "topico": "Gestão Urbana", "chunk_id": "economica_gestao_urbana"}, "Serviços no website da prefeitura": {"dimensao": "Econômica", "topico": "Serviços Online", "chunk_id": "economica_servicos_online"}, "Dados abertos da gestão municipal": {"dimensao": "Econômica", "topico": "Dados Abertos", "chunk_id": "economica_dados_abertos"}, "Indice de equipamentos de tecnologia disponíveis nas escolas públicas municipais": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Taxa de analfabetismo": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Índice de desenvolvimento da educação básica (IDEB) - anos finais": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Vagas no ensino superior": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Centros de educação tecnológica": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Ações de educação para comunidades específicas": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Taxas de distorção idade-série": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Percentual de escolas municipais com acesso à internet": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Computadores para uso dos alunos": {"dimensao": "Sociocultural", "topico": "Educação", "chunk_id": "sociocultural_educacao"}, "Estrutura de equipamentos culturais e esportivos": {"dimensao": "Sociocultural", "topico": "Cultura e Esporte", "chunk_id": "sociocultural_cultura_e_esporte"}, "Proteção do patrimônio cultural material e imaterial": {"dimensao": "Sociocultural", "topico": "Cultura e Esporte", "chunk_id": "sociocultural_cultura_e_esporte"}, "Serviços on-line para promoção de cultura": {"dimensao": "Sociocultural", "topico": "Cultura e Esporte", "chunk_id": "sociocultural_cultura_e_esporte"}, "Serviços culturais on-line oferecidos para a população": {"dimensao": "Sociocultural", "topico": "Cultura e Esporte", "chunk_id": "sociocultural_cultura_e_esporte"}, "Serviços de telemedicina ou telessaúde": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Leitos hospitalares na rede pública municipal": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Médicos disponíveis na rede pública municipal": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Prontuário eletrônico": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Serviços on-line de saúde oferecidos aos pacientes": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Índice de risco e proteção à saúde dos nascidos vivos": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Mortalidade materna": {"dimensao": "Sociocultural", "topico": "Saúde", "chunk_id": "sociocultural_saude"}, "Soluções em monitoramento para a segurança pública": {"dimensao": "Sociocultural", "topico": "Segurança Pública", "chunk_id": "sociocultural_seguranca_publica"}, "Taxa de homicídios": {"dimensao": "Sociocultural", "topico": "Segurança Pública", "chunk_id": "sociocultural_seguranca_publica"}, "Políticas públicas e ações para segurança pública": {"dimensao": "Sociocultural", "topico": "Segurança Pública", "chunk_id": "sociocultural_seguranca_publica"}, "Soluções de tecnologia para gestão e monitoramento de desastres naturais": {"dimensao": "Sociocultural", "topico": "Defesa Civil", "chunk_id": "sociocultural_defesa_civil"}, "Vulnerabilidade a riscos e desastres naturais": {"dimensao": "Sociocultural", "topico": "Defesa Civil", "chunk_id": "sociocultural_defesa_civil"}, "Promoção de inclusão digital": {"dimensao": "Sociocultural", "topico": "Inclusão Digital", "chunk_id": "sociocultural_inclusao_digital"}, "Cursos de capacitação tecnológica": {"dimensao": "Sociocultural", "topico": "Inclusão Digital", "chunk_id": "sociocultural_inclusao_digital"}, "Políticas públicas para mulheres": {"dimensao": "Sociocultural", "topico": "Inclusão e Equidade", "chunk_id": "sociocultural_inclusao_e_equidade"}, "Inclusão social para grupos específicos": {"dimensao": "Sociocultural", "topico": "Inclusão e Equidade", "chunk_id": "sociocultural_inclusao_e_equidade"}, "Formas presenciais para participação pública": {"dimensao": "Sociocultural", "topico": "Participação Cidadã", "chunk_id": "sociocultural_participacao_cidada"}, "Formas on-line para participação pública": {"dimensao": "Sociocultural", "topico": "Participação Cidadã", "chunk_id": "sociocultural_participacao_cidada"}, "Índice de volume de esgoto coletado": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Consumo médio per capita de água": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Soluções inteligentes para gestão na distribuição e consumo de água": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Índice de perdas na distribuição de água": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Índice de volume de esgoto tratado": {"dimensao": "Meio Ambiente", "topico": "Água e Saneamento", "chunk_id": "meio_ambiente_agua_e_saneamento"}, "Percentual de material recolhido pela coleta seletiva": {"dimensao": "Meio Ambiente", "topico": "Resíduos Sólidos", "chunk_id": "meio_ambiente_residuos_solidos"}, "Soluções inteligentes para otimização da coleta de resíduos": {"dimensao": "Meio Ambiente", "topico": "Resíduos Sólidos", "chunk_id": "meio_ambiente_residuos_solidos"}, "Proteção e gestão do meio ambiente e áreas verdes do município": {"dimensao": "Meio Ambiente", "topico": "Áreas Verdes", "chunk_id": "meio_ambiente_areas_verdes"}, "Soluções em monitoramento de gases de efeito estufa e qualidade do ar": {"dimensao": "Meio Ambiente", "topico": "Qualidade do Ar e Emissões", "chunk_id": "meio_ambiente_qualidade_do_ar_e_emissoes"}, "Monitoramento da qualidade do ar": {"dimensao": "Meio Ambiente", "topico": "Qualidade do Ar e Emissões", "chunk_id": "meio_ambiente_qualidade_do_ar_e_emissoes"}, "Soluções inteligentes para gestão do consumo de energia elétrica": {"dimensao": "Meio Ambiente", "topico": "Energia e Iluminação Pública", "chunk_id": "meio_ambiente_energia_e_iluminacao_publica"}, "Soluções para telegestão da iluminação pública": {"dimensao": "Meio Ambiente", "topico": "Energia e Iluminação Pública", "chunk_id": "meio_ambiente_energia_e_iluminacao_publica"}, "Governança Colaborativa - Responsáveis": {"dimensao": "Capacidades Institucionais", "topico": "Governança e Planejamento", "chunk_id": "institucional_governanca_e_planejamento"}, "Incorporação de TICs - Planejamento": {"dimensao": "Capacidades Institucionais", "topico": "Governança e Planejamento", "chunk_id": "institucional_governanca_e_planejamento"}, "Planejamento Estratégico para Transformação Digital": {"dimensao": "Capacidades Institucionais", "topico": "Governança e Planejamento", "chunk_id": "institucional_governanca_e_planejamento"}, "Governança de TI - Práticas": {"dimensao": "Capacidades Institucionais", "topico": "Infraestrutura de TI", "chunk_id": "institucional_infraestrutura_de_ti"}, "Infraestrutura de Hw e Sw - Armazenamento": {"dimensao": "Capacidades Institucionais", "topico": "Infraestrutura de TI", "chunk_id": "institucional_infraestrutura_de_ti"}, "Gestão Integrada de Dados": {"dimensao": "Capacidades Institucionais", "topico": "Serviços Públicos Digitais", "chunk_id": "institucional_servicos_publicos_digitais"}, "Serviços Públicos On-line": {"dimensao": "Capacidades Institucionais", "topico": "Serviços Públicos Digitais", "chunk_id": "institucional_servicos_publicos_digitais"}, "Solicitação de Serviços Públicos": {"dimensao": "Capacidades Institucionais", "topico": "Serviços Públicos Digitais", "chunk_id": "institucional_servicos_publicos_digitais"}, "Segurança de Políticas Públicas - Monitoramento": {"dimensao": "Capacidades Institucionais", "topico": "Monitoramento e Transparência", "chunk_id": "institucional_monitoramento_e_transparencia"}, "Percepção dos Serviços Públicos": {"dimensao": "Capacidades Institucionais", "topico": "Monitoramento e Transparência", "chunk_id": "institucional_monitoramento_e_transparencia"}, "Transparência - Monitoramento": {"dimensao": "Capacidades Institucionais", "topico": "Monitoramento e Transparência", "chunk_id": "institucional_monitoramento_e_transparencia"}, "Transparência - Execução Orçamentária e Financeira": {"dimensao": "Capacidades Institucionais", "topico": "Dados e Segurança da Informação", "chunk_id": "institucional_dados_e_seguranca_da_informacao"}, "Transparência dos Dados - Disponibilização": {"dimensao": "Capacidades Institucionais", "topico": "Dados e Segurança da Informação", "chunk_id": "institucional_dados_e_seguranca_da_informacao"}, "Segurança dos Dados - Práticas": {"dimensao": "Capacidades Institucionais", "topico": "Dados e Segurança da Informação", "chunk_id": "institucional_dados_e_seguranca_da_informacao"}}"""
_MAPA_NIVEL_MATURIDADE_JSON = r"""{"Índice da população total com atendimento de água": "N M Indicador", "Índice da população total com atendimento de esgoto": "N M Indicador.1", "Índice da população urbana com atendimento de esgoto": "N M Indicador.2", "Taxa da população coberta com serviço de coleta de resíduos": "N M Indicador.3", "Coleta seletiva de resíduos no município": "N M Indicador.4", "Percentual de domicílios com população vivendo em aglomerados subnormais": "N M Indicador.5", "Assentamentos urbanos precários ": "N M Indicador.6", "Programas e ações habitacionais": "N M Indicador.7", "Serviços regulares de transporte de passageiros": "N M Indicador.8", "Serviços de compartilhamento de viagens": "N M Indicador.9", "Serviço de informações de transporte público em tempo real": "N M Indicador.10", "Serviços e soluções inteligentes para mobilidade urbana": "N M Indicador.11", "Acessibilidade no transporte público": "N M Indicador.12", "Ciclomobilidade na cidade": "N M Indicador.13", "Índice de pavimentação das vias públicas": "N M Indicador.14", "Escala de acesso a banda larga fixa": "N M Indicador.15", "Escala de acesso a banda larga móvel": "N M Indicador.16", "Cobertura de acesso a banda larga móvel por tecnologias 3G e 4G": "N M Indicador.17", "Cobertura de fibra ótica": "N M Indicador.18", "Rede de tecnologia interligando os equipamentos e edifícios públicos": "N M Indicador.19", "Escala de acesso a banda larga fixa de alta velocidade": "N M Indicador.20", "Números de estações rádio base": "N M Indicador.21", "Qualificação profissional e intermediação de mão de obra": "N M Indicador.22", "Inclusão produtiva urbana": "N M Indicador.23", "Acesso a crédito, microcrédito e seguro": "N M Indicador.24", "Geração de trabalho e renda no município": "N M Indicador.25", "Sistema de informação geográfica da prefeitura": "N M Indicador.26", "Centros de comando e controle para gestão da cidade": "N M Indicador.27", "Plataforma integrada de cidade inteligente": "N M Indicador.28", "Serviços no website da prefeitura": "N M Indicador.29", "Dados abertos da gestão municipal": "N M Indicador.30", "Indice de equipamentos de tecnologia disponíveis nas escolas públicas municipais": "N M Indicador.31", "Taxa de analfabetismo": "N M Indicador.32", "Índice de desenvolvimento da educação básica (IDEB) - anos finais": "N M Indicador.33", "Vagas no ensino superior": "N M Indicador.34", "Centros de educação tecnológica": "N M Indicador.35", "Ações de educação para comunidades específicas": "N M Indicador.36", "Taxas de distorção idade-série": "N M Indicador.37", "Percentual de escolas municipais com acesso à internet": "N M Indicador.38", "Computadores para uso dos alunos": "N M Indicador.39", "Estrutura de equipamentos culturais e esportivos": "N M Indicador.40", "Proteção do patrimônio cultural material e imaterial": "N M Indicador.41", "Serviços on-line para promoção de cultura": "N M Indicador.42", "Serviços culturais on-line oferecidos para a população": "N M Indicador.43", "Serviços de telemedicina ou telessaúde": "N M Indicador.44", "Leitos hospitalares na rede pública municipal": "N M Indicador.45", "Médicos disponíveis na rede pública municipal": "N M Indicador.46", "Prontuário eletrônico": "N M Indicador.47", "Serviços on-line de saúde oferecidos aos pacientes": "N M Indicador.48", "Índice de risco e proteção à saúde dos nascidos vivos": "N M Indicador.49", "Mortalidade materna": "N M Indicador.50", "Soluções em monitoramento para a segurança pública": "N M Indicador.51", "Taxa de homicídios": "N M Indicador.52", "Políticas públicas e ações para segurança pública": "N M Indicador.53", "Soluções de tecnologia para gestão e monitoramento de desastres naturais": "N M Indicador.54", "Vulnerabilidade a riscos e desastres naturais": "N M Indicador.55", "Promoção de inclusão digital": "N M Indicador.56", "Cursos de capacitação tecnológica": "N M Indicador.57", "Políticas públicas para mulheres": "N M Indicador.58", "Inclusão social para grupos específicos": "N M Indicador.59", "Formas presenciais para participação pública": "N M Indicador.60", "Formas on-line para participação pública": "N M Indicador.61", "Índice de volume de esgoto coletado": "N M Indicador.62", "Consumo médio per capita de água": "N M Indicador.63", "Soluções inteligentes para gestão na distribuição e consumo de água": "N M Indicador.64", "Índice de perdas na distribuição de água": "N M Indicador.65", "Índice de volume de esgoto tratado": "N M Indicador.66", "Percentual de material recolhido pela coleta seletiva": "N M Indicador.67", "Soluções inteligentes para otimização da coleta de resíduos": "N M Indicador.68", "Proteção e gestão do meio ambiente e áreas verdes do município": "N M Indicador.69", "Soluções em monitoramento de gases de efeito estufa e qualidade do ar": "N M Indicador.70", "Monitoramento da qualidade do ar": "N M Indicador.71", "Soluções inteligentes para gestão do consumo de energia elétrica": "N M Indicador.72", "Soluções para telegestão da iluminação pública": "N M Indicador.73", "Governança Colaborativa - Responsáveis": "N M Indicador.74", "Incorporação de TICs - Planejamento": "N M Indicador.75", "Planejamento Estratégico para Transformação Digital": "N M Indicador.76", "Governança de TI - Práticas": "N M Indicador.77", "Infraestrutura de Hw e Sw - Armazenamento": "N M Indicador.78", "Gestão Integrada de Dados": "N M Indicador.79", "Serviços Públicos On-line": "N M Indicador.80", "Solicitação de Serviços Públicos": "N M Indicador.81", "Segurança de Políticas Públicas - Monitoramento": "N M Indicador.82", "Percepção dos Serviços Públicos": "N M Indicador.83", "Transparência - Monitoramento": "N M Indicador.84", "Transparência - Execução Orçamentária e Financeira": "N M Indicador.85", "Transparência dos Dados - Disponibilização": "N M Indicador.86", "Segurança dos Dados - Práticas": "N M Indicador.87"}"""

def _normalizar_coluna(texto):
    """Mesma normalização aplicada a df.columns em main() (strip, minúsculas,
    remoção de acentos) — garante que as chaves do mapa batam exatamente
    com os nomes de coluna já normalizados da planilha."""
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", errors="ignore").decode("utf-8")
    return texto


_MAPA_INDICADORES_BRUTO = json.loads(_MAPA_INDICADORES_JSON)
_MAPA_NIVEL_MATURIDADE_BRUTO = json.loads(_MAPA_NIVEL_MATURIDADE_JSON)

MAPA_INDICADORES = {_normalizar_coluna(k): v for k, v in _MAPA_INDICADORES_BRUTO.items()}
MAPA_NIVEL_MATURIDADE = {_normalizar_coluna(k): _normalizar_coluna(v) for k, v in _MAPA_NIVEL_MATURIDADE_BRUTO.items()}

# Colunas que NÃO são indicadores individuais: dados cadastrais/socioeconômicos
# de contexto (PIB, IDH, GINI etc.), colunas "N M Indicador..." (nível de
# maturidade de cada indicador — já usadas via MAPA_NIVEL_MATURIDADE) e
# colunas de agregado por tópico/dimensão (ex.: "Água e esgoto" como score
# somado do tópico, "Econômica" como score da dimensão). Sem essa lista,
# o classificador por palavra-chave acabaria tratando esses agregados como
# se fossem indicadores individuais (ex.: a coluna-agregado "Saúde" seria
# capturada pela mesma palavra-chave do tópico "saude").
_COLUNAS_NAO_INDICADORES_JSON = r"""["Cod. Município", "Município", "Estado", "Avaliada", "População total estimada do município", "PIB per capita do município", "PIB Agropecuária", "PIB Indústria", "PIB Serviços", "PIB Adminstração Pública", "População ocupada com vínculo formal", "Índice de desenvolvimento humano do município (IDH-M)", "Capacidade de pagamento dos municípios (CAPAG)", "Índice de GINI da renda domiciliar per capita", "Empregos em TIC", "Empresas de TICs no municipio", "Número de Campus de Institutos e Universidades Federais", "Número de Empresas em Parques Tecnológicos", "Número de Incubadoras credenciadas - Lei de TIC", "Número de Instituições de Ensino e Pesquisa em PD&I - Lei de TIC", "Número de Centros e/ou Institutos de PD&I - Lei de TIC", "Número de Empresas habilitadas - Lei de TIC", "Número de empresas - Lei do Bem PD&I", "Equipe de TI - Tamanho", "Estrutura Organizacional de TIC", "Incorporação de TICs - Áreas Prioritárias", "Governança Tecnológica - Responsáveis", "Governança de TI - Responsável", "Nível de Maturidade do município", "N M Indicador", "N M Indicador.1", "N M Indicador.2", "N M Indicador.3", "N M Indicador.4", "N M Indicador.5", "N M Indicador.6", "N M Indicador.7", "N M Indicador.8", "N M Indicador.9", "N M Indicador.10", "N M Indicador.11", "N M Indicador.12", "N M Indicador.13", "N M Indicador.14", "N M Indicador.15", "N M Indicador.16", "N M Indicador.17", "N M Indicador.18", "N M Indicador.19", "N M Indicador.20", "N M Indicador.21", "N M Indicador.22", "N M Indicador.23", "N M Indicador.24", "N M Indicador.25", "N M Indicador.26", "N M Indicador.27", "N M Indicador.28", "N M Indicador.29", "N M Indicador.30", "N M Indicador.31", "N M Indicador.32", "N M Indicador.33", "N M Indicador.34", "N M Indicador.35", "N M Indicador.36", "N M Indicador.37", "N M Indicador.38", "N M Indicador.39", "N M Indicador.40", "N M Indicador.41", "N M Indicador.42", "N M Indicador.43", "N M Indicador.44", "N M Indicador.45", "N M Indicador.46", "N M Indicador.47", "N M Indicador.48", "N M Indicador.49", "N M Indicador.50", "N M Indicador.51", "N M Indicador.52", "N M Indicador.53", "N M Indicador.54", "N M Indicador.55", "N M Indicador.56", "N M Indicador.57", "N M Indicador.58", "N M Indicador.59", "N M Indicador.60", "N M Indicador.61", "N M Indicador.62", "N M Indicador.63", "N M Indicador.64", "N M Indicador.65", "N M Indicador.66", "N M Indicador.67", "N M Indicador.68", "N M Indicador.69", "N M Indicador.70", "N M Indicador.71", "N M Indicador.72", "N M Indicador.73", "N M Indicador.74", "N M Indicador.75", "N M Indicador.76", "N M Indicador.77", "N M Indicador.78", "N M Indicador.79", "N M Indicador.80", "N M Indicador.81", "N M Indicador.82", "N M Indicador.83", "N M Indicador.84", "N M Indicador.85", "N M Indicador.86", "N M Indicador.87", "Econômica", "Sociocultural", "Meio Ambiente", "Capacidades Institucionais", "Água e esgoto", "Resíduos sólidos", "Habitação", "Transporte", "Urbanização vias públicas", "Infraestrutura de conectividade", "Inovação", "Sistemas e tecnologia para gestão urbana", "Serviços on-line da prefeitura", "Dados abertos", "Educação", "Cultura", "Saúde", "Segurança Pública", "Gestão de desastres", "Inclusão digital", "Inclusão social", "Participação pública", "Água e esgoto.1", "Resíduos sólidos.1", "Áreas verdes", "Qualidade do ar", "Energia", "Estratégia", "Infraestrutura de Hw e Sw", "Serviços e aplicações", "Monitoramento", "Dados abertos.1"]"""
COLUNAS_NAO_INDICADORES = {_normalizar_coluna(c) for c in json.loads(_COLUNAS_NAO_INDICADORES_JSON)}

# =====================================================
# DADOS CONTEXTUAIS (subconjunto de COLUNAS_NAO_INDICADORES)
# =====================================================
# Dentro de COLUNAS_NAO_INDICADORES existem dois grupos bem distintos:
#   1) metadado puro / agregado da própria metodologia (Cod. Município,
#      Município, Estado, Avaliada, as 88 colunas "N M Indicador...", o
#      "Nível de Maturidade do município" e os scores agregados por
#      tópico/dimensão, ex.: "Água e esgoto", "Educação", "Econômica");
#   2) dado cadastral/socioeconômico e institucional de contexto (PIB,
#      IDH-M, GINI, CAPAG, empregos/empresas de TIC, universidades
#      federais, instituições de PD&I etc.) — esse é o grupo tratado
#      abaixo como DADOS CONTEXTUAIS.
#
# Os dados contextuais:
#   - SÃO usados para complementar a Análise Geral e a análise da
#     dimensão à qual estão associados (ver obter_dados_contextuais);
#   - NÃO são indicadores da metodologia (continuam fora de
#     indicadores_por_dimensao / MAPA_INDICADORES);
#   - NÃO entram no cálculo de nível de maturidade nem na ordenação de
#     prioridades (obter_nivel_indicador / selecionar_chunks_dimensao
#     seguem operando apenas sobre indicadores);
#   - NÃO devem gerar sugestões diretamente — as sugestões continuam
#     fundamentadas exclusivamente nos indicadores de cada dimensão.
#
# O restante de COLUNAS_NAO_INDICADORES (metadado/agregado, grupo 1)
# permanece apenas excluído, como já ocorria antes desta mudança.
#
# Observação sobre ambiguidade: colunas de instituições de pesquisa e
# inovação (PD&I) foram associadas à Dimensão Econômica, por estarem
# alinhadas ao tópico "Inovação" dessa dimensão — em outro contexto
# poderiam igualmente ser lidas como Capacidades Institucionais.
_DADOS_CONTEXTUAIS_JSON = r"""{
    "PIB per capita do município": "Econômica",
    "PIB Agropecuária": "Econômica",
    "PIB Indústria": "Econômica",
    "PIB Serviços": "Econômica",
    "PIB Adminstração Pública": "Econômica",
    "População ocupada com vínculo formal": "Econômica",
    "Capacidade de pagamento dos municípios (CAPAG)": "Econômica",
    "Empregos em TIC": "Econômica",
    "Empresas de TICs no municipio": "Econômica",
    "Número de Empresas em Parques Tecnológicos": "Econômica",
    "Número de Incubadoras credenciadas - Lei de TIC": "Econômica",
    "Número de Instituições de Ensino e Pesquisa em PD&I - Lei de TIC": "Econômica",
    "Número de Centros e/ou Institutos de PD&I - Lei de TIC": "Econômica",
    "Número de Empresas habilitadas - Lei de TIC": "Econômica",
    "Número de empresas - Lei do Bem PD&I": "Econômica",
    "Índice de desenvolvimento humano do município (IDH-M)": "Sociocultural",
    "Índice de GINI da renda domiciliar per capita": "Sociocultural",
    "Número de Campus de Institutos e Universidades Federais": "Sociocultural",
    "Equipe de TI - Tamanho": "Capacidades Institucionais",
    "Estrutura Organizacional de TIC": "Capacidades Institucionais",
    "Incorporação de TICs - Áreas Prioritárias": "Capacidades Institucionais",
    "Governança Tecnológica - Responsáveis": "Capacidades Institucionais",
    "Governança de TI - Responsável": "Capacidades Institucionais"
}"""

_DADOS_CONTEXTUAIS_BRUTO = json.loads(_DADOS_CONTEXTUAIS_JSON)

# chave normalizada (bate com df.columns já normalizado) -> {dimensao, rotulo}
# "rotulo" preserva o nome original, legível, para uso nos prompts.
DADOS_CONTEXTUAIS_MAPA = {
    _normalizar_coluna(coluna): {"dimensao": dimensao, "rotulo": coluna}
    for coluna, dimensao in _DADOS_CONTEXTUAIS_BRUTO.items()
}

INDICADORES_NAO_CLASSIFICADOS = []  # preenchida em tempo de execução


def classificar_indicador(nome_indicador):
    """Retorna {dimensao, topico, chunk_id} para um indicador, ou None.

    Ordem de resolução:
    1) correspondência exata em MAPA_INDICADORES;
    2) fallback por palavra-chave (nome do indicador x PALAVRAS_CHAVE_TOPICO)
       — útil para indicadores novos, ainda não presentes no mapa;
    3) fallback por similaridade semântica de embeddings (nome do indicador
       x tópicos, via INDICE_TOPICOS) — útil quando o indicador usa
       vocabulário diferente do das palavras-chave cadastradas;
    4) se nada for encontrado, registra em INDICADORES_NAO_CLASSIFICADOS.
    """
    if nome_indicador in COLUNAS_NAO_INDICADORES:
        return None  # metadado/agregado conhecido — não é um indicador setorial

    if nome_indicador in MAPA_INDICADORES:
        return MAPA_INDICADORES[nome_indicador]

    nome_norm = _normalizar(nome_indicador)
    melhor, melhor_score = None, 0
    for dimensao, info in DIMENSOES.items():
        for topico in info["topicos"]:
            palavras = PALAVRAS_CHAVE_TOPICO.get(topico, [])
            score = sum(1 for p in palavras if _normalizar(p) in nome_norm)
            if score > melhor_score:
                melhor_score = score
                melhor = {"dimensao": dimensao, "topico": topico.replace("_", " ").title(),
                          "chunk_id": f"{info['prefixo']}_{topico}"}

    if melhor:
        return melhor

    classificacao_embedding = classificar_indicador_por_embedding(nome_indicador)
    if classificacao_embedding:
        return classificacao_embedding

    INDICADORES_NAO_CLASSIFICADOS.append(nome_indicador)
    return None


In [7]:
# =====================================================
# ÍNDICE FAISS DOS TÓPICOS (fallback semântico de classificação)
# =====================================================
# Usado por classificar_indicador quando um indicador não bate por nome
# exato (MAPA_INDICADORES) nem por palavra-chave (PALAVRAS_CHAVE_TOPICO):
# em vez de descartar o indicador direto para INDICADORES_NAO_CLASSIFICADOS,
# tenta encontrar por similaridade de embeddings o tópico mais próximo.
LIMIAR_RELEVANCIA_TOPICO = 0.75  # 0-1; ajuste empiricamente conforme os resultados


def _construir_indice_topicos():
    documentos = []
    for dimensao, info in DIMENSOES.items():
        for topico in info["topicos"]:
            palavras = PALAVRAS_CHAVE_TOPICO.get(topico, [])
            texto = f"{topico.replace('_', ' ')}: {', '.join(palavras)}"
            documentos.append(
                LCDocument(
                    page_content=texto,
                    metadata={
                        "dimensao": dimensao,
                        "topico": topico.replace("_", " ").title(),
                        "chunk_id": f"{info['prefixo']}_{topico}",
                    },
                )
            )
    return FAISS.from_documents(documentos, embeddings)


def carregar_ou_construir_indice_topicos():
    if os.path.exists(FAISS_INDEX_TOPICOS_PATH):
        return FAISS.load_local(
            FAISS_INDEX_TOPICOS_PATH, embeddings, allow_dangerous_deserialization=True
        )
    indice = _construir_indice_topicos()
    indice.save_local(FAISS_INDEX_TOPICOS_PATH)
    return indice


INDICE_TOPICOS = carregar_ou_construir_indice_topicos()
print("🔹 Índice FAISS de tópicos pronto.")


def classificar_indicador_por_embedding(nome_indicador):
    """Fallback semântico: retorna {dimensao, topico, chunk_id} do tópico
    mais próximo do nome do indicador por similaridade de embeddings, ou
    None se a melhor correspondência ficar abaixo de LIMIAR_RELEVANCIA_TOPICO."""
    resultados = INDICE_TOPICOS.similarity_search_with_relevance_scores(nome_indicador, k=1)
    if not resultados:
        return None
    doc, score = resultados[0]
    if score < LIMIAR_RELEVANCIA_TOPICO:
        return None
    return {
        "dimensao": doc.metadata["dimensao"],
        "topico": doc.metadata["topico"],
        "chunk_id": doc.metadata["chunk_id"],
    }


🔹 Índice FAISS de tópicos pronto.


In [8]:
# =====================================================
# CARTA COMPLETA — CHUNKING HIERÁRQUICO + FAISS GRANULAR
# =====================================================
# Corpus: 3 conceitos + 5 princípios + 6 diretrizes + 8 contextos
# de objetivos + 163 recomendações = 185 documentos.
#
# A extração foi feita da camada textual com posição dos blocos (PyMuPDF),
# não OCR. A coluna lateral de atores foi removida do texto e preservada
# apenas em metadados.



# =====================================================
# CONTROLE DE QUOTA DO EMBEDDING — FREE TIER
# =====================================================
# O Gemini Embedding no Free Tier pode limitar a quantidade de requisições
# por minuto. Como o corpus tem 185 documentos, NÃO construímos o índice
# inteiro em uma única chamada.
#
# 45 documentos por lote deixa margem para outras chamadas de embedding
# feitas pelo notebook no mesmo minuto.
LOTE_EMBEDDING_CARTA_COMPLETA = 45

# A pausa precisa atravessar a janela de 1 minuto. O RetryInfo do Google
# pode sugerir poucos segundos para a requisição que falhou, mas repetir
# imediatamente o lote inteiro pode atingir a mesma cota novamente.
PAUSA_ENTRE_LOTES_EMBEDDING = 65

# Em caso de 429 inesperado, espera e tenta novamente o mesmo lote.
MAX_TENTATIVAS_LOTE_EMBEDDING = 4

_CARTA_REVISADA_SNAPSHOT_JSON = '{"fonte":"Carta Brasileira para Cidades Inteligentes — Edição Revisada","arquivo_origem":"carta_brasileira_cidades_inteligentes (1).pdf","pipeline_extracao":"posicional_pymupdf_v1","total_recomendacoes":163,"contagem_por_objetivo":{"1":30,"2":24,"3":22,"4":23,"5":24,"6":12,"7":14,"8":14},"conceitos":[{"doc_id":"conceito_cidades_inteligentes","tipo":"conceito","titulo":"Conceito brasileiro de cidades inteligentes","texto":"CIDADES INTELIGENTES São cidades comprometidas com o desenvolvimento urbano e a transformação digital sustentáveis, em seus aspectos econômico, ambiental e sociocultural, que atuam de forma planejada, inovadora, inclusiva e em rede, promovem o letramento digital, a governança e a gestão colaborativas e utilizam tecnologias para solucionar problemas concretos, criar oportunidades, oferecer serviços com eficiência, reduzir desigualdades, aumentar a resiliência e melhorar a qualidade de vida de todas as pessoas, garantindo o uso seguro e responsável de dados e das tecnologias da informação e comunicação.","pagina_inicio":28,"pagina_fim":28},{"doc_id":"conceito_transformacao_digital_sustentavel","tipo":"conceito","titulo":"Transformação digital sustentável","texto":"TRANSFORMAÇÃO DIGITAL SUSTENTÁVEL é o processo de adoção responsável de tecnologias da informação e comunicação, baseado na ética digital e orientado para o bem comum, compreendendo a segurança cibernética e a transparência na utilização de dados, informações, algoritmos e dispositivos, a disponibilização de dados e códigos abertos, acessíveis a todas as pessoas, a proteção geral de dados pessoais, o letramento e a inclusão digitais, de forma adequada e respeitosa em relação às características socioculturais, econômicas, urbanas, ambientais e político-institucionais específicas de cada território, à conservação dos recursos naturais e das condições de saúde das pessoas.","pagina_inicio":29,"pagina_fim":29},{"doc_id":"conceito_desenvolvimento_urbano_sustentavel","tipo":"conceito","titulo":"Desenvolvimento urbano sustentável","texto":"DESENVOLVIMENTO URBANO SUSTENTÁVEL é o processo de ocupação urbana orientada para o bem comum e para a redução de desigualdades, que equilibra as necessidades sociais, dinamiza a cultura, valoriza e fortalece identidades, utiliza de forma responsável os recursos naturais, tecnológicos, urbanos e financeiros, e promove o desenvolvimento econômico local, impulsionando a criação de oportunidades na diversidade e a inclusão social, produtiva e espacial de todas as pessoas, da presente e das futuras gerações, por meio da distribuição equitativa de infraestrutura, espaços públicos, bens e serviços urbanos e do adequado ordenamento do uso e da ocupação do solo em diferentes contextos e escalas territoriais, com respeito a pactos sociopolíticos estabelecidos em arenas democráticas de governança colaborativa.","pagina_inicio":29,"pagina_fim":29}],"principios":[{"titulo":"RESPEITO À DIVERSIDADE TERRITORIAL BRASILEIRA, EM SEUS ASPECTOS CULTURAIS, SOCIAIS, ECONÔMICOS E AMBIENTAIS","pagina":30,"texto":"O processo de transformação digital precisa ser adequado às realidades locais. Essa adequação deve levar em conta as áreas remotas e as diferenças entre áreas rurais e urbanas dos municípios. Deve seguir as tipologias da Política Nacional de Desenvolvimento Urbano (PNDU)."},{"titulo":"VISÃO SISTÊMICA DA CIDADE E DA TRANSFORMAÇÃO DIGITAL.","pagina":30,"texto":"A transformação digital também é uma transformação urbana. A cidade é um sistema complexo, dinâmico e vivo, que reflete, reage e materializa questões culturais, sociais, ambientais e econômicas."},{"titulo":"INTEGRAÇÃO DOS CAMPOS URBANO E DIGITAL","pagina":30,"texto":"A articulação entre setores e disciplinas científicas combina tecnologias digitais e sociais, inclusive de forma experimental. O objetivo é desenvolver novos processos para melhorar a qualidade de vida nas cidades."},{"titulo":"CONSERVAÇÃO DO MEIO AMBIENTE","pagina":30,"texto":"Inclui: uso sustentável dos recursos naturais; combate e reversão de práticas de degradação do meio ambiente; reconhecimento e adoção de soluções baseadas natureza, e; reconhecimento e adoção de outras abordagens ambientais inovadoras nas matrizes de desenvolvimento."},{"titulo":"INTERESSE PÚBLICO ACIMA DE TUDO","pagina":30,"texto":"As ações de cidades inteligentes devem respeitar os princípios que a Constituição Federal define para a Administração Pública e para a política urbana. No caso da Administração Pública: legalidade, impessoalidade, moralidade, publicidade e eficiência. No caso da política urbana: a cidade e a propriedade devem atender ao bem coletivo e cumprir sua função social."}],"diretrizes":[{"titulo":"PROMOVER O DESENVOLVIMENTO URBANO SUSTENTÁVEL","pagina":31,"texto":"Agir conforme a perspectiva de desenvolvimento urbano sustentável que está na legislação, nas políticas brasileiras e em acordos internacionais."},{"titulo":"CONSTRUIR RESPOSTAS PARA OS PROBLEMAS LOCAIS","pagina":31,"texto":"Avaliar e promover ações levando em conta o potencial que elas têm de responder aos desafios locais, adequando-as ao estágio tecnológico do município."},{"titulo":"PROMOVER EDUCAÇÃO E INCLUSÃO DIGITAL","pagina":31,"texto":"Impulsionar e promover ações que estimulem a formação cidadã e o letramento digital, de forma contínua. As ações devem atender pessoas de todas as idades, gêneros, raças e classes sociais, fortalecendo a sua autonomia."},{"titulo":"ESTIMULAR O PROTAGONISMO COMUNITÁRIO","pagina":31,"texto":"Estimular e garantir o envolvimento de pessoas de todas as idades, gêneros, raças e classes sociais e dos coletivos locais, inclusive povos e comunidades tradicionais."},{"titulo":"COLABORAR E ESTABELECER PARCERIAS","pagina":31,"texto":"Realizar ações de cooperação entre setores público, privado, organizações da sociedade civil e instituições de ensino e pesquisa."},{"titulo":"DECIDIR COM BASE EM EVIDÊNCIAS","pagina":31,"texto":"Usar dados e sistemas de forma responsável, transparente e compartilhada."}],"objetivos":{"1":{"objetivo":1,"pagina_inicio":32,"pagina_fim":33,"texto_completo":"Integrar a transformação digital nas políticas, programas e ações de desenvolvimento urbano sustentável, respeitando as diversidades e considerando as desigualdades presentes nas cidades brasileiras Contexto › Para reduzir desigualdades socioespaciais, é preciso considerar o desenvolvimento territorial a partir de uma visão ampla. Essa visão deve levar em conta vários aspectos, especialmente a localização, a disponibilização e o acesso a recursos, infraestruturas, bens e serviços essenciais, educação, cultura e informação. A transformação digital traz oportunidades para compreender melhor e enfrentar os problemas urbanos brasileiros, que são históricos. Mas ações de tecnologia sem direcionamento podem até aumentar desigualdades antigas, como a falta ou deficiência no acesso a serviços urbanos básicos. Governos e sociedade precisam agir para que a tecnologia atenda às necessidades reais das cidades. Iniciativas e soluções digitais devem estar alinhadas com uma visão estratégica de desenvolvimento urbano sustentável e de qualidade de vida. Além disso devem estar sintonizadas com a grande diversidade brasileira. Esse processo requer que a sociedade e as instituições locais se fortaleçam para assumir o protagonismo na adaptação da transformação digital às suas realidades. Para isso, elas devem adequar políticas, programas e ações de desenvolvimento urbano ao novo contexto da transformação digital. Devem aperfeiçoar infraestruturas, ferramentas e sistemas digitais para a prestação de serviços públicos de qualidade.","titulo":"Integrar a transformação digital nas políticas, programas e ações de desenvolvimento urbano sustentável, respeitando as diversidades e considerando as desigualdades presentes nas cidades brasileiras","contexto":"Para reduzir desigualdades socioespaciais, é preciso considerar o desenvolvimento territorial a partir de uma visão ampla. Essa visão deve levar em conta vários aspectos, especialmente a localização, a disponibilização e o acesso a recursos, infraestruturas, bens e serviços essenciais, educação, cultura e informação. A transformação digital traz oportunidades para compreender melhor e enfrentar os problemas urbanos brasileiros, que são históricos. Mas ações de tecnologia sem direcionamento podem até aumentar desigualdades antigas, como a falta ou deficiência no acesso a serviços urbanos básicos. Governos e sociedade precisam agir para que a tecnologia atenda às necessidades reais das cidades. Iniciativas e soluções digitais devem estar alinhadas com uma visão estratégica de desenvolvimento urbano sustentável e de qualidade de vida. Além disso devem estar sintonizadas com a grande diversidade brasileira. Esse processo requer que a sociedade e as instituições locais se fortaleçam para assumir o protagonismo na adaptação da transformação digital às suas realidades. Para isso, elas devem adequar políticas, programas e ações de desenvolvimento urbano ao novo contexto da transformação digital. Devem aperfeiçoar infraestruturas, ferramentas e sistemas digitais para a prestação de serviços públicos de qualidade."},"2":{"objetivo":2,"pagina_inicio":33,"pagina_fim":33,"texto_completo":"Prover acesso equitativo à internet de qualidade para todas as pessoas Contexto › Integrar o urbano e o digital nas políticas públicas e nos instrumentos de ordenamento territorial é importante, mas essa ação deve vir acompanhada de conectividade. O desenvolvimento sustentável depende de todas as pessoas acessarem internet e ferramentas digitais de qualidade. Uma boa conectividade digital determina a inclusão social e produtiva e a justa distribuição de oportunidades. Em função disso, governos e iniciativa privada devem conhecer os territórios onde o acesso é precário e corrigir essa distorção.","titulo":"Prover acesso equitativo à internet de qualidade para todas as pessoas","contexto":"Integrar o urbano e o digital nas políticas públicas e nos instrumentos de ordenamento territorial é importante, mas essa ação deve vir acompanhada de conectividade. O desenvolvimento sustentável depende de todas as pessoas acessarem internet e ferramentas digitais de qualidade. Uma boa conectividade digital determina a inclusão social e produtiva e a justa distribuição de oportunidades. Em função disso, governos e iniciativa privada devem conhecer os territórios onde o acesso é precário e corrigir essa distorção."},"3":{"objetivo":3,"pagina_inicio":33,"pagina_fim":34,"texto_completo":"Estabelecer sistemas de governança de dados e de tecnologias, com transparência, segurança e privacidade Contexto › Políticas públicas e conectividade são elementos básicos, mas insuficientes para equidade (distribuição justa, capaz de atender necessidades diferentes de todas as pessoas) de oportunidades no contexto da transformação digital. É preciso estruturar sistemas de governança de dados e de TICs (tecnologias de informação e comunicação) adequados a cada realidade. Somente a partir desses sistemas será possível integrar infraestrutura, sistemas, ferramentas e soluções digitais no desenvolvimento urbano de todas as cidades. Diferentes governos e setores da sociedade devem cooperar para os sistemas funcionarem de forma integrada, responsável e inovadora. Com segurança cibernética e garantia de privacidade pessoal. Devem cooperar para oferecer um ambiente de ética digital que assegure dados compartilhados e abertos, sempre que possível, e que garanta proteção jurídica às pessoas.","titulo":"Estabelecer sistemas de governança de dados e de tecnologias, com transparência, segurança e privacidade","contexto":"Políticas públicas e conectividade são elementos básicos, mas insuficientes para equidade (distribuição justa, capaz de atender necessidades diferentes de todas as pessoas) de oportunidades no contexto da transformação digital. É preciso estruturar sistemas de governança de dados e de TICs (tecnologias de informação e comunicação) adequados a cada realidade. Somente a partir desses sistemas será possível integrar infraestrutura, sistemas, ferramentas e soluções digitais no desenvolvimento urbano de todas as cidades. Diferentes governos e setores da sociedade devem cooperar para os sistemas funcionarem de forma integrada, responsável e inovadora. Com segurança cibernética e garantia de privacidade pessoal. Devem cooperar para oferecer um ambiente de ética digital que assegure dados compartilhados e abertos, sempre que possível, e que garanta proteção jurídica às pessoas."},"4":{"objetivo":4,"pagina_inicio":34,"pagina_fim":34,"texto_completo":"Adotar modelos inovadores e inclusivos de governança urbana e fortalecer o papel do poder público como gestor de impactos da transformação digital nas cidades Contexto › A governança de informação tratada no objetivo anterior faz parte de uma governança urbana mais ampla, que estimula a colaboração e cria inteligência territorial (baseada em sistemas e informações que orientam decisões estratégicas baseadas em evidências para planejar, executar, gerenciar e monitorar ações no território). Pessoas e instituições precisam conversar, discutir os problemas e construir soluções que atendam à coletividade. Nesse sentido, a transformação digital pode melhorar os tradicionais modelos de participação, tornando-os mais inovadores e inclusivos. Pode-se criar ambientes que aproximem e reconfigurem a relação entre Estado, setores da sociedade. Ou que aproximem e reconfigurem a relação entre setores urbanos (como habitação, saneamento e mobilidade) e entre os entes da federação (União, Estados, Distrito Federal e Municípios). Uma governança inovadora e inclusiva estimula a colaboração, pois esta é uma forma de identificar problemas urbanos reais com base em evidências e desenvolver soluções. O poder público municipal é protagonista da execução da política urbana, um dos guardiões do interesse coletivo. Daí o seu papel estratégico para promover e facilitar as ações de governança urbana. E deve coordenar os processos que decidem sobre promoção, regulamentação ou desestímulo de instrumentos surgidos com a transformação digital, tais como dados, sistemas de informação e modelos de negócios.","titulo":"Adotar modelos inovadores e inclusivos de governança urbana e fortalecer o papel do poder público como gestor de impactos da transformação digital nas cidades","contexto":"A governança de informação tratada no objetivo anterior faz parte de uma governança urbana mais ampla, que estimula a colaboração e cria inteligência territorial (baseada em sistemas e informações que orientam decisões estratégicas baseadas em evidências para planejar, executar, gerenciar e monitorar ações no território). Pessoas e instituições precisam conversar, discutir os problemas e construir soluções que atendam à coletividade. Nesse sentido, a transformação digital pode melhorar os tradicionais modelos de participação, tornando-os mais inovadores e inclusivos. Pode-se criar ambientes que aproximem e reconfigurem a relação entre Estado, setores da sociedade. Ou que aproximem e reconfigurem a relação entre setores urbanos (como habitação, saneamento e mobilidade) e entre os entes da federação (União, Estados, Distrito Federal e Municípios). Uma governança inovadora e inclusiva estimula a colaboração, pois esta é uma forma de identificar problemas urbanos reais com base em evidências e desenvolver soluções. O poder público municipal é protagonista da execução da política urbana, um dos guardiões do interesse coletivo. Daí o seu papel estratégico para promover e facilitar as ações de governança urbana. E deve coordenar os processos que decidem sobre promoção, regulamentação ou desestímulo de instrumentos surgidos com a transformação digital, tais como dados, sistemas de informação e modelos de negócios."},"5":{"objetivo":5,"pagina_inicio":34,"pagina_fim":35,"texto_completo":"Fomentar o desenvolvimento econômico local no contexto da transformação digital Contexto › Uma governança bem estruturada, colaborativa e inclusiva torna as cidades mais habitáveis e fortalece a economia local. O mesmo ocorre quando as decisões são tomadas com base em dados e evidências científicas. A transformação digital pode gerar valor, emprego e renda para as pessoas das cidades. A economia do compartilhamento, a economia criativa e a economia circular podem potencializar essas oportunidades. Mas é indispensável que diferentes setores e pessoas se articulem para evitar que uma transformação digital mal conduzida cause mais desigualdade social.","titulo":"Fomentar o desenvolvimento econômico local no contexto da transformação digital","contexto":"Uma governança bem estruturada, colaborativa e inclusiva torna as cidades mais habitáveis e fortalece a economia local. O mesmo ocorre quando as decisões são tomadas com base em dados e evidências científicas. A transformação digital pode gerar valor, emprego e renda para as pessoas das cidades. A economia do compartilhamento, a economia criativa e a economia circular podem potencializar essas oportunidades. Mas é indispensável que diferentes setores e pessoas se articulem para evitar que uma transformação digital mal conduzida cause mais desigualdade social."},"6":{"objetivo":6,"pagina_inicio":35,"pagina_fim":35,"texto_completo":"Estimular modelos e instrumentos de financiamento do desenvolvimento urbano sustentável no contexto da transformação digital Contexto › Recursos financeiros viabilizam, aceleram e potencializam os processos de desenvolvimento econômico e urbano sustentáveis. Os recursos são necessários para implementar ambientes de estímulo à inovação, à pesquisa e à implantação de infraestruturas. Estado e sociedade devem trabalhar juntos, seguindo na mesma direção. A ação conjunta deve incluir bancos públicos, investidores privados, instituições financeiras e de fomento, agências de apoio à pesquisa e inovação. O trabalho em colaboração irá identificar, sistematizar, criar e disponibilizar instrumentos, linhas diversificadas de financiamento e soluções de autofinanciamento da transformação digital. Todas as ações devem estar associadas ao desenvolvimento urbano sustentável.","titulo":"Estimular modelos e instrumentos de financiamento do desenvolvimento urbano sustentável no contexto da transformação digital","contexto":"Recursos financeiros viabilizam, aceleram e potencializam os processos de desenvolvimento econômico e urbano sustentáveis. Os recursos são necessários para implementar ambientes de estímulo à inovação, à pesquisa e à implantação de infraestruturas. Estado e sociedade devem trabalhar juntos, seguindo na mesma direção. A ação conjunta deve incluir bancos públicos, investidores privados, instituições financeiras e de fomento, agências de apoio à pesquisa e inovação. O trabalho em colaboração irá identificar, sistematizar, criar e disponibilizar instrumentos, linhas diversificadas de financiamento e soluções de autofinanciamento da transformação digital. Todas as ações devem estar associadas ao desenvolvimento urbano sustentável."},"7":{"objetivo":7,"pagina_inicio":35,"pagina_fim":35,"texto_completo":"Fomentar um movimento massivo e inovador de educação e comunicação públicas para maior engajamento da sociedade no processo de transformação digital e de desenvolvimento urbano sustentáveis Contexto › Por outro lado, pessoas, coletivos e organizações devem fazer a transição de usuários passivos para agentes da transformação. Devem ser agentes conscientes e criadores das próprias realidades. Isso requer novas capacidades, habilidades e atitudes. Trata-se de uma tarefa coletiva e desafiadora. Logo, ela deve ser apoiada por um movimento educativo massivo sobre a transformação digital nas cidades. Essa tarefa também requer um processo de comunicação qualificado para engajar, sincronizar, coordenar e articular distintos agentes públicos e privados em torno dos objetivos da Carta. Entre os agentes, devem constar organizações da sociedade civil, veículos de comunicação, instituições de ensino e pesquisa.","titulo":"Fomentar um movimento massivo e inovador de educação e comunicação públicas para maior engajamento da sociedade no processo de transformação digital e de desenvolvimento urbano sustentáveis","contexto":"Por outro lado, pessoas, coletivos e organizações devem fazer a transição de usuários passivos para agentes da transformação. Devem ser agentes conscientes e criadores das próprias realidades. Isso requer novas capacidades, habilidades e atitudes. Trata-se de uma tarefa coletiva e desafiadora. Logo, ela deve ser apoiada por um movimento educativo massivo sobre a transformação digital nas cidades. Essa tarefa também requer um processo de comunicação qualificado para engajar, sincronizar, coordenar e articular distintos agentes públicos e privados em torno dos objetivos da Carta. Entre os agentes, devem constar organizações da sociedade civil, veículos de comunicação, instituições de ensino e pesquisa."},"8":{"objetivo":8,"pagina_inicio":36,"pagina_fim":36,"texto_completo":"Construir meios para compreender e avaliar, de forma contínua e sistêmica, os impactos da transformação digital nas cidades Contexto › Finalmente, precisamos assimilar e aprender com as transformações enquanto elas acontecem, pois são fatos novos, dinâmicos, inéditos e ainda pouco estudados. É necessário compreender e avaliar os impactos sistêmicos (impactos no nosso sistema social, ambiental, econômico, político) que o processo de transformação digital causa nas cidades. Isso deve ser feito de forma contínua e estruturada, a partir de uma abordagem complexa e sistêmica. A avaliação dos impactos é uma tarefa essencial para identificar novos desafios e corrigir os rumos desta agenda ao longo da sua implementação. Tamanha tarefa só será possível com a união de diferentes pessoas e com a valorização dos saberes locais e comunitários.","titulo":"Construir meios para compreender e avaliar, de forma contínua e sistêmica, os impactos da transformação digital nas cidades","contexto":"Finalmente, precisamos assimilar e aprender com as transformações enquanto elas acontecem, pois são fatos novos, dinâmicos, inéditos e ainda pouco estudados. É necessário compreender e avaliar os impactos sistêmicos (impactos no nosso sistema social, ambiental, econômico, político) que o processo de transformação digital causa nas cidades. Isso deve ser feito de forma contínua e estruturada, a partir de uma abordagem complexa e sistêmica. A avaliação dos impactos é uma tarefa essencial para identificar novos desafios e corrigir os rumos desta agenda ao longo da sua implementação. Tamanha tarefa só será possível com a união de diferentes pessoas e com a valorização dos saberes locais e comunitários."}},"recomendacoes":[{"recomendacao_id":"1.1","objetivo":1,"pagina_inicio":37,"pagina_fim":37,"publicos":["GF","GE","GM","AR","ET"],"texto":"Desigualdade digital e política urbana: Usar o acesso à internet de qualidade como um indicador de desigualdade socioespacial na política urbana. Reconhecer pelo indicador que há um déficit de conectividade que deve ser enfrentado em políticas, programas, projetos e ações de desenvolvimento urbano sustentável e de telecomunicações. Essas iniciativas devem estar alinhadas com o Plano Estratégico da Anatel 2015-2024 e com a Estratégia Brasileira para a Transformação Digital E-digital."},{"recomendacao_id":"1.1.1","objetivo":1,"pagina_inicio":38,"pagina_fim":38,"publicos":["GF","GE","GM","CIH","AR","ET","SP"],"texto":"Infraestrutura digital como infraestrutura urbana básica: Planejar e implementar a infraestrutura digital como parte da infraestrutura básica da cidade. Essas ações devem ser facilitadas inclusive por meio de alteração à lei do parcelamento do solo urbano (Lei no 6.766/1979) e de outras normas gerais de política urbana. Tais alterações devem convergir com normas e diretrizes da União relativas aos serviços de telecomunicações e sua respectiva infraestrutura de suporte. A integração da infraestrutura digital na infraestrutura urbana básica contribuirá para alcançar os objetivos da Estratégia Brasileira para a Transformação Digital (Decreto no 9.319/2018 - E-Digital)."},{"recomendacao_id":"1.1.2","objetivo":1,"pagina_inicio":38,"pagina_fim":38,"publicos":["GF","GE","GM","SP","IEP","IFF","OSC"],"texto":"Informações sobre exclusão digital: Entender melhor os fatores associados à exclusão digital. Exemplos: (1) compreender quais são as condições de conectividade dos grupos vulneráveis; e (2) compreender quais são as condições de conexão em cada localização. Para isso, usar dados georreferenciados (com localização geográfica) separados por critérios como renda, raça, gênero, escolaridade e idade. Incluir análises específicas para as pessoas com deficiência. O uso e tratamento dos dados deve respeitar a legislação sobre proteção de dados pessoais (LGPD)."},{"recomendacao_id":"1.2","objetivo":1,"pagina_inicio":38,"pagina_fim":38,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Visão de território para o desenvolvimento urbano sustentável:"},{"recomendacao_id":"1.2.1","objetivo":1,"pagina_inicio":38,"pagina_fim":39,"publicos":["GF","CIV","IEP","IFF"],"texto":"Tipologias urbanas: Estabelecer tipologias (categorias) de território que apoiem a compreensão do urbano no Brasil. Esse trabalho deve ser feito no processo de formulação da Política Nacional de Desenvolvimento Urbano (PNDU). Deve compreender o território a partir de diferentes níveis: municipal, supramunicipal (agrupamento de municípios) e regional. As tipologias também devem se adequar à diversidade territorial do país. O objetivo é orientar agendas, programas e iniciativas para o desenvolvimento urbano sustentável, inclusive de cidades inteligentes, nos três níveis (municipal, supramunicipal e regional)."},{"recomendacao_id":"1.2.2","objetivo":1,"pagina_inicio":39,"pagina_fim":39,"publicos":["GF","GE","GM","CIV","OSC"],"texto":"Instrumentos e metodologias para a diversidade territorial: Desenvolver e adaptar instrumentos e metodologias de informação, planejamento, gestão e governança para o desenvolvimento urbano sustentável, considerando diferentes graus de complexidade. Esses instrumentos e metodologias devem ser adequados às tipologias (categorias de territórios) da Política Nacional de Desenvolvimento (PNDU). Devem considerar a diversidade territorial das cidades brasileiras. Devem ser fáceis de implementar, considerando diferentes capacidades presentes no nível local."},{"recomendacao_id":"1.2.3","objetivo":1,"pagina_inicio":39,"pagina_fim":39,"publicos":["GF","GE","GM","CIV","OSC"],"texto":"Visão de contexto: Estimular a atuação local com visão de contexto, disponibilizando ferramentas para facilitar que os municípios percebam seus próprios contextos e inserções regionais. O objetivo é qualificar o planejamento e a gestão integrada de suas áreas urbanas, rurais e naturais. Deve haver articulação com outros municípios e demais entes federados (União, Estados, Municípios e Distrito Federal). Essas ações devem estar em linha com a Política Nacional de Desenvolvimento Regional (PNDR) e com a Política Nacional de Desenvolvimento Urbano (PNDU)."},{"recomendacao_id":"1.2.4","objetivo":1,"pagina_inicio":40,"pagina_fim":40,"publicos":["GF","GE","GM","CIV","CIH","SP","OSC"],"texto":"Visão de futuro da cidade: Construir a visão de futuro da cidade de forma participativa e inclusiva. Estabelecer essa visão em instrumentos de planejamento municipal (exemplos: Plano Diretor - PD, Plano Plurianual - PPA, Lei de Diretrizes Orçamentárias - LDO, Lei Orçamentária Anual - LOA). Na construção da visão de futuro, considerar a perspectiva e os impactos específicos da transformação digital no território da cidade. Considerar também o contexto regional e as características locais nos aspectos econômico-financeiro, sociocultural, urbano-ambiental e político-institucional. Refletir a visão em metas, com etapas, atividades e prazos associados."},{"recomendacao_id":"1.2.5","objetivo":1,"pagina_inicio":40,"pagina_fim":40,"publicos":["GF","GE","GM","CIV","CIH","IFF"],"texto":"Articulação setorial no território: Desenvolver estratégias para que as políticas, planos e programas de desenvolvimento urbano e de setores afins sejam integradas no território, em todos os níveis de governo. As estratégias devem enfatizar as áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs)."},{"recomendacao_id":"1.3","objetivo":1,"pagina_inicio":40,"pagina_fim":41,"publicos":["GF","GE","GM","CIV","CIH","IFF"],"texto":"Transformação digital e setores urbanos: Desenvolver metodologia para mapear necessidades específicas das políticas setoriais urbanas que possam ser apoiadas por soluções digitais. As ações devem incluir infraestrutura e dispositivos digitais, bem como dados e informações georreferenciadas (com localização geográfica). Também devem estar em linha com a diversidade territorial e com as tipologias municipais e supramunicipais (agrupamentos de municípios) da Política Nacional de Desenvolvimento Urbano (PNDU). O objetivo é possibilitar o planejamento e a implementação de projetos e ações locais integradas."},{"recomendacao_id":"1.3.1","objetivo":1,"pagina_inicio":41,"pagina_fim":41,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Estratégias setoriais para transformação digital: Elaborar estratégias setoriais para a transformação digital nas cidades, nas áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs). As estratégias devem: (1) ser elaboradas com base em metodologia única que permita sua consolidação em uma estratégia global; (2) ser elaboradas de forma alinhada com esta Carta; (3) ser desenvolvidas pelos respectivos setores, com apoio da Comunidade da Carta. Os objetivos são: (a) identificar, organizar e endereçar demandas específicas de cada setor; e (b) permitir uma visão global que evite sobreposições e otimize esforços no território."},{"recomendacao_id":"1.3.2","objetivo":1,"pagina_inicio":41,"pagina_fim":41,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","SP","IEP","IFF"],"texto":"Eficiência energética e economia circular: Desenvolver projetos, utilizar mecanismos e tecnologias que ampliem a eficiência energética de infraestruturas e edifícios urbanos. Promover processos e desenvolver soluções que incorporem a lógica da economia circular (aproveitamento de resíduos). O objetivo é promover o uso responsável dos recursos naturais e garantir a qualidade de vida das pessoas das atuais e futuras gerações."},{"recomendacao_id":"1.4","objetivo":1,"pagina_inicio":42,"pagina_fim":42,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","OSC"],"texto":"Transformação digital e meio ambiente: Desenvolver e usar metodologias, dados e indicadores que respondam às mudanças ambientais e climática (aumento da temperatura média global com aumento da ocorrência de eventos climáticos extremos). Atuar nas frentes de adaptação (como prevenção a eventos climáticos extremos – deslizamentos, inundações, secas, erosões etc.) e de mitigação (redução de emissões de carbono)."},{"recomendacao_id":"1.4.1","objetivo":1,"pagina_inicio":42,"pagina_fim":42,"publicos":["GF","GE","GM","CIV","EC","SP","IEP","IFF"],"texto":"Dispositivos digitais no ambiente urbano: Estimular o uso de metodologias, dados e indicadores, digitais ou não, para monitorar e avaliar os impactos ambientais causados por infraestruturas e dispositivos digitais nos ambientes urbanos. Promover o uso responsável de recursos nas soluções de modernização tecnológica de serviços urbanos. O objetivo é reduzir a pegada de carbono na transformação digital das cidades."},{"recomendacao_id":"1.4.2","objetivo":1,"pagina_inicio":42,"pagina_fim":42,"publicos":["GF","GE","GM","EC","SP","IEP","IFF"],"texto":"Instrumentos ambientais: Introduzir o conceito e desenvolver projetos de infraestrutura verde em áreas urbanas. Sempre que possível, substituir a infraestrutura cinza pela infraestrutura verde. Integrar as perspectivas de serviços ecossistêmicos e de soluções baseadas na natureza nos instrumentos de política urbana. Estimular o desenvolvimento de regiões produtoras de alimentos próximas dos centros urbanos. Utilizar as TICs para estimular padrões responsáveis de produção e consumo e ativação da economia local."},{"recomendacao_id":"1.4.3","objetivo":1,"pagina_inicio":43,"pagina_fim":43,"publicos":["GF","GE","GM","IEP"],"texto":"Riscos e vulnerabilidades no espaço urbano: Desenvolver metodologias para identificar e definir os riscos e as vulnerabilidades no espaço urbano, subsidiar a tomada de decisões e desenvolver planos de contingência. Para isso, usar dados e informações coletadas pelas tecnologias de informação e comunicação (TICs). O objetivo é ampliar a resiliência das cidades."},{"recomendacao_id":"1.5","objetivo":1,"pagina_inicio":43,"pagina_fim":43,"publicos":["GF","GE","GM","CIV","CIH","EC","ET","SP","IEP","OSC"],"texto":"Transformação digital e política urbana: Desenvolver, usar e compartilhar soluções digitais que ajudem a implementar instrumentos de informação, planejamento, gestão e governança voltados ao desenvolvimento urbano sustentável, em diferentes escalas do território. As soluções digitais devem aumentar a eficácia e a efetividade desses instrumentos. Também devem estar alinhadas com a diversidade territorial e com as tipologias municipal, supramunicipal (agrupamentos de municípios) e regional da Política Nacional de Desenvolvimento Urbano (PNDU)."},{"recomendacao_id":"1.5.1","objetivo":1,"pagina_inicio":43,"pagina_fim":43,"publicos":["GF","GE","GM","CIV","CIH","EC","IEP","IFF","OSC"],"texto":"Dados e informações para o desenvolvimento urbano sustentável: Formular, implementar, monitorar e avaliar políticas, programas, projetos e ações de desenvolvimento urbano que sejam baseados em dados e informações públicas e auditáveis (que podem ser verificadas em uma auditoria)."},{"recomendacao_id":"1.5.1.1","objetivo":1,"pagina_inicio":43,"pagina_fim":44,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","IFF"],"texto":"TICs para o diagnóstico e a gestão urbana: Usar ferramentas de geoprocessamento (processamento de dados com localização geográfica) para entender melhor os fenômenos urbanos e para aperfeiçoar a capacidade de gestão dos governos locais. Incorporar nessas ações mecanismos inovadores da ciência de dados. Exemplos: (1) Inteligência Artificial (AI); e (2) análise de grandes quantidades de dados anonimizados (sem elementos que identifiquem as pessoas), conhecidos como Big Data. Respeitar a Lei Geral de Proteção de Dados Pessoais (LGPD). [Ver recomendação 3.2.]"},{"recomendacao_id":"1.5.1.2","objetivo":1,"pagina_inicio":44,"pagina_fim":44,"publicos":["GF","CIV","CIH","IEP"],"texto":"Sistema nacional de informações para o desenvolvimento urbano: Identificar, sistematizar e disponibilizar dados e informações públicas que sejam relevantes para o desenvolvimento urbano sustentável. Esses dados e informações devem ser elaborados para formular, implementar e monitorar a Política Nacional de Desenvolvimento Urbano (PNDU). Essas ações têm duas finalidades: (1) apoiar a implementação de iniciativas locais pelos entes federados (União, Estados, Distrito Federal e Municípios) e órgãos interfederativos (que representam mais de um ente federado); e (2) atender ao Art. 16-A do Estatuto da Metrópole. [ver recomendação 3.9.]"},{"recomendacao_id":"1.5.1.3","objetivo":1,"pagina_inicio":44,"pagina_fim":45,"publicos":["GF","GE","GM","CIV","CIH","EC","SP"],"texto":"Integração de dados para a política urbana: Promover a constante integração de setores e instituições para o intercâmbio de dados, como os dados fiscais, de serviços urbanos e de registros imobiliários. Essa integração permitirá entender melhor o uso e a ocupação do solo urbano. Essas ações irão viabilizar a aplicação de instrumentos de política urbana, como o Imposto Predial e Territorial Urbano (IPTU) progressivo no tempo e o Parcelamento, Edificação e Utilização Compulsório (PEUC)."},{"recomendacao_id":"1.5.1.4","objetivo":1,"pagina_inicio":45,"pagina_fim":45,"publicos":["GF","GE","GM","CIV","CIH","IEP","OSC"],"texto":"Mapeamento de áreas verdes urbanas e serviços ecossistêmicos: Apoiar os municípios e órgãos interfederativos (que representam mais de um ente federado - União, Estados, Distrito Federal e Municípios) a mapear as suas áreas verdes urbanas. Essa ação contribuirá com a meta 11.7 do Objetivo de Desenvolvimento Sustentável 11 da Agenda 2030 da ONU. Além das áreas verdes urbanas, apoiar municípios e órgãos interfederativos a mapear, atribuir valor financeiro e gerir de forma responsável seus recursos naturais e serviços ecossistêmicos. Para isso, disponibilizar sistema e metodologia de cadastro que sejam unificados em âmbito nacional."},{"recomendacao_id":"1.5.1.5","objetivo":1,"pagina_inicio":45,"pagina_fim":45,"publicos":["GF","GE","GM","CIV","CIH","IEP","IFF"],"texto":"Cadastros territoriais integrados: Apoiar municípios e órgãos interfederativos (que representam mais de um ente federativo - União, Estados, Distrito Federal e Municípios) a elaborar, revisar e integrar as suas bases territoriais. Essas bases podem ser bases cartográficas, cadastros imobiliários ou Cadastros Técnicos Multifinalitários (de diversas finalidades) – CTM. Além disso, apoiar a integração dessas bases com os sistemas de informações geográficas locais. Essas ações devem se basear em metodologias e recursos adequados às diferentes realidades e às tipologias municipais e supramunicipais (agrupamentos de municípios) da Política Nacional de Desenvolvimento Urbano (PNDU)."},{"recomendacao_id":"1.5.1.6","objetivo":1,"pagina_inicio":46,"pagina_fim":46,"publicos":["GM","CIV","CIH","SP","IEP","OSC"],"texto":"Mapeamentos colaborativos: Ampliar o uso de ferramentas de mapeamento colaborativo na gestão pública como estratégia para mobilizar saberes e engajamento comunitários. Essas ferramentas também são estratégicas no controle social das políticas públicas, especialmente para levantar necessidades habitacionais, bens comuns, ativos urbanos, ambientais e culturais de interesse coletivo. Além disso, contribuem para identificar e gerir conflitos urbanos. Essas ferramentas devem incluir tecnologias assistivas, de forma a possibilitar a participação da pessoa com deficiência ou mobilidade reduzida. Nessas ações, privilegiar o uso de plataformas e ferramentas gratuitas e de código aberto, como o OpenStreetMap. [Ver recomendação 3.9]"},{"recomendacao_id":"1.5.2","objetivo":1,"pagina_inicio":46,"pagina_fim":46,"publicos":["GF","GE","GM","CIV","CIH","EC","SP","IEP","IFF","OSC"],"texto":"Planejamento do desenvolvimento urbano sustentável:"},{"recomendacao_id":"1.5.2.1","objetivo":1,"pagina_inicio":46,"pagina_fim":46,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","OSC"],"texto":"Medidas para o alcance da visão de futuro: Elaborar ou revisar normas, políticas, programas e estratégias para adequá-los à visão de futuro da cidade, conforme estabelecido nos instrumentos de planejamento municipal (exemplos: Plano Diretor - PD, Plano Plurianual - PPA, Lei de Diretrizes Orçamentárias - LDO, Lei Orçamentária Anual - LOA). Essa adequação irá garantir que os projetos urbanos, inclusive iniciativas de cidades inteligentes, contribuam para realizar a visão de futuro. [Ver recomendação 1.2.4]"},{"recomendacao_id":"1.5.2.2","objetivo":1,"pagina_inicio":47,"pagina_fim":47,"publicos":["GF","GE","GM","CIV","CIH","EC","IFF"],"texto":"Intersetorialidade no planejamento urbano: Construir e consolidar uma visão integrada do planejamento municipal com base nos instrumentos de planejamento setorial. Enfatizar as áreas de urbanismo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente e tecnologias de informação e comunicação (TICs). Exemplo de instrumentos de tecnologias de informação e comunicação nas cidades: Plano Diretor de Cidades Inteligentes e Plano Diretor de TICs. O objetivo é possibilitar que as iniciativas sejam implementadas de forma coordenada no território, usando mecanismos locais de gestão e governança. Para isso, devem ser incluídos mecanismos de dados e informações."},{"recomendacao_id":"1.5.2.3","objetivo":1,"pagina_inicio":47,"pagina_fim":47,"publicos":["GF","GE","GM","CIV","CIH","EC","IFF","OSC"],"texto":"Planejamento urbano interfederativo: Apoiar processos de planejamento urbano integrado e intersetorial (com cooperação entre as diferentes áreas de política pública) nas seguintes realidades: (1) regiões metropolitanas, (2) municípios conurbados (municípios com zonas urbanas unidas) e (3) municípios que apresentem relações de interdependência porque compartilham funções públicas de interesse comum. Esses processos de planejamento devem ser integrados de duas formas: pela elaboração de Planos de Desenvolvimento Urbano Integrado (PDUIs) ou pela elaboração conjunta e simultânea de Planos Diretores municipais (PDs). Ao elaborar os planos, é necessário articular dados, ferramentas, estratégias e as abordagens setoriais que façam parte dos planos municipais específicos."},{"recomendacao_id":"1.5.2.4","objetivo":1,"pagina_inicio":48,"pagina_fim":48,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF"],"texto":"Planejamento na escala de projetos urbanos: Desenvolver, consolidar e disseminar metodologias para elaborar projetos na escala intermediária da cidade (regiões, conjuntos de bairros ou outro agrupamento de áreas que seja menor que o território municipal). O objetivo é implementar processos de renovação urbana, de estruturação urbana ou de expansão urbana. Usar os projetos como oportunidades para distribuir infraestruturas para inclusão digital no espaço urbano. Na elaboração desses projetos, observar os princípios de desenho universal (que viabiliza o uso por todas as pessoas) e as normas de acessibilidade (Estatuto da Pessoa com Deficiência, Art. 55)."},{"recomendacao_id":"1.5.3","objetivo":1,"pagina_inicio":48,"pagina_fim":49,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Gestão e governança para o desenvolvimento urbano sustentável: [ver Objetivos Estratégicos 3 e 4]. Prover acesso equitativo à internet de qualidade para todas as pessoas"},{"recomendacao_id":"2.1","objetivo":2,"pagina_inicio":49,"pagina_fim":49,"publicos":["GF","GE","GM","CIV","CIH","AR","ET","SP","IEP","OSC"],"texto":"Direito de acesso à internet: Reconhecer e tornar efetivo o direito de acesso à internet por todas as pessoas (Marco Civil da Internet no Brasil, Art. 4o). Para isso, desenvolver e implantar políticas, programas e projetos de infraestrutura. Incluir nessas ações projetos da infraestrutura de suporte para redes de telecomunicações, indispensável para a prestação dos serviços de telecomunicações e internet. Incluir também outros aspectos relacionados à inclusão digital. Essas ações devem ser feitas respeitando as diretrizes nacionais editadas pela União Federal e Agências Reguladoras."},{"recomendacao_id":"2.2","objetivo":2,"pagina_inicio":49,"pagina_fim":50,"publicos":["GF","GE","GM","CIV","CIH","AR","ET"],"texto":"Infraestrutura digital para todas as pessoas: Viabilizar a instalação e a manutenção da infraestrutura para inclusão digital em regiões do país que carecem dessa infraestrutura e em áreas municipais com baixa conectividade. Manter a infraestrutura atualizada de forma a garantir a inclusão digital em todas as cidades, de forma permanente. Nessas ações, enfatizar os núcleos urbanos informais e as localidades afastadas. Respeitar as prioridades definidas nas políticas nacionais de desenvolvimento regional, de desenvolvimento urbano e de telecomunicações."},{"recomendacao_id":"2.2.1","objetivo":2,"pagina_inicio":50,"pagina_fim":50,"publicos":["GF","GE","GM","AR","ET","SP"],"texto":"Editais de faixas de frequência: Prever contrapartidas para ampliação da infraestrutura para inclusão digital nos editais de faixas de frequência de serviços de telecomunicações. Priorizar o atendimento de áreas que carecem de infraestrutura de qualidade e o atendimento a todas as cidades e comunidades do país. Os municípios devem acompanhar e viabilizar as implantações decorrentes de leilão de faixas de frequência."},{"recomendacao_id":"2.3","objetivo":2,"pagina_inicio":50,"pagina_fim":50,"publicos":["GF","GM","AR","SP","IEP","OSC"],"texto":"Meios diversos de acesso à internet: Incentivar e apoiar o estabelecimento de redes compartilhadas e comunitárias e outros meios alternativos de conexão e acesso à internet. Essas ações devem ser feitas incluindo o uso de Rádio e TV digitais, redes locais e pequenos provedores de Internet. Para isso, estabelecer parcerias com o setor privado, comunidades e organizações da sociedade civil. Essas parcerias devem ter como objetivos oferecer formação, garantir conhecimento técnico e fortalecer os elos comunitários através de infraestruturas de conectividade."},{"recomendacao_id":"2.3.1","objetivo":2,"pagina_inicio":50,"pagina_fim":51,"publicos":["GF","GE","GM","AR","ET","SP","IEP"],"texto":"Iniciativas locais de conexão e soluções digitais: Estabelecer mecanismos junto às agências reguladoras para a realização de estudos, experiências e testes de alocação de faixas do espectro eletromagnético para utilização aberta. Os objetivos são: (1) democratizar o acesso à comunicação sem fio; (2) possibilitar o desenvolvimento de iniciativas locais de conexão; e possibilitar o desenvolvimento local de soluções digitais para problemas comunitários."},{"recomendacao_id":"2.4","objetivo":2,"pagina_inicio":51,"pagina_fim":51,"publicos":["GF","GE","GM"],"texto":"Enfrentamento da exclusão digital: Promover soluções para os diferentes fatores de exclusão digital nas estratégias de universalização e democratização do acesso à internet e a tecnologias digitais. Essas ações devem estar alinhadas com a Estratégia Brasileira de Transformação Digital, para ajudar a alcançar suas metas."},{"recomendacao_id":"2.4.1","objetivo":2,"pagina_inicio":51,"pagina_fim":51,"publicos":["GF","GE","GM","CIV","CIH","ET","SP","IEP","IFF","OSC"],"texto":"Inclusão digital de pessoas com deficiência: Criar e usar soluções, elaborar e difundir normas e procedimentos para ampliar a acessibilidade da pessoa com deficiência à computação e à internet. Realizar essas ações também na oferta de serviços públicos digitais e outras iniciativas de governo digital (Estatuto da Pessoa com Deficiência, Art. 78). Estimular o desenvolvimento de soluções técnicas previstas no Plano Nacional de Internet das Coisas (Decreto 9.854/2019)."},{"recomendacao_id":"2.4.2","objetivo":2,"pagina_inicio":51,"pagina_fim":51,"publicos":["GF","GE","GM","CIV","CIH","ET","SP","IEP","IFF","OSC"],"texto":"Inclusão digital na perspectiva de gênero: Cumprir as metas nacionais para garantir a igualdade de gênero nas seguintes situações: (1) no acesso, nas habilidades de uso e na produção de tecnologias da informação e comunicação; (2) no acesso e na produção do conhecimento científico; e (3) no acesso e na produção de informação, conteúdos de comunicação e mídias (Agenda 2030, ODS 5, 5.b)."},{"recomendacao_id":"2.4.3","objetivo":2,"pagina_inicio":52,"pagina_fim":52,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Letramento digital: [ver Objetivo Estratégico 7]"},{"recomendacao_id":"2.5","objetivo":2,"pagina_inicio":52,"pagina_fim":52,"publicos":["GF","GE","GM","CIV","CIH","SP","OSC"],"texto":"Integração urbano-digital: Planejar e implementar as ações municipais de transformação digital de forma articulada com o planejamento territorial local. Para isso, observar as necessidades e a visão de futuro da cidade estabelecida no plano diretor ou em outros instrumentos de planejamento territorial. Se for necessário, adequar normas, políticas, programas, planos e estratégias."},{"recomendacao_id":"2.5.1","objetivo":2,"pagina_inicio":52,"pagina_fim":52,"publicos":["GF","CIH"],"texto":"Desenvolvimento urbano sustentável nas estratégias nacionais de TICs: Integrar o desenvolvimento urbano sustentável e os desafios da transformação digital nas cidades na Estratégia Nacional de Ciência e Tecnologia e na Estratégia Brasileira para a Transformação Digital (E-Digital)."},{"recomendacao_id":"2.5.2","objetivo":2,"pagina_inicio":52,"pagina_fim":52,"publicos":["GF","GE","GM","AR","ET","SP"],"texto":"Transparência nos dados de conectividade digital: Disponibilizar dados de conectividade digital (tais como banda larga, dispositivos móveis e internet por satélite) nas escalas intramunicipal (dentro dos limites municipais) e intraurbana (dentro da mancha urbana). Garantir que esses dados possam ser georreferenciados (ter a localização geográfica). Apresentar e disponibilizar os dados em linguagem inclusiva, de forma transparente e fácil de usar. Além disso, disponibilizar dados e estatísticas sobre acessos e atendimentos completos à população relacionados a serviços públicos digitais. Com essas atividades, será possível planejar ações de transformação digital na escala municipal."},{"recomendacao_id":"2.5.3","objetivo":2,"pagina_inicio":53,"pagina_fim":53,"publicos":["GF","GE","GM","CIV","CIH"],"texto":"Tipologias para “cidades inteligentes”: Reconhecer as diferentes características das cidades brasileiras, inclusive quanto ao acesso a tecnologias da informação e comunicação (TICs). A partir desse reconhecimento, tratar os municípios de forma diferenciada nas iniciativas de “cidades inteligentes”. Para isso, usar as tipologias (categorias de território) da Política Nacional de Desenvolvimento Urbano (PNDU). Com essas ações, será possível agir de modo a reduzir desigualdades de acesso à internet nas escalas intramunicipal (dentro dos limites municipais), intraurbana (dentro da mancha urbana), municipal (entre municípios), supramunicipal (entre conjuntos de municípios) e regional (entre regiões)."},{"recomendacao_id":"2.5.4","objetivo":2,"pagina_inicio":53,"pagina_fim":53,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Planejamento para “cidades inteligentes”: Considerar as determinações do Plano Diretor (ver Estatuto da Cidade) ao elaborar estratégias e planos municipais para a transformação digital. Da mesma forma, considerar as determinações do Plano de Desenvolvimento Urbano Integrado (Estatuto da Metrópole), caso exista. Alinhar o planejamento para “cidades inteligentes” com as recomendações desta Carta e seus desdobramentos em termos de normas, diretrizes e padrões. Exemplos de planos municipais para a transformação digital: Plano Diretor de Cidades Inteligentes e Plano Diretor de Tecnologias de Informação e Comunicação–TICs."},{"recomendacao_id":"2.5.5","objetivo":2,"pagina_inicio":53,"pagina_fim":54,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Conectividade digital e integração de equipamentos públicos: Fortalecer iniciativas que integrem instituições e equipamentos públicos de ensino e pesquisa. Para isso, formar parcerias entre instituições de modo a prover redes de infraestrutura digital. Ampliar o modelo de Redes Comunitárias de Ensino e Pesquisa para instituições e equipamentos públicos que atendam outras finalidades."},{"recomendacao_id":"2.5.6","objetivo":2,"pagina_inicio":54,"pagina_fim":54,"publicos":["GF","GE","GM","CIV","CIH","EC","ET","SP","IEP","IFF","OSC"],"texto":"Wi-Fi livre: Providenciar redes de Wi-Fi livre, seguro e de qualidade em equipamentos e espaços públicos, especialmente em áreas remotas e de baixa renda. Garantir segurança cibernética e proteção geral de dados pessoais nesses acessos. Buscar viabilizar o acesso a plataformas e aplicativos de serviços essenciais (exemplos: serviços públicos digitais, educação, saúde, mobilidade) sem consumo de dados móveis. Essa ação deve ser voltada a pessoas e grupos sociais vulneráveis, como ferramenta de inclusão social. Assegurar a ampliação do espectro de frequências de uso para novas redes Wi-Fi com mais capacidade, mais rápidas e eficientes."},{"recomendacao_id":"2.6","objetivo":2,"pagina_inicio":54,"pagina_fim":54,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","OSC"],"texto":"Solo, subsolo e espaço aéreo, mobiliário urbano e implantação de infraestrutura de TICs: Estabelecer normas e padrões para o planejamento, a utilização e a gestão do subsolo, do solo e do espaço aéreo nos municípios. Estabelecer normas e padrões também para a localização e o compartilhamento de infraestrutura para inclusão digital nas cidades (exemplos: postes, torres e dutos). Essas ações devem respeitar as normas gerais de telecomunicações editadas pela União. Disseminar melhores práticas relativas a normas, padrões e procedimentos adotados por municípios e outros níveis de governo."},{"recomendacao_id":"2.6.1","objetivo":2,"pagina_inicio":55,"pagina_fim":55,"publicos":["GF","GE","GM","CIV","CIH","AR","SP"],"texto":"Ações integradas no território: Estabelecer acordos interinstitucionais (entre instituições) e interfederativos (entre os entes da federação - União, Estados, Municípios e Distrito Federal) para regulação conjunta, quando for o caso. Instituir como serviço público independente a gestão do subsolo, do solo, do mobiliário urbano e do espaço aéreo, com vistas à sua ocupação compartilhada pelas empresas e órgãos responsáveis pelos serviços públicos e privados que demandam sua utilização."},{"recomendacao_id":"2.7","objetivo":2,"pagina_inicio":55,"pagina_fim":55,"publicos":["GM","CIV","CIH","ET","SP","IFF"],"texto":"Projetos de expansão, estruturação e requalificação urbana: prever e implementar infraestrutura para inclusão digital nos projetos específicos de expansão urbana (Estatuto da Cidade, Art. 42-A) e em projetos de requalificação urbana. Coordenar processos de expansão, estruturação e requalificação urbana com ações de implantação de infraestrutura de telecomunicações das operadoras de serviços móveis celulares e de banda larga fixa. Estreitar o relacionamento dos municípios com as empresas de telecomunicações. O objetivo é garantir o acesso à infraestrutura digital para todas as pessoas."},{"recomendacao_id":"2.8","objetivo":2,"pagina_inicio":55,"pagina_fim":55,"publicos":["GF","GE","GM","CIV","CIH","EC","SP"],"texto":"Projetos de iluminação pública: Promover a equidade de acesso ao serviço de iluminação pública nas cidades. Nos projetos de expansão e modernização das redes de iluminação pública, priorizar as seguintes áreas: (1) espaços públicos de utilização intensiva; (2) áreas urbanas desservidas; e (3) áreas urbanas inseguras, com índices de violência urbana acima da média da cidade. Essa priorização e as características de cada área devem ser observadas para a definição de padrões luminotécnicos adequados. Implantar projetos de iluminação pública adequados à diversidade dos municípios brasileiros."},{"recomendacao_id":"2.8.1","objetivo":2,"pagina_inicio":56,"pagina_fim":56,"publicos":["GF","GE","GM","AR","EC","SP"],"texto":"Sustentabilidade em iluminação pública: Elevar os padrões de eficiência energética em projetos de modernização e expansão da rede de iluminação pública. Nesses projetos, buscar a redução da poluição luminosa (poluição gerada pelo excesso de luz artificial). Promover a gestão eficiente do serviço por meio da adoção de soluções digitais integradas à rede. O objetivo é minimizar impactos da prestação do serviço de iluminação pública no meio ambiente e na saúde humana, assim como melhorar a qualidade de vida das pessoas nas cidades."},{"recomendacao_id":"2.8.2","objetivo":2,"pagina_inicio":56,"pagina_fim":56,"publicos":["GF","GM","AR","EC","ET"],"texto":"Aproveitamento da infraestrutura: Considerar a utilização potencial da rede de iluminação pública como infraestrutura de suporte para a oferta de serviços digitais. Buscar esse aproveitamento especialmente nos projetos de modernização e de expansão da rede de iluminação pública. Garantir o compartilhamento em condições justas, razoáveis e não discriminatórias de acesso aos postes de distribuição de energia elétrica. [Ver recomendação 2.6]."},{"recomendacao_id":"2.9","objetivo":2,"pagina_inicio":56,"pagina_fim":57,"publicos":["GF","GE","GM","AR","SP","IFF"],"texto":"Projetos de Internet das Coisas (IoT): Garantir padrões de segurança cibernética e de proteção de dados pessoais em todos os componentes de projetos de Internet das Coisas em áreas urbanas. Garantir o controle de procedência e qualidade dos dispositivos conectados à rede por meio de procedimentos oficiais de certificação. Enfatizar a garantia de transparência, controle e alternativa em processos de automação. Enfatizar também a garantia do direito à privacidade por meio da anonimização (sem elementos que identifiquem as pessoas) de dados e de outros procedimentos, principalmente quando houver atividades de videomonitoramento. Seguir o disposto no Plano Nacional de Internet das Coisas. (Decreto 9.854/2019)."},{"recomendacao_id":"2.10","objetivo":2,"pagina_inicio":57,"pagina_fim":58,"publicos":["GF","GE","GM","CIV","CIH","ET","IFF","AR","EC","SP","IEP","OSC"],"texto":"Apoio técnico e financeiro para a conectividade: Oferecer soluções para implantar e manter infraestrutura para inclusão digital. Isso deve ser feito por meio de apoio técnico e financeiro ou outros mecanismos de prestação de serviços públicos essenciais. Considerar as capacidades governativas dos municípios brasileiros. Considerar também as condições socioeconômicas e a localização da moradia da população beneficiária. Fomentar e facilitar a articulação dos municípios e de entidades supramunicipais (entidades que atuam sobre um agrupamento de municípios) com operadoras de serviços de telecomunicações. Estabelecer sistemas de governança de dados e de tecnologias, com transparência, segurança e privacidade"},{"recomendacao_id":"3.1","objetivo":3,"pagina_inicio":58,"pagina_fim":58,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Segurança cibernética: Garantir a segurança cibernética em infraestrutura, dispositivos, sistemas, dados e informações digitais. Estabelecer diretrizes, normas e procedimentos que avaliem, melhorem e validem a confiabilidade de hardwares, sistemas operacionais, dispositivos de acesso pessoal e ferramentas individuais (aplicativos)."},{"recomendacao_id":"3.2","objetivo":3,"pagina_inicio":58,"pagina_fim":59,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Proteção geral de dados pessoais: Garantir a proteção de dados pessoais, aderindo completamente à Lei Geral de Proteção de Dados Pessoais (LGPD). Respeitar a titularidade da pessoa natural sobre os seus próprios dados pessoais, garantindo, ao mesmo tempo, os direitos fundamentais de liberdade, intimidade e privacidade. Assegurar que o compartilhamento de dados pessoais obedeça aos princípios de finalidade e transparência. Para possibilitar essas ações, estabelecer normas e procedimentos que viabilizem o desenvolvimento seguro e ético de negócios inovadores baseados em dados. Seguir definições estabelecidas pela Agência Nacional de Proteção de Dados (ANPD)."},{"recomendacao_id":"3.2.1","objetivo":3,"pagina_inicio":59,"pagina_fim":59,"publicos":["GF","GE","GM","AR"],"texto":"Normas locais de proteção de dados pessoais: Apoiar os municípios para que adéquem normas e procedimentos à Lei Geral de Proteção de Dados Pessoais (LGPD). Nessa ação, regular de forma prioritária:(1) a regulação do tratamento de dados em serviços públicos essenciais; e (2) os cadastros em serviços digitais. Articular ações junto à Autoridade Nacional de Proteção de Dados (ANPD). O objetivo é garantir a coesão entre as políticas de compartilhamento de dados com aplicação geral e as propostas de cidades inteligentes."},{"recomendacao_id":"3.3","objetivo":3,"pagina_inicio":59,"pagina_fim":59,"publicos":["GF","CIV","CIH","SP","OSC"],"texto":"Transparência nos algoritmos de empresas de TICs: Incentivar que empresas de tecnologia de informação e comunicação digital tenham padrões elevados de transparência sobre os critérios e pressupostos que usam nos seus algoritmos. Possibilitar e fortalecer processos de auditoria algorítmica e fomentar o uso de softwares de código fonte aberto ou livres. Essas ações contribuem e devem estar alinhadas com o Sistema Nacional para a Transformação Digital."},{"recomendacao_id":"3.4","objetivo":3,"pagina_inicio":59,"pagina_fim":60,"publicos":["GF","GE","GM","CIV","CIH","AR","SP","IEP","IFF","OSC"],"texto":"Interoperabilidade: Garantir a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) ao implementar soluções de TICs (Tecnologias de Informação e Comunicação) em governos. Garantir a interoperabilidade também em iniciativas interinstitucionais, inclusive público-privadas. Em todos os casos, respeitar e usar normas, padrões e protocolos públicos oficiais (Programa de Interoperabilidade do Governo Eletrônico - e-PING)."},{"recomendacao_id":"3.5","objetivo":3,"pagina_inicio":60,"pagina_fim":60,"publicos":["GF","GE","GM","CIV","CIH"],"texto":"Políticas de dados abertos: Implementar políticas de dados abertos em todos os níveis de governo. Usar experiências e recursos já disponíveis e em operação, tais como: Portal Brasileiro de Dados Abertos, Infraestrutura Nacional de Dados Abertos (INDA) e Infraestrutura Nacional de Dados Espaciais (INDE). Usar as políticas de dados abertos para cumprir o princípio da transparência na administração pública e a Lei de Acesso à Informação (LAI). Usar os modelos e recomendações produzidos pela Parceria para Governo Aberto (OGP - Open Government Partnership)."},{"recomendacao_id":"3.5.1","objetivo":3,"pagina_inicio":60,"pagina_fim":60,"publicos":["GF","GE","GM","CIV","CIH"],"texto":"Registros administrativos: Coletar, sistematizar, digitalizar, georreferenciar (inserir localização geográfica) e disponibilizar dados e informações gerados ao executar políticas públicas e ao prestar serviços públicos, em todos os níveis de governo. Tratar e anonimizar dados sensíveis para possibilitar sua abertura. Todas as etapas devem cumprir as políticas de dados abertos e os padrões de interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) existentes para o nível de governo que executa as ações. Também devem respeitar a Lei Geral de Proteção de Dados Pessoais (LGPD) e a Lei de Acesso à Informação (LAI). Os dados e informações devem ser disponibilizados em linguagem inclusiva."},{"recomendacao_id":"3.5.2","objetivo":3,"pagina_inicio":61,"pagina_fim":61,"publicos":["GF","GE","GM","CIV","CIH","IEP"],"texto":"Dados geoespaciais: Fortalecer a Infraestrutura Nacional de Dados Espaciais (INDE) como plataforma que facilita o intercâmbio de dados geoespaciais (dados espaciais com localização geográfica). Estabelecer a Política Nacional de Geoinformação (PNGeo) e consolidar um vocabulário uniforme e específico em sistemas de informação geográfica urbana."},{"recomendacao_id":"3.5.3","objetivo":3,"pagina_inicio":61,"pagina_fim":61,"publicos":["GF","GE","GM","CIV","AR","EC","ET","IEP","IFF"],"texto":"Padronização para elaboração de cadastros territoriais: Articular iniciativas governamentais que elaboram, ou contribuem para elaborar, cadastros imobiliários. Essa articulação deve ter como foco uniformizar conceitos, nomenclaturas, métodos e meios de implementação. Isso irá otimizar esforços e garantir a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) de dados."},{"recomendacao_id":"3.6","objetivo":3,"pagina_inicio":61,"pagina_fim":61,"publicos":["GE","GM","CIH","IEP"],"texto":"Governo Digital: Formular e implementar estratégias estaduais e municipais de governo digital que sejam adequadas a cada realidade. O objetivo é tornar a Administração Pública mais acessível e mais eficiente ao prover serviços, como indica a Estratégia de Governo Digital e a Estratégia Brasileira para a Transformação Digital."},{"recomendacao_id":"3.6.1","objetivo":3,"pagina_inicio":61,"pagina_fim":62,"publicos":["GF","GE","GM","CIV","CIH","SP","IFF","OSC"],"texto":"Ampliar o acesso a serviços públicos e direitos sociais por meio de TICs: Usar tecnologias de informação e comunicação (TICs) para promover o direito à cidade e para ampliar os direitos sociais. Focar em áreas urbanas com carências de serviços públicos e em pessoas e grupos sociais vulneráveis. Para realizar esses direitos, as TICs devem ajudar a simplificar o acesso a serviços de saúde, educação, moradia, transporte, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), telecomunicações (inclusive serviços de internet), lazer e cultura."},{"recomendacao_id":"3.6.2","objetivo":3,"pagina_inicio":62,"pagina_fim":62,"publicos":["GF","GE","GM","CIV","CIH"],"texto":"Otimização e melhoria de processos administrativos: Estabelecer sistema de processo administrativo eletrônico. Aderir preferencialmente à infraestrutura pública colaborativa do Processo Eletrônico Nacional (PEN) e suas ações, como o Sistema Eletrônico de Informações – SEI. O objetivo é diminuir custos e tornar a tramitação (o andamento) de documentos públicos mais rápida, transparente e acessível."},{"recomendacao_id":"3.6.3","objetivo":3,"pagina_inicio":62,"pagina_fim":62,"publicos":["GF","GE","GM","EC","IFF"],"texto":"Serviços analógicos e medidas de transição para o digital: Manter e melhorar procedimentos analógicos e presenciais quando ofertar serviços públicos digitais. Essas ações também devem ser feitas ao implementar medidas de transição, especialmente quando for um serviço essencial. Considerar a grande quantidade de fatores de exclusão digital."},{"recomendacao_id":"3.6.4","objetivo":3,"pagina_inicio":62,"pagina_fim":62,"publicos":["GF","GE","GM"],"texto":"Identidade digital: Adotar e apoiar a implementação da “identidade digital ao cidadão”, conforme consta da Estratégia de Governo Digital."},{"recomendacao_id":"3.7","objetivo":3,"pagina_inicio":63,"pagina_fim":63,"publicos":["GF","GE","GM","AR","SP","IEP"],"texto":"Compras públicas: Promover parcerias entre os setores público e privado para revisar e adequar os processos de compras públicas, inclusive as compras que envolvam soluções inovadoras. Para isso, buscar o apoio do Ministério Público e dos Tribunais de Contas, atualizar a legislação e adaptar procedimentos administrativos."},{"recomendacao_id":"3.7.1","objetivo":3,"pagina_inicio":63,"pagina_fim":63,"publicos":["GF","GE","GM","CIV","CIH","SP"],"texto":"Contratações governamentais de TICs: Instituir, testar e normatizar novos modelos de governos contratarem Tecnologias de Informação e Comunicação (TICs). Essas ações devem ser feitas de forma conjunta, em cooperação intergovernamental (entre governos). Os novos modelos de contratação devem ter como base o uso de softwares livres e códigos abertos. Assegurar a contratação de instituições, entidades e empresas que tenham: (1) compromisso com os direitos humanos; (2) compromisso com a liberdade de expressão; (3) reputação ilibada; (4) comprovada experiência na área; e (5) responsabilidade e compromisso com a coisa pública. Priorizar a contratação de instituições, entidades e empresas locais. Usar mecanismos de colaboração para compartilhar experiências e boas práticas, tal como acontece na Comunidade de TICs da Plataforma GestGov."},{"recomendacao_id":"3.7.2","objetivo":3,"pagina_inicio":63,"pagina_fim":63,"publicos":["GF","GE","GM","SP","OSC"],"texto":"Regulação da propriedade de dados: Definir com precisão os direitos sobre a propriedade e as condições para usar dados em contratos públicos e na atuação pública de caráter regulatório. O mesmo deve ocorrer em iniciativas interinstitucionais que impliquem na geração e no compartilhamento de dados, incluindo as iniciativas público-privadas. Priorizar a abertura e uso dos dados em políticas públicas. Em todos os casos mencionados, respeitar o princípio da função social da propriedade, conforme consta do artigo constitucional sobre ordem econômica. (Art. 170 da Constituição Federal)."},{"recomendacao_id":"3.8","objetivo":3,"pagina_inicio":64,"pagina_fim":64,"publicos":["GF","GE","GM","CIV","CIH"],"texto":"Gestão territorial integrada: Usar sistemas de planejamento integrado e de gestão territorial integrada, com base em plataformas interoperáveis (que trabalham em conjunto para a troca eficaz de informações) de dados georreferenciados (plataformas que possibilitem a troca eficaz de dados com localização geográfica), em todos os níveis de governo. Os sistemas devem ser adequados às diferentes escalas das políticas públicas e respeitar a proteção de dados pessoais. Também devem atender às especificidades, demandas e capacidades locais, nos casos de sistemas municipais."},{"recomendacao_id":"3.8.1","objetivo":3,"pagina_inicio":64,"pagina_fim":64,"publicos":["GE","GM","CIV","CIH"],"texto":"Governança intermunicipal de dados: Estabelecer instituições de cooperação intermunicipal (entre municípios) para implantar, gerir e operar bases de dados, sistemas digitais e soluções compartilhadas de tecnologia de informação e comunicação. O objetivo deve ser otimizar recursos e ampliar a sustentabilidade dessas ações. Exemplos de instituições de cooperação intermunicipal (entre municípios): consórcios públicos, instâncias de governança metropolitana e associações de municípios."},{"recomendacao_id":"3.8.2","objetivo":3,"pagina_inicio":64,"pagina_fim":64,"publicos":["GF","GE","GM","CIV","CIH","EC","SP","IEP","IFF","OSC"],"texto":"Centros de gestão integrada: Implantar centros de informações integradas e protocolos públicos para apoiar a tomada de decisões em tempo real. Priorizar a gestão de emergências e a resposta a desastres. Centros articulados com instituições de Ensino e Pesquisa e com o ecossistema de inovação local. O objetivo dessa articulação é produzir conhecimento e construir respostas para problemas públicos. Para essa finalidade, disponibilizar dados coletados pela infraestrutura digital urbana e de registros administrativos anonimizados. Articular os recursos e meios dos Centros de gestão integrada com os dos laboratórios de experimentação urbana."},{"recomendacao_id":"3.9","objetivo":3,"pagina_inicio":65,"pagina_fim":65,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Plataformas públicas de compartilhamento de dados: Disponibilizar dados abertos e informações públicas em linguagem inclusiva, de forma organizada, compreensível e, sempre que possível, georreferenciados (com localização geográfica). As plataformas de visualização de dados e informações devem ser fáceis de usar por pessoas não-especialistas. Deste modo, as plataformas devem ser programadas em código aberto e com base em softwares livres. Os objetivos são: (1) possibilitar o uso dos dados e das informações pelo ecossistema de inovação local; (2) produzir conhecimento e soluções de interesse público; (3) promover a colaboração para aprimorar dados e análises geradas; e (4) reduzir a dependência de recursos para contratação e manutenção de licenças de softwares."},{"recomendacao_id":"3.10","objetivo":3,"pagina_inicio":65,"pagina_fim":66,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Transparência orçamentária na Administração Pública: Padronizar dados e informações relativos a contas públicas de todos os poderes e níveis de governo. Garantir a qualidade e a interoperabilidade (capacidade de sistemas trabalharem em conjunto para a troca eficaz de informações) desses dados e informações. Incluir mecanismos que permitam a geolocalização de investimentos públicos. Implementar a transparência ativa, adotando portais públicos organizados que facilitem a compreensão e o manuseio dos dados e informações por pessoas não especializadas. Os objetivos são: (1) facilitar o planejamento e a gestão orçamentária, financeira e patrimonial na Administração Pública; (2) permitir a integração de dados e informações; (3) facilitar o controle interno e externo, bem como o controle social das contas públicas. Adotar modelos inovadores e inclusivos de governança urbana e fortalecer o papel do poder público como gestor de impactos da transformação digital nas cidades"},{"recomendacao_id":"4.1","objetivo":4,"pagina_inicio":66,"pagina_fim":66,"publicos":["GF","GE","GM","CIV","CIH"],"texto":"Articulação intergovernamental: Fortalecer a articulação entre governos para consolidar a governança urbana multinível (que atua em vários níveis - nacional, regional, estadual e local), interfederativa (com cooperação entre diferentes entes da federação - União, Estados, Municípios e Distrito Federal) e intersetorial (com cooperação entre as diferentes áreas de política pública). Firmar o papel dos governos estaduais e federal no apoio à adaptação de recomendações e políticas para os contextos locais em conjunto com os municípios."},{"recomendacao_id":"4.1.1","objetivo":4,"pagina_inicio":66,"pagina_fim":67,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","OSC"],"texto":"Câmara interministerial: Fortalecer espaço de governança institucional de âmbito federal para cidades inteligentes, com participação aberta aos setores interessados. Os objetivos são: (1) construir condições para implementar esta Agenda compartilhada para cidades inteligentes; e (2) criar condições para a continuidade da plataforma colaborativa da Carta Brasileira para Cidades Inteligentes."},{"recomendacao_id":"4.1.2","objetivo":4,"pagina_inicio":67,"pagina_fim":67,"publicos":["GF","GE","GM","CIV","CIH"],"texto":"Cooperação interfederativa em governo digital: Promover o intercâmbio de informações em governo digital. Implementar medidas conjuntas de natureza colaborativa por arranjos de cooperação entre governos. Exemplo: adesão voluntária à Rede Nacional de Governo Digital – Rede Gov.br (Decreto 10.332/20, Art. 7o). O objetivo é otimizar recursos e tempo."},{"recomendacao_id":"4.2","objetivo":4,"pagina_inicio":67,"pagina_fim":67,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Atuação em rede e plataformas colaborativas Estado-Sociedade: Mobilizar saberes de diferentes segmentos da sociedade, pessoas e instituições, para construir soluções criativas para problemas urbanos contemporâneos com mais agilidade."},{"recomendacao_id":"4.2.1","objetivo":4,"pagina_inicio":67,"pagina_fim":67,"publicos":["GF","GE","GM","CIV","CIH","EC","SP","IEP","IFF","OSC"],"texto":"Rede digital para colaboração urbana: Estimular a formação de uma rede para o desenvolvimento urbano sustentável. A rede deve ser multinível (atuar nos níveis nacionais, regionais, estaduais e locais), interinstitucional (cooperação entre diferentes instituições) e intersetorial (com cooperação entre as diferentes áreas de política pública). A rede deve oferecer recursos digitais e inclusivos para realizar trabalhos colaborativos, incluindo a implementação e a retroalimentação desta Carta Brasileira para Cidades Inteligentes."},{"recomendacao_id":"4.2.2","objetivo":4,"pagina_inicio":67,"pagina_fim":68,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Rede de assistência técnica remota para ações no território: Expandir e adaptar o modelo da assistência técnica remota baseada em recursos digitais que foi implementado de forma pioneira pela Rede Universitária de Telemedicina. Essa rede de assistência técnica remota deve apoiar órgãos oficiais interfederativos (que agrupam diferentes entes da federação com interesse compartilhado - União, Estados, Municípios e Distrito Federal) e municípios para implementar políticas, projetos e ações de desenvolvimento urbano sustentável, incluindo iniciativas de cidades inteligentes. Apoiar principalmente os municípios de menor capacidade institucional."},{"recomendacao_id":"4.3","objetivo":4,"pagina_inicio":68,"pagina_fim":68,"publicos":["GF","GE","GM","SP","OSC"],"texto":"Construção de ambientes para inovação: Promover processos de governança e gestão urbana que sejam interinstitucionais (com cooperação entre diferentes instituições) e colaborativos. O objetivo é construir ambientes político-jurídico-institucionais que sejam: (1) favoráveis à inovação; e (2) adaptados ao contexto territorial e ao nível de atuação das instituições."},{"recomendacao_id":"4.3.1","objetivo":4,"pagina_inicio":68,"pagina_fim":68,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Políticas de inovação: Estimular e integrar fóruns de inovação no setor público que sejam interfederativos (agrupando diferentes entes da federação com interesse compartilhado - União, Estados, Municípios e Distrito Federal) e abertos à participação ampla de pessoas, instituições e setores interessados. O objetivo é trocar experiências, construir estratégias, políticas e programas, e formular propostas de aperfeiçoamento legislativo e de mecanismos jurídicos. Essas propostas devem reduzir os obstáculos burocráticos à inovação no setor público, incluindo as relações dos governos com a sociedade e a realização de negócios e contratos com empresas de inovação."},{"recomendacao_id":"4.3.2","objetivo":4,"pagina_inicio":69,"pagina_fim":69,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Colaboração em processos legislativos: Estimular o uso de processos participativos para elaborar leis e normas infralegais (de nível regulamentar, subordinadas às leis - decretos, portarias, resoluções, instruções normativas etc.), diretrizes, parâmetros e formas de atuação pública. Estimular especialmente nos casos de tecnologias disruptivas (que causam ruptura com padrões e modelos existentes) e temas inovadores ainda não regulados. Usar ferramentas de TICs (tecnologias de informação e comunicação) e tecnologias assistivas (com funcionalidade para garantir autonomia, independência, qualidade de vida e inclusão social da pessoa com deficiência ou com mobilidade reduzida). O uso dessas tecnologias deve ampliar o engajamento de pessoas e instituições interessadas."},{"recomendacao_id":"4.3.3","objetivo":4,"pagina_inicio":69,"pagina_fim":69,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","SP","IEP","OSC"],"texto":"Diálogo com órgãos de controle: Estabelecer fóruns regulares de diálogo entre: (1) instituições públicas que formulam e implementam políticas públicas; (2) órgãos de controle dos poderes executivo, legislativo e judiciário; (3) Ministério Público; (4) setores envolvidos; (5) organizações da sociedade civil. Esses fóruns devem ter caráter estratégico na tarefa de construir conjuntamente caminhos e suporte à tomada de decisões sobre a transformação digital nas cidades. O objetivo é assegurar a boa condução das políticas sobre o tema da transformação digital nas cidades, em todos os níveis de governo."},{"recomendacao_id":"4.3.4","objetivo":4,"pagina_inicio":69,"pagina_fim":70,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","IEP"],"texto":"Agências reguladoras: Alinhar normas, técnicas e operações relativas a serviços públicos que requeiram a instalação de infraestruturas no espaço urbano. Para isso, estabelecer espaço de governança permanente entre agências reguladoras desses serviços públicos. Os objetivos são: (1) racionalizar a instalação e a manutenção de infraestruturas no espaço urbano, otimizando sua utilização; (2) assegurar a observância das normas urbanísticas locais pelas concessionárias dos serviços regulados."},{"recomendacao_id":"4.3.5","objetivo":4,"pagina_inicio":70,"pagina_fim":70,"publicos":["GF","GE","GM","SP","IEP","IFF"],"texto":"Programas de fomento à inovação: Promover processos de formação e programas de fomento à inovação e ao desenvolvimento tecnológico. Os objetivos são: (1) orientar ações nos setores público e privado; e (2) apoiar o desenvolvimento urbano e a transformação digital sustentáveis, conforme as necessidades e prioridades locais e regionais."},{"recomendacao_id":"4.4","objetivo":4,"pagina_inicio":70,"pagina_fim":70,"publicos":["GF","GE","GM","CIV","CIH","IEP","IFF","OSC"],"texto":"Capacidades na administração pública para a transformação digital: Desenvolver capacidades e competências na Administração Pública que sejam voltadas à atuação no contexto da transformação digital e seus desdobramentos territoriais. Implementar e fortalecer programas de desenvolvimento institucional em todos os níveis de governo."},{"recomendacao_id":"4.4.1","objetivo":4,"pagina_inicio":70,"pagina_fim":70,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Apoio técnico para municípios: Desenvolver e implementar ações de capacitação e assistência técnica federais e estaduais para municípios. Essas ações devem ser acessíveis a todas as pessoas interessadas no território nacional, de preferência por meio de plataforma única que integre diferentes recursos e iniciativas [ver recomendação 4.2.1]. Devem estar de acordo com as respectivas capacidades governativas (capacidades de gestão e de sustentabilidade institucional) locais. Também devem estar de acordo com as tipologias (categorias de território) definidas na Política Nacional de Desenvolvimento Urbano (PNDU). O objetivo é apoiar a administração municipal na direção da transformação digital e do desenvolvimento urbano sustentáveis."},{"recomendacao_id":"4.4.2","objetivo":4,"pagina_inicio":71,"pagina_fim":71,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP"],"texto":"Competências governamentais em TICs (tecnologias da informação e comunicação): Fortalecer órgãos locais de processamentos de dados. Desenvolver habilidades governamentais em tecnologias de informação e comunicação para servidoras e servidores públicos de diversas áreas do conhecimento. Isso deve ser feito em todos os níveis de governo e por meio de cooperações interfederativas (entre União, Estados, Municípios e Distrito Federal). Os objetivos são: (1) possibilitar o diálogo e o trabalho conjunto entre áreas meio e fim; e (2) estimular a plena capacidade de usar conhecimentos avançados de tecnologias disruptivas (que causam ruptura com padrões e modelos existentes) e ciência de dados, para gerir grandes volumes de dados (Big Data)."},{"recomendacao_id":"4.4.3","objetivo":4,"pagina_inicio":71,"pagina_fim":71,"publicos":["GF","GE","GM","CIV","CIH","EC","SP","IEP","OSC"],"texto":"Metodologias inovadoras para desenho de soluções: Usar metodologias e mecanismos inovadores para elaborar e implementar políticas de desenvolvimento urbano sustentável e soluções para problemas urbanos. Exemplos de mecanismos inovadores: jogos (“gamificação”) e maratonas de programação (hackathons)."},{"recomendacao_id":"4.4.4","objetivo":4,"pagina_inicio":71,"pagina_fim":71,"publicos":["GF","GE","GM","IEP"],"texto":"Valorização de servidores públicos inovadores: Estabelecer mecanismos para identificar servidores públicos inovadores em todos os níveis de governo. Oferecer incentivos e oportunidades para o desenvolvimento e uso das potencialidades dos servidores em trabalhos institucionais e no aprimoramento de políticas públicas."},{"recomendacao_id":"4.5","objetivo":4,"pagina_inicio":71,"pagina_fim":71,"publicos":["GF","GE","GM","CIV","CIH","EC","ET","SP","IEP","OSC"],"texto":"Adoção de processos inovadores de gestão e governança no nível local:"},{"recomendacao_id":"4.5.1","objetivo":4,"pagina_inicio":72,"pagina_fim":72,"publicos":["GF","GE","GM","SP","OSC"],"texto":"Gestão democrática das cidades: Estimular o engajamento e a participação pública inclusiva: na elaboração e na revisão do Plano Diretor e de outros instrumentos de planejamento municipal; (1) em aspectos cotidianos de zeladoria e gestão urbana; e (2) na interação governo-pessoas. Esse estímulo deve se dar por meio de mecanismos inovadores e soluções digitais, e com o uso de tecnologias assistivas (com funcionalidade para garantir autonomia, independência, qualidade de vida e inclusão social da pessoa com deficiência ou com mobilidade reduzida). As ações devem estar de acordo com as demandas e necessidades locais e devem ser adequadas às características organizacionais e institucionais do município. Buscar alinhamento com a Estratégia de Governo Digital (Decreto 10.332/2020, objetivo 14.2) e executar a gestão democrática da cidade (Estatuto da Cidade, Capítulo IV)."},{"recomendacao_id":"4.5.2","objetivo":4,"pagina_inicio":72,"pagina_fim":72,"publicos":["GF","GE","GM","CIV","CIH","IEP"],"texto":"Intersetorialidade no nível local: Estabelecer espaços institucionais para cooperação e atuação intersetorial (cooperação entre as diferentes áreas de política pública), inclusive entre órgãos de municípios diferentes (escala supramunicipal). O objetivo é facilitar que as políticas, planos e programas de desenvolvimento urbano e de setores relacionados sejam implementados de forma integrada no território. Incluir ações de diferentes setores: por exemplo, habitação, saneamento básico (abastecimento de água potável, esgotamento sanitário, limpeza urbana e manejo de resíduos sólidos, drenagem e manejo das águas pluviais urbanas), mobilidade urbana, segurança hídrica, redução de desastres, meio ambiente, saúde, educação e segurança urbana. Adotar abordagem contínua e incremental (que aumenta e melhora algo que já existe) para alcançar a integração."},{"recomendacao_id":"4.5.3","objetivo":4,"pagina_inicio":73,"pagina_fim":73,"publicos":["GF","GE","GM","CIV","CIH","EC","SP","IEP","IFF","OSC"],"texto":"Soluções inovadoras para problemas locais: Mapear demandas locais concretas e a oferta de soluções inovadoras para problemas levantados. Para esse mapeamento, mobilizar o ecossistema (conjunto e relações de pessoas e instituições que desenvolvem tecnologia e inovam) e estabelecer cooperação local. Essas atividades devem buscar coordenar, entre os setores interessados, ações voltadas ao desenvolvimento urbano e à transformação digital sustentáveis."},{"recomendacao_id":"4.5.4","objetivo":4,"pagina_inicio":73,"pagina_fim":73,"publicos":["GF","GE","GM","CIV","CIH","EC","ET","SP","IEP","IFF","OSC"],"texto":"Laboratórios de experimentação urbana: Incentivar o surgimento de soluções urbanas inovadoras, criando espaços colaborativos transdisciplinares (que possibilitam a cooperação entre diferentes disciplinas e saberes) para cidades inteligentes. Essas ações devem considerar a visão ampla da transformação digital nas cidades. Para garantir que as soluções sejam realizáveis, deve-se focar em pesquisa e experimentação em ambientes reais. Para isso, articular instituições de ensino e pesquisa e outros setores envolvidos na produção de conhecimento, com apoio institucional e jurídico da Administração Pública Municipal. Integrar esses Laboratórios ao Observatório da Transformação Digital nas cidades e a outros fóruns oficiais relacionados à transformação digital [ver recomendação 8.2]."},{"recomendacao_id":"4.5.5","objetivo":4,"pagina_inicio":73,"pagina_fim":75,"publicos":["GF","GE","GM","CIV","CIH","EC","SP","IEP","OSC","AR","ET","IFF"],"texto":"Serviços urbanos disruptivos: Estruturar espaços de gestão e governança e usar metodologias ágeis para garantir: (1) a tomada de decisão informada por evidências; e (2) a regulação de soluções urbanas em momento adequado. Exemplos de soluções que demandam essas ações: soluções que usam mecanismos ou tecnologias disruptivas (que causam ruptura com padrões e modelos existentes); soluções que geram bases de dados com informações pessoais ou de interesse público; e soluções que usam ou interferem em espaços públicos urbanos (calçadas, praças, sistema viário, soluções de transporte motorizado ou não motorizado, serviços de entrega) etc. Fomentar o desenvolvimento econômico local no contexto da transformação digital"},{"recomendacao_id":"5.1","objetivo":5,"pagina_inicio":75,"pagina_fim":75,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","SP","IEP","IFF","OSC"],"texto":"Economias alternativas e inovadoras para a diversidade: Apoiar o desenvolvimento de modelos econômicos locais verdes, justos e inovadores. Incluir iniciativas de economias solidária, compartilhada, criativa, circular e colaborativa. Usar essas iniciativas para criar soluções de modo a atender as diferentes realidades locais e gerar oportunidades a todas as pessoas, especialmente para incluir pessoas e grupos sociais vulneráveis."},{"recomendacao_id":"5.1.1","objetivo":5,"pagina_inicio":75,"pagina_fim":76,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","SP","IEP","IFF","OSC"],"texto":"Decrescimento e economia zero emissões: Incluir perspectivas de decrescimento, descarbonização e outras variáveis inovadoras de sustentabilidade na exploração de novas alternativas de organização social e econômica. Introzudir a redução de desigualdades socioeconomicas e a distribuição de riquezas na discusssão de modelos econômicos verdes, justos e inovadores. O objetivo é lidar com a escassez de recursos naturais e com a precarização do mundo do trabalho."},{"recomendacao_id":"5.2","objetivo":5,"pagina_inicio":76,"pagina_fim":76,"publicos":["GF","GE","GM","CIV","CIH","EC","SP","IEP","IFF","OSC"],"texto":"Economia verde, solidária e sustentável: Promover incentivos econômicos ambientais, tais como modelos de pagamento por serviços ambientais, utilização de títulos verdes, compras públicas sustentáveis e programas de aquisição da produção agrícola sustentável. Também promover esquemas econômicos autogeridos (quando membros têm autonomia para planejar e executar as tarefas), de base comunitária e avaliar a possibilidade do seu escalonamento (produção em grande escala) com base em tecnologias de registro distribuído (sistemas digitais para registrar transações de forma descentralizada, em vários lugares ao mesmo tempo) (Agenda 2030, ODS 12 - Meta 12.7)."},{"recomendacao_id":"5.2.1","objetivo":5,"pagina_inicio":76,"pagina_fim":76,"publicos":["GF","GE","GM","CIV","CIH","EC","SP","IEP","IFF","OSC"],"texto":"Padrões sustentáveis de produção e consumo: Utilizar as TICs para estimular padrões responsáveis de produção e consumo e ativação da economia local."},{"recomendacao_id":"5.3","objetivo":5,"pagina_inicio":76,"pagina_fim":76,"publicos":["GF","GM","CIV","SP","IFF","OSC"],"texto":"Economia de plataforma: Usar mecanismos da economia de plataforma (atividade econômica e social facilitada por plataformas) para aproximar produtores e consumidores locais. O objetivo é fortalecer vínculos comunitários e territoriais, tais como relações de vizinhança, relações urbano-rurais e relações com microempreendedores individuais."},{"recomendacao_id":"5.4","objetivo":5,"pagina_inicio":77,"pagina_fim":77,"publicos":["GF","GE","GM","CIV","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Economia e mercado de dados: Implementar políticas, leis, regulamentos e outros instrumentos para estabelecer um mercado de dados ético e inclusivo. Garantir a proteção de dados pessoais, aderindo completamente à Lei Geral de Proteção de Dados Pessoais (LGPD). Devem ser considerados os efeitos sistêmicos desse mercado, assim como as características e as necessidades específicas de diferentes setores produtivos. O objetivo é aumentar a inovação, a competição, a transparência e a segurança jurídica na economia de dados."},{"recomendacao_id":"5.5","objetivo":5,"pagina_inicio":77,"pagina_fim":77,"publicos":["GF","GE","GM","CIV","CIH","EC","IFF"],"texto":"Pagamentos digitais de serviços públicos: Facilitar o uso de meios de pagamentos digitais para serviços públicos, desenvolvendo e compartilhando ferramentas que estejam alinhadas com a Plataforma de Cidadania Digital. Adotar o PIX (pagamento instantâneo do Banco Central) como forma de pagamento para serviços públicos. As ações devem ocorrer em todos os níveis de governo e em cooperação interfederativa (entre União, Estados, Municípios e Distrito Federal)."},{"recomendacao_id":"5.6","objetivo":5,"pagina_inicio":77,"pagina_fim":77,"publicos":["GF","GE","GM","CIV","CIH","EC","SP"],"texto":"Competitividade em serviços digitais urbanos: Buscar formas de garantir competitividade aos ecossistemas (conjunto e relações de pessoas e instituições que desenvolvem tecnologia e inovam) de serviços digitais urbanos. Para isso, devem-se usar práticas que evitem monopólios e promovam a escolha livre dos usuários. As ações devem estar alinhadas com a Declaração de Direitos de Liberdade Econômica."},{"recomendacao_id":"5.6.1","objetivo":5,"pagina_inicio":77,"pagina_fim":78,"publicos":["GF","GE","GM","CIV","SP","IFF"],"texto":"Crédito para pequenas empresas de TICs: Facilitar o acesso a condições especiais de crédito por pessoas microempreendedoras individuais e por pequenas empresas de TICs (tecnologias de informação e comunicação). Estabelecer incentivos financeiros e técnicos à operação de pequenos provedores de Internet de forma a garantir a provisão e a sustentabilidade de iniciativas de acesso à internet em parceria com o poder público."},{"recomendacao_id":"5.6.2","objetivo":5,"pagina_inicio":78,"pagina_fim":78,"publicos":["GF","GE","GM","CIV","SP","IEP","IFF","OSC"],"texto":"Apoio à inclusão produtiva e digital: Criar subsídios e outros mecanismos para a inclusão produtiva e digital de micro e pequenas empresas, pessoas empreendedoras ou pessoas que trabalham informalmente. Esses mecanismos devem viabilizar economicamente o acesso dessas pessoas e empresas: (1) à internet; (2) a dispositivos digitais de qualidade, tais como smartphones, tablets e notebooks; e (3) a plataformas para comércio eletrônico. As ações também devem apoiar a legalização das pessoas que trabalham informalmente."},{"recomendacao_id":"5.7","objetivo":5,"pagina_inicio":78,"pagina_fim":78,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"TICs para a redução da pobreza urbana: Usar as tecnologias de informação e comunicação para reduzir a pobreza urbana, contribuindo para a Meta 1.4 do Objetivo de Desenvolvimento Sustentável 1."},{"recomendacao_id":"5.7.1","objetivo":5,"pagina_inicio":78,"pagina_fim":78,"publicos":["GF","GE","GM","CIV","SP","IFF","OSC"],"texto":"Acesso a serviços financeiros e microfinanças: Promover a inclusão financeira de pessoas e grupos sociais vulneráveis. Para isso, deve-se possibilitar o acesso dessas pessoas e grupos a serviços financeiros, microfinanças e outras formas de participação econômica. Essas ações devem ser feitas com o apoio de produtos e serviços digitais. O objetivo deve ser reduzir desigualdades de acesso a recursos econômicos."},{"recomendacao_id":"5.7.2","objetivo":5,"pagina_inicio":79,"pagina_fim":79,"publicos":["GF","GE","GM","CIV","SP","IEP","IFF","OSC"],"texto":"Acesso à terra urbana regular: Usar tecnologias de informação e comunicação para facilitar a regularização fundiária de núcleos urbanos informais de baixa renda (REURB-S). A regularização fundiária deve acontecer com o apoio de programas de assistência técnica às comunidades. Essas ações têm como objetivo reconhecer direitos sociais e patrimoniais."},{"recomendacao_id":"5.7.3","objetivo":5,"pagina_inicio":79,"pagina_fim":79,"publicos":["GF","GE","GM","CIV","CIH","SP","IFF","OSC"],"texto":"Negócios sociais para a ampliação de serviços e direitos: Estimular parcerias e negócios sociais que ampliem o acesso a serviços essenciais e assegurem direitos, inclusive para pessoas motoristas e entregadoras por aplicativos. Estimular também parcerias e negócios que promovam a inclusão social e produtiva de pessoas e grupos sociais vulneráveis, gerando renda e emprego. As ações de inclusão devem ser apoiadas por processos de formação continuada e inclusão digital."},{"recomendacao_id":"5.8","objetivo":5,"pagina_inicio":79,"pagina_fim":80,"publicos":["GF","GE","GM","CIV","CIH","ET","SP","IEP","IFF","OSC"],"texto":"Desenvolvimento econômico regional e local: Apoiar cadeias produtivas e ecossistemas de inovação (conjunto e relações de pessoas e instituições que desenvolvem tecnologia e inovam) nos territórios, de modo a reduzir desigualdades socioeconômicas e espaciais. Fortalecer arranjos produtivos locais, ofertar incentivos econômicos e implementar infraestruturas e tecnologias sociais de suporte, tais como parques tecnológicos, laboratórios especializados e incubadoras. Essas ações devem estar alinhadas com a Política Nacional de Desenvolvimento Regional (PNDR) e com a Política Nacional de Desenvolvimento Urbano (PNDU). Também devem estar alinhadas com os planos regionais de desenvolvimento: Plano Regional de Desenvolvimento da Amazônia (PRDA), Plano Regional de Desenvolvimento do Nordeste (PRDNE) e Plano de Desenvolvimento do Centro-Oeste (PRDCO)."},{"recomendacao_id":"5.8.1","objetivo":5,"pagina_inicio":80,"pagina_fim":80,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Arranjos Produtivos Locais: Desenvolver, usar e compartilhar soluções digitais para identificar e fortalecer Arranjos Produtivos Locais. Disseminar metodologias e ampliar iniciativas de ativação e articulação produtiva no território. Por exemplo, estimular o desenvolvimento de regiões produtoras de alimentos próximas dos centros urbanos. Essas ações devem ser facilitadas pelo uso de recursos e métodos da economia de plataforma (atividades econômicas facilitadas por plataformas digitais). As ações buscam fortalecer e ampliar os elos da cadeia produtiva do país, indo além da base produtiva e agregando segmentos à produção brasileira."},{"recomendacao_id":"5.8.2","objetivo":5,"pagina_inicio":80,"pagina_fim":80,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Cadeia produtiva de resíduos eletrônicos: Estimular projetos de Pesquisa & Desenvolvimento (P&D) voltados ao aproveitamento econômico de resíduos eletrônicos. Esses projetos devem estimular que a indústria nacional adote princípios da economia circular. As ações devem contribuir para reduzir os impactos negativos da transformação digital nas cidades (Agenda 2030 ODS 11 - Meta 11.6; ODS 12 - Metas 12.4 e 12.5)."},{"recomendacao_id":"5.8.3","objetivo":5,"pagina_inicio":80,"pagina_fim":81,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Compatibilizar soluções digitais às demandas urbanas: Fazer seleções e consultas públicas para identificar e sistematizar necessidades dos municípios relacionadas à melhoria das informações, do planejamento, da gestão e da governança urbanas. O objetivo é facilitar o desenvolvimento de soluções digitais pelo setor privado, especialmente por empresas de base tecnológica. Essas soluções digitais devem ser adequadas à diversidade territorial brasileira e estar alinhadas com as tipologias (categorias de território) municipal e supramunicipal (agrupamento de municípios) da Política Nacional de Desenvolvimento Urbano (PNDU)."},{"recomendacao_id":"5.8.4","objetivo":5,"pagina_inicio":81,"pagina_fim":81,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Startups e transformação digital nas cidades: Aproximar o ecossistema (conjunto e relações de pessoas e instituições que desenvolvem tecnologia e inovam) de startups (Lei das Startups) das necessidades municipais relacionadas a melhorar as informações, o planejamento, a gestão e a governança urbanas. Para esse fim, deve-se divulgar esta Carta e os seus desdobramentos em eventos do setor. Também deve-se articular linhas de financiamento para startups de natureza incremental (aumentam e melhoram gradualmente algo que já existe) ou de natureza disruptiva (rompem com padrões e modelos existentes). Aproveitar o ambiente do Comitê Nacional de Iniciativas de Apoio a Start-ups."},{"recomendacao_id":"5.8.5","objetivo":5,"pagina_inicio":81,"pagina_fim":81,"publicos":["GF","GE","GM","SP","IEP","OSC"],"texto":"Formação e mercado profissional: Estimular a formação profissional na área de TICs (exemplos: programadores, cientistas de dados), por meio de ensino profissionalizante e de nível superior. Fomentar mercado de trabalho para alocação e retenção das pessoas formadas por meio da articulação de estratégias locais que respondam a demandas das cidades, apoiadas pela rede de Institutos Nacionais de Ciência e Tecnologia (INCT)."},{"recomendacao_id":"5.9","objetivo":5,"pagina_inicio":82,"pagina_fim":82,"publicos":["GF","GE","GM","CIV","CIH","EC"],"texto":"Ambiente de negócios nas cidades: Aperfeiçoar, compatibilizar e dar ampla publicidade a normas e procedimentos municipais. Padronizar os processos burocráticos, tornando-os mais claros e eficientes. O objetivo é estimular o desenvolvimento econômico local. Os estados e a União devem atuar da mesma forma nos assuntos que forem de sua competência."},{"recomendacao_id":"5.9.1","objetivo":5,"pagina_inicio":82,"pagina_fim":82,"publicos":["GF","GE","GM","CIV"],"texto":"Classificação das atividades econômicas: Usar os códigos da Classificação Nacional de Atividades Econômicas–Fiscal (CNAE–Fiscal) do Instituto Brasileiro de Geografia e Estatística (IBGE) nos registros administrativos de todos os níveis de governo. Estabelecer fluxos para a criação de novas atividades no CNAE-Fiscal conforme a necessidade (exemplo: serviços que se caraceterizam pelo uso intensivo de tecnologias). O objetivo é criar uma medida unificadora de caráter nacional e mantê-la atualizada com novas atividades econômicas."},{"recomendacao_id":"5.9.2","objetivo":5,"pagina_inicio":82,"pagina_fim":82,"publicos":["GF","GE","GM"],"texto":"Liberação da atividade econômica: Facilitar a realização de negócios nas cidades. Para isso, simplificar os processos e atos públicos de liberação da atividade econômica (atos exigidos como condição para exercer uma atividade econômica), conforme os níveis de risco das atividades. Na definição de níveis de risco das atividades econômicas, observar o princípio da razoabilidade e da proporcionalidade, a segurança física e sanitária, e a competência constitucional dos municípios no ordenamento, uso e ocupação do solo."},{"recomendacao_id":"5.9.3","objetivo":5,"pagina_inicio":83,"pagina_fim":84,"publicos":["GM","CIH","GF","GE","CIV","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Normas urbanísticas municipais: Compatibilizar normas urbanísticas municipais, simplificar procedimentos e instituir serviço digital de licenciamento urbanístico e edilício (de edificações). Atender às normas gerais e diretrizes editadas pela União, inclusive relacionadas com os serviços de telecomunicações e sua respectiva infraestrutura de suporte. Estabelecer prazos razoáveis para analisar e emitir os atos públicos necessários. Tornar os regramentos e os processos de licenciamento acessíveis às pessoas, organizar e disponibilizar as informações de forma clara e em linguagem simples e inclusiva. Buscar disponibilizar essas informações em portal público de dados georreferenciados (com localização geográfica) relativos ao ordenamento territorial do município. O portal deve ser fácil de usar pelo público não especializado. Estimular modelos e instrumentos de financiamento do desenvolvimento urbano sustentável no contexto da transformação digital"},{"recomendacao_id":"6.1","objetivo":6,"pagina_inicio":84,"pagina_fim":84,"publicos":["GF","GE","GM"],"texto":"TICs no orçamento público: Incluir a transformação digital no orçamento público em todos os níveis de governo. O orçamento deve ser usado para realizar investimentos nas seguintes áreas: (1) modernização tecnológica; (2) digitalização de dados; (3) digitalização de serviços públicos; e (4) infraestrutura para inclusão digital. Os investimentos devem ser viabilizados inclusive com transferências de recursos. As ações do Governos Federal devem se adequar às tipologias (categorias de território) da Política Nacional de Desenvolvimento Urbano (PNDU)."},{"recomendacao_id":"6.2","objetivo":6,"pagina_inicio":84,"pagina_fim":85,"publicos":["GF","CIV","AR","ET"],"texto":"FUST e outros fundos para acesso à internet: Reformular a legislação do Fundo de Universalização das Telecomunicações (FUST) para permitir que seja aplicado em expansão do acesso à internet, por diferentes meios. A reformulação também deve ampliar o uso do FUST em ambientes urbanos e em áreas rurais e remotas. Utilizar fundos setoriais, como o Fundo de Fiscalização das Telecomunicações (Fistel) e o Fundo para o Desenvolvimento Tecnológico das Telecomunicações (Funttel), para massificar o acesso de todas as pessoas à internet. Considerar as ações previstas na Estratégia Brasileira para a Transformação Digital (E-digital)."},{"recomendacao_id":"6.3","objetivo":6,"pagina_inicio":85,"pagina_fim":85,"publicos":["GF","GE","CIV","CIH","IFF","OSC"],"texto":"Estratégias financeiras e tributárias para ampliação da conectividade digital: Incentivar os governos estaduais a implantarem políticas de redução de carga tributária. O objetivo é interiorizar (levar a cobertura das redes para o interior do país) a cobertura das redes do Serviço Móvel Pessoal (Estratégia Brasileira para a Transformação Digital E-digital) e os serviços de oferta de banda larga. Além disso, incentivar os governos estaduais a disponibilizarem recursos onerosos (com encargos financeiros) e não onerosos (sem encargos financeiros) para fornecer e ampliar a conectividade digital. Esses recursos devem apoiar a elaboração de projetos e a implementação de plataformas digitais."},{"recomendacao_id":"6.4","objetivo":6,"pagina_inicio":85,"pagina_fim":85,"publicos":["GF","GE","GM","CIV","CIH","ET","SP","IEP","IFF","OSC"],"texto":"Utilização de TICs para melhorar a arrecadação municipal:"},{"recomendacao_id":"6.4.1","objetivo":6,"pagina_inicio":85,"pagina_fim":86,"publicos":["GF","GE","GM","CIV","CIH","SP","IEP","IFF","OSC"],"texto":"Cadastros municipais: Disponibilizar assistência técnica e recursos financeiros onerosos (com encargos financeiros) ou não onerosos (sem encargos financeiros) aos municípios para elaborar e atualizar cadastros municipais, tais como: (1) bases cartográficas georreferenciadas (com localização geográfica); (2) cadastros territoriais municipais; e (3) plantas genéricas de valores (cadastro do valor do metro quadrado em cada área da cidade; usado como base para o cálculo do IPTU e do ITBI). Os cadastros devem: (1) obedecer a metrologia e padronização estabelecida por órgãos ou entidades competentes; (2) ser adequados aos diferentes tipos de municípios. Os municípios serão classificados em tipos na Política Nacional de Desenvolvimento Urbano (PNDU). Essas ações são estratégicas para aprimorar a gestão urbana e melhorar a arrecadação de tributos municipais. Envolver órgãos de pesquisa, geografia e estatística da União e dos Estados nessas ações, para execução direta ou em apoio aos municípios."},{"recomendacao_id":"6.4.2","objetivo":6,"pagina_inicio":86,"pagina_fim":86,"publicos":["GF","GE","GM","CIH","ET"],"texto":"TICs e mecanismos extrafiscais de arrecadação: Usar tecnologias de informação e comunicação para viabilizar ou melhorar a implementação de instrumentos para capturar e recuperar mais-valias urbanas (valorização do terreno por causa de ações públicas). Alguns desses instrumentos estão previstos no Estatuto da Cidade."},{"recomendacao_id":"6.5","objetivo":6,"pagina_inicio":86,"pagina_fim":86,"publicos":["GF","GE","GM","CIV","SP","IFF"],"texto":"Parcerias com instituições financeiras e de fomento: Estabelecer parcerias com instituições financeiras e de fomento para desenvolver linhas de financiamento para cidades inteligentes que estejam associadas às recomendações desta Carta. As parcerias devem incluir instituições brasileiras e internacionais. Nas linhas de financiamento, priorizar projetos de abordagem sistêmica (que considera que cada elemento ou ação em uma cidade tem efeitos que se entrelaçam e se afetam entre si, impactando de maneira complexa a vida na cidade) e intersetorial (com cooperação entre as diferentes áreas de política pública). As ações devem se adequar às tipologias (categorias de território) da Política Nacional de Desenvolvimento Urbano (PNDU)."},{"recomendacao_id":"6.6","objetivo":6,"pagina_inicio":86,"pagina_fim":87,"publicos":["GF","GE","GM","CIV","SP","IFF"],"texto":"Captação de recursos para projetos de cidades inteligentes: Dar apoio técnico para municípios captarem recursos onerosos (com encargos financeiros) e não onerosos (sem encargos financeiros) junto a instituições financeiras e de fomento. Para esse apoio, deve-se: (1) disponibilizar informações sobre linhas de financiamento e repasses de recursos disponíveis; e (2) dar suporte à elaboração de projetos de cidades inteligentes. As ações devem se adequar às tipologias (categorias de território) da Política Nacional de Desenvolvimento Urbano (PNDU)."},{"recomendacao_id":"6.7","objetivo":6,"pagina_inicio":87,"pagina_fim":87,"publicos":["GF","GE","GM","CIV","SP","IFF","OSC"],"texto":"Projetos de Concessão e Parcerias Público-Privadas: Desenvolver estudos de viabilidade para modelagens inovadoras proporcionadas pela transformação digital (integração de serviços públicos, valoração e transação de ativos ligados a economia de dados e à economia verde, por exemplo). Respeitar a Lei Geral de Proteção de Dados Pessoais nesses novos modelos de negócios. Considerar a inclusão de novas linhas para desenvolver modelagens inovadoras no Fundo de Apoio à Estruturação de Concessão e Parcerias Público-Privadas (FEP). [ver recomendações 5.1 a 5.4]"},{"recomendacao_id":"6.8","objetivo":6,"pagina_inicio":87,"pagina_fim":87,"publicos":["GF","GE","GM","CIH","AR"],"texto":"Contrapartidas pelo uso do espaço público: Estimular mecanismos para estabelecer contrapartida e cobrar de empresas de inovação e tecnologias de informação e comunicação (TICs) que usam infraestrutura urbana, espaços públicos e mobiliários urbanos. Esses mecanismos devem financiar o desenvolvimento urbano sustentável."},{"recomendacao_id":"6.9","objetivo":6,"pagina_inicio":87,"pagina_fim":87,"publicos":["GF","GE","GM","CIV","SP","IFF"],"texto":"Fomento à inovação pelo setor privado: Mapear e reunir a indústria e os setores de tecnologia de informação e comunicação em torno de ações que estimulem a inovação em prol do desenvolvimento urbano sustentável."},{"recomendacao_id":"6.10","objetivo":6,"pagina_inicio":88,"pagina_fim":89,"publicos":["GF","GE","GM","CIV","CIH","IEP","IFF","OSC","AR","EC","ET","SP"],"texto":"Estratégias inovadoras de financiamento: Realizar estudos exploratórios para identificar possibilidades de tributar serviços digitais privados. Os estudos também devem identificar as possibilidades de usar tecnologias de registro distribuído (sistemas digitais para registrar transações em vários lugares ao mesmo tempo) para valorar (atribuir valor financeiro) ativos públicos ou comuns. Os ativos a serem valorados devem ter potencial para gerar receitas e devem poder ser usados para compor novos modelos de negócios no contexto do desenvolvimento urbano sustentável. Fomentar um movimento massivo e inovador de educação e comunicação públicas para maior engajamento da sociedade no processo de transformação digital e de desenvolvimento urbano sustentáveis."},{"recomendacao_id":"7.1","objetivo":7,"pagina_inicio":89,"pagina_fim":89,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Uso sustentável da internet: Realizar ações de comunicação educacional para estimular padrões sustentáveis de uso de internet. Seguir a recomendação 7.2 nas ações de comunicação educacional."},{"recomendacao_id":"7.2","objetivo":7,"pagina_inicio":89,"pagina_fim":90,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Comunicação pública inclusiva e acessível: Usar linguagem simples, inclusiva, sem marcador de gênero e com recursos de acessibilidade na comunicação pública e na divulgação desta Carta. As mensagens devem ser claras, respeitando a diversidade de gênero e étnico-racial. Essas ações de comunicação devem garantir o direito da pessoa com deficiência acessar leitura, informação e comunicação (Estatuto da Pessoa com Deficiência, Art. 68). Exemplos de recursos de acessibilidade: Libras, Braille, arquivos digitais reconhecidos e acessados por leitores de tela, audiodescrição, Comunicação Alternativa etc."},{"recomendacao_id":"7.3","objetivo":7,"pagina_inicio":90,"pagina_fim":90,"publicos":["GF","GE","GM","CIV","CIH"],"texto":"Transformação digital e educação urbana: Promover ações de comunicação pública inclusiva e acessível que sejam voltadas ao desenvolvimento urbano e à transformação digital sustentáveis. Abordar grandes transformações globais (ex. mudança do clima). O objetivo dessas ações é sensibilizar e ampliar a consciência da sociedade sobre os impactos desses processos."},{"recomendacao_id":"7.3.1","objetivo":7,"pagina_inicio":90,"pagina_fim":90,"publicos":["GF","GE","GM","SP","IEP","IFF"],"texto":"Cidade educadora: Usar a cidade como suporte para a educação urbana. Para isso, deve-se incentivar que as pessoas e instituições deem valor aos recursos naturais, as áreas verdes e espaços públicos, equipamentos e mobiliário urbano. Também deve-se informar o público sobre a história e o significado dos lugares. Essas ações devem ser associadas ao uso de ferramentas de mapeamento colaborativo que levantem e registrem aspectos subjetivos relacionados a espaços urbanos."},{"recomendacao_id":"7.3.2","objetivo":7,"pagina_inicio":90,"pagina_fim":90,"publicos":["GF","GE","GM"],"texto":"Campanha de comunicação pública: Realizar campanha de comunicação pública para promover e informar sobre o desenvolvimento urbano sustentável. A campanha deve usar diferentes mídias, formatos e métodos digitais ou analógicos. O objetivo é alcançar: (1) crianças, pessoas jovens e adultas de diferentes raças, etnias, graus de instrução e papéis sociais; (2) diferentes cidades e contextos."},{"recomendacao_id":"7.4","objetivo":7,"pagina_inicio":91,"pagina_fim":91,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Disseminação da agenda brasileira para cidades inteligentes: Desenvolver e implementar estratégia de comunicação pública da Carta em linguagem simples e inclusiva, com a participação de segmentos adeptos da cultura digital. O objetivo é alcançar a sociedade de forma ampla e sensibilizá-la, particularmente quanto a duas questões: (1) as relações existentes entre as cidades e as tecnologias de informação e comunicação (TICs); e (2) os direitos digitais das pessoas."},{"recomendacao_id":"7.4.1","objetivo":7,"pagina_inicio":91,"pagina_fim":91,"publicos":["GF"],"texto":"Guia prático da Carta: Desenvolver e disponibilizar um Guia Prático para implementar a Carta voltado para técnicos e gestores municipais, escrito em linguagem simples e inclusiva. O Guia deve comunicar, disseminar e apoiar a efetivação dos objetivos e recomendações da Carta."},{"recomendacao_id":"7.4.2","objetivo":7,"pagina_inicio":91,"pagina_fim":91,"publicos":["GF","GE","GM","CIV","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Vinculação de iniciativas de cidades inteligentes à Carta: Fortalecer esta Carta como instrumento político para articular uma agenda nacional para a transformação digital nas cidades. Para isso, deve-se estabelecer vínculos entre Objetivos Estratégicos e Recomendações desta Carta, de um lado, e as iniciativas correspondentes de cidades inteligentes existentes ou futuras, de outro (indexação de produtos-filhos). Como resultado desse processo, haverá o registro de um conjunto de saberes sobre cidades inteligentes e sua evolução."},{"recomendacao_id":"7.5","objetivo":7,"pagina_inicio":91,"pagina_fim":92,"publicos":["GF","GE","GM","IEP"],"texto":"Letramento digital: Estimular ações para promover o letramento digital e aumentar o número de pessoas que participam da transformação digital. Os objetivos são aumentar as capacidades de inovação da sociedade brasileira e reduzir a vulnerabilidade da população a crimes cibernéticos."},{"recomendacao_id":"7.5.1","objetivo":7,"pagina_inicio":92,"pagina_fim":92,"publicos":["GF","GE","GM","SP","IEP","OSC"],"texto":"Letramento digital nos currículos escolares: Observar, cumprir e ampliar as propostas contidas na Base Nacional Comum Curricular (BNCC) para integrar a cultura digital nos currículos escolares."},{"recomendacao_id":"7.5.2","objetivo":7,"pagina_inicio":92,"pagina_fim":92,"publicos":["GF","GE","GM","SP","IEP","OSC"],"texto":"Cultura digital na comunidade escolar: Estimular processos de capacitação e aprendizagem em tecnologias digitais para toda a comunidade escolar. Desenvolver ações de educação especificas para o letramento digital de pessoas educadoras capacitando-as para atuar como multiplicadoras da inclusão digital. O objetivo é ampliar, agilizar e facilitar o letramento digital desde a infância até a fase adulta."},{"recomendacao_id":"7.5.3","objetivo":7,"pagina_inicio":92,"pagina_fim":92,"publicos":["GF","GE","GM","SP","IEP","OSC"],"texto":"Recursos digitais na educação formal: Promover o aparelhamento tecnológico das instituições de ensino por meio de laboratórios, equipamentos, programas, ferramentas, softwares e outros recursos digitais."},{"recomendacao_id":"7.6","objetivo":7,"pagina_inicio":92,"pagina_fim":93,"publicos":["GF","GE","GM","ET","IEP","OSC"],"texto":"Práticas comunitárias urbanas: Articular ações de comunicação integrada (com campanhas planejadas e elaboradas em cooperação entre setores e instituições e que passam uma mensagem unificada) em linguagem simples e inclusiva. O objetivo é aumentar o engajamento social em plataformas que mobilizam e desenvolvem práticas comunitárias urbanas sustentáveis no contexto da transformação digital."},{"recomendacao_id":"7.6.1","objetivo":7,"pagina_inicio":93,"pagina_fim":94,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Comunicação comunitária: Adotar o fortalecimento comunitário e a interface Estado e comunidade como estratégia para a transformação do território e a valorização do pertencimento, da identidade e da memória locais. Estimular projetos de educomunicação digital de base comunitária para produção de conteúdos. Os objetivos são: (1) disseminar perspectivas e pautas de interesse das comunidades envolvidas; (2) ampliar o acesso à inclusão digital; (3) fomentar a emancipação comunitária; (4) oferecer possibilidades de formação profissional. Construir meios para compreender e avaliar, de forma contínua e sistêmica, os impactos da transformação digital nas cidades."},{"recomendacao_id":"8.1","objetivo":8,"pagina_inicio":94,"pagina_fim":94,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"TICs e direitos humanos: Reduzir os impactos negativos da transformação digital, criando tecnologias e processos centrados nos direitos humanos e no uso sustentável de recursos naturais. O foco nos direitos humanos deve incluir as perspectivas do direito digital."},{"recomendacao_id":"8.1.1","objetivo":8,"pagina_inicio":94,"pagina_fim":94,"publicos":["GF","GE","GM","IEP","IFF","OSC"],"texto":"Avaliação de impactos: Construir meios para compreender e avaliar, de forma continuada, sistêmica e transparente, os impactos de políticas, planos, programas, projetos, atividades e ações de transformação digital nas cidades. Utilizar dados e indicadores confiáveis e comparáveis (séries históricas). Dar publicidade e disseminar as metodologias adotadas e os resultados obtidos nas avaliações (transparência ativa)."},{"recomendacao_id":"8.1.2","objetivo":8,"pagina_inicio":95,"pagina_fim":95,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Monitoramento de ações públicas: Estabelecer e disseminar mecanismos de monitoramento de políticas, planos, programas, projetos, atividades e ações de transformação digital nas cidades. Engajar todos os poderes e níveis de governo nessas iniciativas. Usar TICs e uniformizar mecanismos (indicadores, plataformas de disseminação) para promover a transparência ativa e facilitar o controle social."},{"recomendacao_id":"8.1.3","objetivo":8,"pagina_inicio":95,"pagina_fim":95,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Revisão humana: Garantir e facilitar a revisão humana em processos de tomada de decisão automatizados ou baseados em Inteligência Artificial, em respeito aos direitos humanos. Implantar mecanismos de transparência ativa e assegurar ampla comunicação pública, em linguagem simples e inclusiva, às pessoas titulares de dados utilizados em serviços automatizados."},{"recomendacao_id":"8.2","objetivo":8,"pagina_inicio":95,"pagina_fim":95,"publicos":["GF","GM","CIV","CIH","SP","IEP","OSC"],"texto":"Observatório para a transformação digital nas cidades: Integrar o tema das cidades inteligentes ao Observatório para a Transformação Digital (OTD), considerando cidades inteligentes na perspectiva ampla de transformação digital nas cidades. Estimular que esse Observatório e outros fóruns oficiais relacionados à transformação digital busquem: (1) compreender e avaliar os impactos da transformação digital nas cidades; (2) incentivar a implementação desta Carta; e (3) fomentar, articular, integrar e disseminar as experiências provenientes dos Laboratórios de Experimentação Urbana [ver recomendação 4.5.4]."},{"recomendacao_id":"8.3","objetivo":8,"pagina_inicio":95,"pagina_fim":96,"publicos":["GF","GM","CIV","IEP"],"texto":"Maturidade para cidades inteligentes: Desenvolver e disponibilizar um Sistema Brasileiro de Maturidade para Cidades Inteligentes em uma plataforma digital própria a ser criada e mantida pelo governo federal. O Sistema deve usar metodologia e indicadores adequados à realidade brasileira e às tipologias municipais da Política Nacional de Desenvolvimento Urbano (PNDU). O objetivo é apoiar ações municipais voltadas ao desenvolvimento urbano e à transformação digital sustentáveis, além de monitorar nacionalmente o progresso dessas ações."},{"recomendacao_id":"8.4","objetivo":8,"pagina_inicio":96,"pagina_fim":96,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Impactos locais da transformação digital e controle social: Estimular que os temas do desenvolvimento urbano e da transformação digital sejam discutidos de forma integrada. Para isso, deve-se estimular a articulação institucional de conselhos ou fóruns que debatem sobre esses temas e que atuem no controle social de políticas públicas. Essas instituições devem acompanhar, avaliar e dar suporte à atuação do município sobre os impactos da transformação digital no território. As ações junto aos municípios devem considerar as condições político-institucionais específicas de cada cidade."},{"recomendacao_id":"8.5","objetivo":8,"pagina_inicio":96,"pagina_fim":96,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Ciência, tecnologia e inovação para a transformação digital e o desenvolvimento urbano sustentáveis: Mobilizar diferentes setores da sociedade para ampliar a compreensão sobre os impactos da transformação digital nas cidades. Devem ser considerados os impactos sobre os aspectos econômico-financeiro, sociocultural, urbano-ambiental e político-institucional."},{"recomendacao_id":"8.5.1","objetivo":8,"pagina_inicio":96,"pagina_fim":96,"publicos":["GF","GE","GM","CIV","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Linhas de pesquisa: Incentivar linhas de pesquisa e bolsas de fomento que favoreçam projetos transdisciplinares. O objetivo é produzir conhecimento científico de ponta e de forma contínua sobre a transformação digital nas cidades e seus impactos."},{"recomendacao_id":"8.5.2","objetivo":8,"pagina_inicio":97,"pagina_fim":97,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"“Ciberinfraestrutura” para geração de conhecimento sobre desenvolvimento urbano sustentável: Apoiar projetos de pesquisa, desenvolvimento e inovação que precisem de “ciberinfraestrutura” (infraestrutura de sistemas operacionais, gestão e processamento de dados, instrumentos avançados e ambientes de visualização) de grande porte. Para tal apoio, devem-se realizar investimentos de longo prazo e articular iniciativas desse tipo de infraestrutura."},{"recomendacao_id":"8.5.3","objetivo":8,"pagina_inicio":97,"pagina_fim":97,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Integração de campos disciplinares: Promover eventos técnicos e científicos e linhas de pesquisa que reúnam pessoas e instituições das áreas de desenvolvimento urbano e tecnologias da informação e comunicação. Esses eventos e linhas de pesquisa devem avançar na compreensão do fenômeno da transformação digital e das relações que esse fenômeno tem com diferentes disciplinas. O objetivo é consolidar uma abordagem transdisciplinar de pesquisa e ação."},{"recomendacao_id":"8.5.4","objetivo":8,"pagina_inicio":97,"pagina_fim":97,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Compreensão e atuação sobre impactos negativos: Entender e quantificar os impactos negativos de produtos, serviços e processos inovadores ligados a tecnologias de comunicação e informação (TICs) nas cidades brasileiras. Esse levantamento deve considerar a diversidade territorial das cidades. O objetivo é propor mecanismos para prevenir e, quando forem inevitáveis, reduzir e compensar esses impactos negativos, bem como acompanhar a sua evolução."},{"recomendacao_id":"8.5.5","objetivo":8,"pagina_inicio":97,"pagina_fim":98,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Tecnologias assistivas: Estimular a pesquisa, o desenvolvimento, a inovação e a difusão de tecnologias assistivas (recursos que ampliam o acesso e a participação de pessoas com deficiência) que tenham como objetivo: (1) ampliar o acesso de pessoas com deficiência às tecnologias da informação e comunicação; (2) ampliar o acesso de pessoas com deficiência às tecnologias sociais; (3) aumentar a autonomia de pessoas com deficiência nas cidades; e (4) aumentar o engajamento de pessoas com deficiência nas questões urbanas relacionadas à transformação digital (Estatuto da Pessoa com Deficiência, Art. 78; Plano Nacional dos Direitos da Pessoa com Deficiência, Art. 3o, VIII; Comitê Interministerial de Tecnologia Assistiva)."},{"recomendacao_id":"8.6","objetivo":8,"pagina_inicio":98,"pagina_fim":98,"publicos":["GF","GE","GM","CIV","CIH","AR","EC","ET","SP","IEP","IFF","OSC"],"texto":"Logística reversa de produtos eletrônicos: Acelerar e dar transparência à estruturação e à implementação de sistemas de logística reversa (coletar e devolver resíduos sólidos ao setor empresarial ou descartá-los corretamente). Esses sistemas devem incluir, por exemplo, fábricas, importadoras, distribuidoras e comércios de produtos eletroeletrônicos e seus componentes. As empresas devem oferecer às pessoas consumidoras dos itens a possibilidade de devolver os resíduos, sem usar serviços públicos de limpeza urbana ou manejo de resíduos sólidos (Política Nacional de Resíduos Sólidos, Art. 33)."}]}'
CARTA_REVISADA_SNAPSHOT = json.loads(_CARTA_REVISADA_SNAPSHOT_JSON)

OBJETIVO_PRIOR_DIMENSOES = {
    1: ["Econômica", "Sociocultural", "Meio Ambiente", "Capacidades Institucionais"],
    2: ["Econômica", "Sociocultural"],
    3: ["Capacidades Institucionais"],
    4: ["Capacidades Institucionais"],
    5: ["Econômica"],
    6: ["Econômica", "Capacidades Institucionais"],
    7: ["Sociocultural", "Capacidades Institucionais"],
    8: ["Capacidades Institucionais"],
}


def _classificar_metadados_carta(texto, objetivo=None):
    texto_norm = _normalizar(texto)
    scores_topicos = {}
    topico_para_dimensao = {}

    for dimensao, info in DIMENSOES.items():
        for topico in info["topicos"]:
            topico_para_dimensao[topico] = dimensao
            termos = [topico.replace("_", " ")] + PALAVRAS_CHAVE_TOPICO.get(topico, [])
            score = 0.0
            for termo in termos:
                termo_norm = _normalizar(termo)
                if termo_norm and termo_norm in texto_norm:
                    score += 1.25 if " " in termo_norm else 1.0
            if score > 0:
                scores_topicos[topico] = score

    scores_dimensoes = defaultdict(float)
    for topico, score in scores_topicos.items():
        scores_dimensoes[topico_para_dimensao[topico]] += score

    if objetivo in OBJETIVO_PRIOR_DIMENSOES:
        for dimensao in OBJETIVO_PRIOR_DIMENSOES[objetivo]:
            scores_dimensoes[dimensao] += 0.75

    dimensoes_ordenadas = [
        d for d, s in sorted(scores_dimensoes.items(), key=lambda item: (-item[1], item[0]))
        if s > 0
    ]
    if not dimensoes_ordenadas:
        dimensoes_ordenadas = list(DIMENSOES.keys())

    topicos_ordenados = [
        t for t, s in sorted(scores_topicos.items(), key=lambda item: (-item[1], item[0]))
    ][:3]
    return dimensoes_ordenadas[:3], topicos_ordenados


def _resumir_contexto_objetivo(texto, limite=700):
    texto = re.sub(r"\s+", " ", str(texto)).strip()
    if len(texto) <= limite:
        return texto
    corte = texto[:limite]
    ultimo_ponto = corte.rfind(".")
    if ultimo_ponto >= int(limite * 0.55):
        return corte[:ultimo_ponto + 1]
    return corte.rstrip() + "…"


def _titulo_e_corpo_recomendacao(texto):
    if ":" in texto:
        titulo, corpo = texto.split(":", 1)
        return titulo.strip(), corpo.strip()
    return texto.strip(), ""


def _pai_recomendacao(rec_id):
    partes = rec_id.split(".")
    if len(partes) <= 2:
        return None
    return ".".join(partes[:-1])


def _preparar_documentos_carta(snapshot):
    objetivos = {int(k): v for k, v in snapshot["objetivos"].items()}
    recs = snapshot["recomendacoes"]
    titulos = {}
    for r in recs:
        titulo, _ = _titulo_e_corpo_recomendacao(r["texto"])
        titulos[r["recomendacao_id"]] = titulo

    documentos = []

    for item in snapshot["conceitos"]:
        dims, tops = _classificar_metadados_carta(item["texto"])
        documentos.append({
            "chunk_id": item["doc_id"], "tipo": "conceito", "topico": item["titulo"],
            "texto": item["texto"], "dimensoes": dims, "topicos_relacionados": tops,
            "objetivo": None, "recomendacao_id": None,
            "pagina_inicio": item["pagina_inicio"], "pagina_fim": item["pagina_fim"],
            "publicos": [],
        })

    for i, item in enumerate(snapshot["principios"], 1):
        texto = f"{item['titulo']}: {item['texto']}"
        dims, tops = _classificar_metadados_carta(texto)
        documentos.append({
            "chunk_id": f"principio_{i}", "tipo": "principio", "topico": item["titulo"].title(),
            "texto": texto, "dimensoes": dims, "topicos_relacionados": tops,
            "objetivo": None, "recomendacao_id": None,
            "pagina_inicio": item["pagina"], "pagina_fim": item["pagina"], "publicos": [],
        })

    for i, item in enumerate(snapshot["diretrizes"], 1):
        texto = f"{item['titulo']}: {item['texto']}"
        dims, tops = _classificar_metadados_carta(texto)
        documentos.append({
            "chunk_id": f"diretriz_{i}", "tipo": "diretriz", "topico": item["titulo"].title(),
            "texto": texto, "dimensoes": dims, "topicos_relacionados": tops,
            "objetivo": None, "recomendacao_id": None,
            "pagina_inicio": item["pagina"], "pagina_fim": item["pagina"], "publicos": [],
        })

    for numero in range(1, 9):
        obj = objetivos[numero]
        texto = f"Objetivo Estratégico {numero}: {obj['titulo']}\nContexto: {obj['contexto']}"
        dims, tops = _classificar_metadados_carta(texto, objetivo=numero)
        documentos.append({
            "chunk_id": f"objetivo_{numero}_contexto", "tipo": "contexto_objetivo",
            "topico": f"Objetivo Estratégico {numero} — {obj['titulo']}",
            "texto": texto, "dimensoes": dims, "topicos_relacionados": tops,
            "objetivo": numero, "recomendacao_id": None,
            "pagina_inicio": obj["pagina_inicio"], "pagina_fim": obj["pagina_fim"], "publicos": [],
        })

    for rec in recs:
        numero_obj = int(rec["objetivo"])
        obj = objetivos[numero_obj]
        rec_id = rec["recomendacao_id"]
        titulo, corpo = _titulo_e_corpo_recomendacao(rec["texto"])
        pai_id = _pai_recomendacao(rec_id)
        pai_txt = f"\nSeção superior {pai_id}: {titulos[pai_id]}" if pai_id and pai_id in titulos else ""

        subtemas = []
        profundidade = len(rec_id.split("."))
        prefixo = rec_id + "."
        for outro_id, outro_titulo in titulos.items():
            if outro_id.startswith(prefixo) and len(outro_id.split(".")) == profundidade + 1:
                subtemas.append(f"{outro_id} {outro_titulo}")
        subtemas_txt = ""
        if len(corpo) < 120 and subtemas:
            subtemas_txt = "\nSubtemas: " + "; ".join(subtemas[:8])

        contexto_curto = _resumir_contexto_objetivo(obj["contexto"])
        texto_indexacao = (
            f"Objetivo Estratégico {numero_obj}: {obj['titulo']}\n"
            f"Contexto do objetivo: {contexto_curto}{pai_txt}\n"
            f"Recomendação {rec_id}: {rec['texto']}{subtemas_txt}"
        )
        dims, tops = _classificar_metadados_carta(
            f"{titulo}. {corpo}. {obj['titulo']}", objetivo=numero_obj
        )
        documentos.append({
            "chunk_id": "rec_" + rec_id.replace(".", "_"), "tipo": "recomendacao",
            "topico": f"Recomendação {rec_id} — {titulo}", "texto": texto_indexacao,
            "dimensoes": dims, "topicos_relacionados": tops, "objetivo": numero_obj,
            "recomendacao_id": rec_id, "pagina_inicio": rec["pagina_inicio"],
            "pagina_fim": rec["pagina_fim"], "publicos": rec.get("publicos", []),
        })
    return documentos


def _documentos_para_lc(documentos):
    return [
        LCDocument(
            page_content=c["texto"],
            metadata={
                "chunk_id": c["chunk_id"], "tipo": c["tipo"], "topico": c["topico"],
                "dimensoes": c["dimensoes"], "topicos_relacionados": c["topicos_relacionados"],
                "objetivo": c["objetivo"] if c["objetivo"] is not None else -1,
                "recomendacao_id": c["recomendacao_id"] or "",
                "pagina_inicio": int(c["pagina_inicio"]), "pagina_fim": int(c["pagina_fim"]),
                "publicos": c["publicos"],
                "fonte": "Carta Brasileira para Cidades Inteligentes — Edição Revisada",
            },
        )
        for c in documentos
    ]


def _hash_documentos_carta(documentos):
    payload = json.dumps({
        "pipeline": VERSAO_PIPELINE_CARTA_COMPLETA,
        "embedding": MODELO_EMBEDDING,
        "documentos": documentos,
    }, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _manifesto_indice_valido(hash_atual):
    if not os.path.exists(FAISS_CARTA_COMPLETA_PATH):
        return False
    if not os.path.exists(FAISS_CARTA_COMPLETA_MANIFEST_PATH):
        return False
    try:
        with open(FAISS_CARTA_COMPLETA_MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifesto = json.load(f)
        return (
            manifesto.get("hash_documentos") == hash_atual
            and manifesto.get("modelo_embedding") == MODELO_EMBEDDING
            and manifesto.get("pipeline") == VERSAO_PIPELINE_CARTA_COMPLETA
        )
    except Exception:
        return False


DOCUMENTOS_CARTA_COMPLETA = _preparar_documentos_carta(CARTA_REVISADA_SNAPSHOT)
if len(DOCUMENTOS_CARTA_COMPLETA) != 185:
    raise RuntimeError(
        f"Corpus granular inesperado: {len(DOCUMENTOS_CARTA_COMPLETA)} documentos (esperado: 185)."
    )

CARTA_COMPLETA_POR_CHUNK_ID = {c["chunk_id"]: c for c in DOCUMENTOS_CARTA_COMPLETA}
CARTA_COMPLETA_POR_RECOMENDACAO = {
    c["recomendacao_id"]: c for c in DOCUMENTOS_CARTA_COMPLETA if c["recomendacao_id"]
}

_hash_atual_carta = _hash_documentos_carta(DOCUMENTOS_CARTA_COMPLETA)


def _ler_manifesto_carta_completa():
    if not os.path.exists(FAISS_CARTA_COMPLETA_MANIFEST_PATH):
        return {}
    try:
        with open(FAISS_CARTA_COMPLETA_MANIFEST_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}


def _salvar_manifesto_carta_completa(
    hash_documentos,
    total_documentos,
    documentos_embutidos,
    completo,
):
    os.makedirs(os.path.dirname(FAISS_CARTA_COMPLETA_MANIFEST_PATH), exist_ok=True)
    with open(FAISS_CARTA_COMPLETA_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "hash_documentos": hash_documentos,
                "modelo_embedding": MODELO_EMBEDDING,
                "pipeline": VERSAO_PIPELINE_CARTA_COMPLETA,
                "total_documentos": total_documentos,
                "documentos_embutidos": documentos_embutidos,
                "completo": bool(completo),
                "total_recomendacoes": 163,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )


def _erro_quota_embedding(exc):
    texto = str(exc).lower()
    return (
        "429" in texto
        or "resource_exhausted" in texto
        or "quota exceeded" in texto
        or "rate limit" in texto
    )


def _construir_ou_retomar_indice_carta_completa(documentos):
    """Constrói o FAISS em lotes, respeitando a cota de embeddings.

    Recursos de segurança:
    - lotes pequenos;
    - pausa de 65 s entre lotes;
    - salva o FAISS após cada lote;
    - salva quantos documentos já foram embutidos;
    - se o Colab cair, retoma do último lote salvo;
    - se mudar o corpus/modelo/pipeline, descarta o progresso incompatível.
    """
    docs_lc = _documentos_para_lc(documentos)
    total = len(docs_lc)

    manifesto = _ler_manifesto_carta_completa()

    mesma_versao = (
        manifesto.get("hash_documentos") == _hash_atual_carta
        and manifesto.get("modelo_embedding") == MODELO_EMBEDDING
        and manifesto.get("pipeline") == VERSAO_PIPELINE_CARTA_COMPLETA
    )

    indice = None
    inicio = 0

    # 1) Índice completo e compatível: apenas carrega.
    if (
        mesma_versao
        and manifesto.get("completo") is True
        and int(manifesto.get("documentos_embutidos", 0)) == total
        and os.path.exists(FAISS_CARTA_COMPLETA_PATH)
    ):
        indice = FAISS.load_local(
            FAISS_CARTA_COMPLETA_PATH,
            embeddings,
            allow_dangerous_deserialization=True,
        )
        print(f"🔹 FAISS da Carta completa carregado ({total} documentos).")
        return indice

    # 2) Índice parcial compatível: retoma do ponto salvo.
    if (
        mesma_versao
        and os.path.exists(FAISS_CARTA_COMPLETA_PATH)
        and 0 < int(manifesto.get("documentos_embutidos", 0)) < total
    ):
        inicio = int(manifesto["documentos_embutidos"])
        indice = FAISS.load_local(
            FAISS_CARTA_COMPLETA_PATH,
            embeddings,
            allow_dangerous_deserialization=True,
        )
        print(
            f"🔄 Retomando FAISS da Carta completa: "
            f"{inicio}/{total} documentos já processados."
        )

    # 3) Manifesto antigo/incompatível: reconstrói do zero.
    elif os.path.exists(FAISS_CARTA_COMPLETA_PATH) and not mesma_versao:
        print(
            "♻️ O corpus/modelo mudou. O índice granular será reconstruído "
            "em lotes sem alterar o índice antigo de 20 chunks."
        )
        inicio = 0
        indice = None

    if inicio == 0:
        print(
            f"🔹 Construindo FAISS granular da Carta completa "
            f"({total} documentos) em lotes de "
            f"{LOTE_EMBEDDING_CARTA_COMPLETA}..."
        )

    for lote_inicio in range(
        inicio,
        total,
        LOTE_EMBEDDING_CARTA_COMPLETA,
    ):
        lote_fim = min(
            lote_inicio + LOTE_EMBEDDING_CARTA_COMPLETA,
            total,
        )
        lote = docs_lc[lote_inicio:lote_fim]

        print(
            f"\n🧩 Embedding do lote "
            f"{lote_inicio + 1}-{lote_fim}/{total}..."
        )

        sucesso = False

        for tentativa in range(1, MAX_TENTATIVAS_LOTE_EMBEDDING + 1):
            try:
                if indice is None:
                    indice = FAISS.from_documents(lote, embeddings)
                else:
                    indice.add_documents(lote)

                sucesso = True
                break

            except Exception as exc:
                if not _erro_quota_embedding(exc):
                    raise

                if tentativa >= MAX_TENTATIVAS_LOTE_EMBEDDING:
                    raise

                espera = PAUSA_ENTRE_LOTES_EMBEDDING
                print(
                    f"⚠️ Quota de embedding atingida no lote. "
                    f"Aguardando {espera}s antes da tentativa "
                    f"{tentativa + 1}/{MAX_TENTATIVAS_LOTE_EMBEDDING}..."
                )
                time.sleep(espera)

        if not sucesso:
            raise RuntimeError(
                f"Não foi possível processar o lote "
                f"{lote_inicio + 1}-{lote_fim}."
            )

        # Salva imediatamente: se o notebook cair, não perde o que já fez.
        indice.save_local(FAISS_CARTA_COMPLETA_PATH)

        concluido = lote_fim == total

        _salvar_manifesto_carta_completa(
            hash_documentos=_hash_atual_carta,
            total_documentos=total,
            documentos_embutidos=lote_fim,
            completo=concluido,
        )

        print(
            f"✅ Lote salvo. Progresso: {lote_fim}/{total} documentos."
        )

        # Só espera se ainda houver outro lote.
        if not concluido:
            print(
                f"⏳ Aguardando {PAUSA_ENTRE_LOTES_EMBEDDING}s "
                "para respeitar a janela de RPM do embedding..."
            )
            time.sleep(PAUSA_ENTRE_LOTES_EMBEDDING)

    print(
        f"\n✅ FAISS granular da Carta completa concluído "
        f"({total} documentos)."
    )
    return indice


if MODO_RAG_CARTA == "completa":
    INDICE_CARTA_COMPLETA = _construir_ou_retomar_indice_carta_completa(
        DOCUMENTOS_CARTA_COMPLETA
    )
else:
    INDICE_CARTA_COMPLETA = None
    print(
        "🔹 MODO_RAG_CARTA='antigo': "
        "índice granular não será usado nesta execução."
    )


print(
    "🔹 Corpus Carta revisada:",
    f"{sum(c['tipo']=='recomendacao' for c in DOCUMENTOS_CARTA_COMPLETA)} recomendações,",
    f"{sum(c['tipo']=='contexto_objetivo' for c in DOCUMENTOS_CARTA_COMPLETA)} contextos,",
    f"{sum(c['tipo']=='principio' for c in DOCUMENTOS_CARTA_COMPLETA)} princípios,",
    f"{sum(c['tipo']=='diretriz' for c in DOCUMENTOS_CARTA_COMPLETA)} diretrizes,",
    f"{sum(c['tipo']=='conceito' for c in DOCUMENTOS_CARTA_COMPLETA)} conceitos."
)


🔹 FAISS da Carta completa carregado (185 documentos).
🔹 Corpus Carta revisada: 163 recomendações, 8 contextos, 5 princípios, 6 diretrizes, 3 conceitos.


In [9]:
# =====================================================
# NÍVEL DE MATURIDADE E SELEÇÃO DE CHUNKS POR DIMENSÃO
# =====================================================
# Nível de maturidade de um indicador, usado apenas para priorizar QUAIS
# tópicos recebem chunks da Carta primeiro (nunca para gerar números no
# texto final — isso é proibido pelo prompt).
#
# Fonte primária: a própria planilha já traz, para cada indicador, uma
# coluna "N M Indicador..." com o nível de maturidade (0-7) calculado
# oficialmente (ver MAPA_NIVEL_MATURIDADE). Só quando um indicador não
# tiver essa coluna pareada (ex.: indicadores novos adicionados depois)
# é que se recorre à estimativa por texto abaixo, como fallback.
NIVEIS_TEXTUAIS = {
    "inexistente": 0, "nao": 0, "não": 0, "ausente": 0,
    "inicial": 1,
    "basico": 2, "básico": 2,
    "em desenvolvimento": 3,
    "em implantacao": 4, "em implantação": 4,
    "intermediario": 5, "intermediário": 5,
    "avancado": 6, "avançado": 6,
    "consolidado": 7, "sim": 7,
}


def estimar_nivel_indicador(valor):
    """Estima um nível de maturidade (0 a 7) a partir do valor bruto do
    indicador (fallback, usado quando não há coluna N M Indicador
    pareada). Retorna None quando não é possível estimar — nesse caso o
    indicador não entra na priorização, mas continua disponível ao modelo."""
    if pd.isna(valor):
        return None
    if isinstance(valor, (int, float)) and not isinstance(valor, bool):
        nivel = float(valor)
        return nivel if 0 <= nivel <= 7 else None
    return NIVEIS_TEXTUAIS.get(_normalizar(valor))


def obter_nivel_indicador(nome_indicador, valor, linha_planilha=None):
    """Retorna o nível de maturidade (0-7) de um indicador, priorizando a
    coluna oficial "N M Indicador..." da planilha (via
    MAPA_NIVEL_MATURIDADE) e caindo para a estimativa por texto quando
    essa coluna não existir ou não puder ser lida."""
    nm_col = MAPA_NIVEL_MATURIDADE.get(nome_indicador)
    if nm_col is not None and linha_planilha is not None and nm_col in linha_planilha.index:
        nivel_real = linha_planilha[nm_col]
        if not pd.isna(nivel_real):
            try:
                nivel_real = float(nivel_real)
                if 0 <= nivel_real <= 7:
                    return nivel_real
            except (TypeError, ValueError):
                pass
    return estimar_nivel_indicador(valor)


def _selecionar_chunks_dimensao_antigo(
    dimensao,
    indicadores_dimensao,
    linha_planilha=None,
    usar_fallback_semantico=True,
):
    """Lógica original dos 20 chunks — preservada sem alteração funcional."""
    chunks_gerais = [
        CHUNKS_CARTA[f"geral_{t}"] for t in CHUNKS_GERAIS if f"geral_{t}" in CHUNKS_CARTA
    ]
    pior_nivel_por_chunk = {}
    for nome, valor in indicadores_dimensao.items():
        classificacao = classificar_indicador(nome)
        if not classificacao or classificacao["dimensao"] != dimensao:
            continue
        nivel = obter_nivel_indicador(nome, valor, linha_planilha)
        chunk_id = classificacao["chunk_id"]
        nivel_efetivo = 99 if nivel is None else nivel
        pior_nivel_por_chunk[chunk_id] = min(pior_nivel_por_chunk.get(chunk_id, 99), nivel_efetivo)

    topicos_ordenados = sorted(pior_nivel_por_chunk, key=lambda cid: pior_nivel_por_chunk[cid])
    chunks_topicos = []
    chunk_ids_incluidos = set()
    for cid in topicos_ordenados:
        if cid in CHUNKS_CARTA:
            chunks_topicos.append(CHUNKS_CARTA[cid])
            chunk_ids_incluidos.add(cid)
        elif usar_fallback_semantico and cid in TOPICOS_SEM_CHUNK_NA_CARTA:
            nome_topico = cid.split("_", 1)[1].replace("_", " ")
            resultados = INDICE_CHUNKS.similarity_search(
                nome_topico, k=1, filter={"dimensao": dimensao}, fetch_k=50
            )
            for doc in resultados:
                chunk_id_encontrado = doc.metadata["chunk_id"]
                if chunk_id_encontrado not in chunk_ids_incluidos:
                    chunks_topicos.append(CHUNKS_CARTA[chunk_id_encontrado])
                    chunk_ids_incluidos.add(chunk_id_encontrado)
    return chunks_gerais + chunks_topicos


def _peso_maturidade_rag(nivel):
    if nivel is None:
        return 1.0
    nivel = max(0.0, min(7.0, float(nivel)))
    return 1.0 + PESO_MAXIMO_MATURIDADE_RAG * ((7.0 - nivel) / 7.0)


def _normalizar_score_faiss(distancia):
    try:
        distancia = max(0.0, float(distancia))
    except Exception:
        return 0.0
    return 1.0 / (1.0 + distancia)


def _topicos_prioritarios_para_rag(dimensao, indicadores_dimensao, linha_planilha=None):
    melhores_por_topico = {}
    for nome, valor in indicadores_dimensao.items():
        classificacao = classificar_indicador(nome)
        if not classificacao or classificacao["dimensao"] != dimensao:
            continue
        nivel = obter_nivel_indicador(nome, valor, linha_planilha)
        topico = classificacao["topico"]
        nivel_ordenacao = 99 if nivel is None else float(nivel)
        registro = {
            "indicador": nome, "valor": valor, "nivel": nivel,
            "topico": topico, "nivel_ordenacao": nivel_ordenacao,
        }
        atual = melhores_por_topico.get(topico)
        if atual is None or nivel_ordenacao < atual["nivel_ordenacao"]:
            melhores_por_topico[topico] = registro
    ordenados = sorted(
        melhores_por_topico.values(),
        key=lambda x: (x["nivel_ordenacao"], _normalizar(x["topico"]), _normalizar(x["indicador"])),
    )
    return ordenados[:MAX_INDICADORES_RAG_POR_DIMENSAO]


def _aderencia_topico_metadata(topico_indicador, metadata):
    topico_norm = _normalizar(topico_indicador)
    relacionados = metadata.get("topicos_relacionados", []) or []
    for topico_meta in relacionados:
        meta_norm = _normalizar(str(topico_meta).replace("_", " "))
        if topico_norm == meta_norm or topico_norm in meta_norm or meta_norm in topico_norm:
            return True
    return False


def selecionar_chunks_carta_completa(dimensao, indicadores_dimensao, linha_planilha=None):
    """RAG principal da Carta revisada com reranking por maturidade."""
    if INDICE_CARTA_COMPLETA is None:
        return []
    prioridades = _topicos_prioritarios_para_rag(
        dimensao, indicadores_dimensao, linha_planilha=linha_planilha
    )
    if not prioridades:
        return []

    candidatos = {}
    for prioridade in prioridades:
        nome = prioridade["indicador"]
        valor = prioridade["valor"]
        nivel = prioridade["nivel"]
        topico = prioridade["topico"]
        query = (
            f"Dimensão municipal: {dimensao}. Tópico: {topico}. "
            f"Indicador prioritário: {nome}. Situação observada no município: {valor}. "
            "Recupere da Carta Brasileira para Cidades Inteligentes os conceitos, diretrizes "
            "e recomendações mais úteis para interpretar esta capacidade ou lacuna municipal."
        )
        resultados = INDICE_CARTA_COMPLETA.similarity_search_with_score(
            query, k=K_CANDIDATOS_RAG_POR_INDICADOR
        )
        peso_maturidade = _peso_maturidade_rag(nivel)
        for doc, distancia in resultados:
            score_semantico = _normalizar_score_faiss(distancia)
            if (
                LIMIAR_SEMANTICO_CARTA_COMPLETA is not None
                and score_semantico < LIMIAR_SEMANTICO_CARTA_COMPLETA
            ):
                continue
            metadata = doc.metadata
            chunk_id = metadata["chunk_id"]
            bonus_dimensao = BONUS_DIMENSAO_RAG if dimensao in (metadata.get("dimensoes", []) or []) else 0.0
            bonus_topico = BONUS_TOPICO_RAG if _aderencia_topico_metadata(topico, metadata) else 0.0
            score_base = score_semantico * peso_maturidade + bonus_dimensao + bonus_topico
            if chunk_id not in candidatos:
                candidatos[chunk_id] = {
                    "doc": doc, "score_base_max": score_base,
                    "score_semantico_max": score_semantico,
                    "distancia_min": float(distancia), "indicadores": set(),
                    "niveis": [], "topicos_origem": set(),
                }
            reg = candidatos[chunk_id]
            reg["score_base_max"] = max(reg["score_base_max"], score_base)
            reg["score_semantico_max"] = max(reg["score_semantico_max"], score_semantico)
            reg["distancia_min"] = min(reg["distancia_min"], float(distancia))
            reg["indicadores"].add(nome)
            reg["topicos_origem"].add(topico)
            if nivel is not None:
                reg["niveis"].append(float(nivel))

    # Poucas âncoras manuais, somente quando configuradas.
    for prioridade in prioridades:
        nome = prioridade["indicador"]
        for rec_id in ANCORAS_DETERMINISTICAS_CARTA.get(nome, []):
            chunk = CARTA_COMPLETA_POR_RECOMENDACAO.get(rec_id)
            if not chunk:
                continue
            chunk_id = chunk["chunk_id"]
            # Em vez de uma nova busca aproximada, cria o LCDocument diretamente.
            doc = LCDocument(
                page_content=chunk["texto"],
                metadata={
                    "chunk_id": chunk["chunk_id"], "tipo": chunk["tipo"],
                    "topico": chunk["topico"], "dimensoes": chunk["dimensoes"],
                    "topicos_relacionados": chunk["topicos_relacionados"],
                    "objetivo": chunk["objetivo"], "recomendacao_id": chunk["recomendacao_id"],
                    "pagina_inicio": chunk["pagina_inicio"], "pagina_fim": chunk["pagina_fim"],
                    "publicos": chunk["publicos"],
                    "fonte": "Carta Brasileira para Cidades Inteligentes — Edição Revisada",
                },
            )
            candidatos[chunk_id] = {
                "doc": doc, "score_base_max": 10.0, "score_semantico_max": 1.0,
                "distancia_min": 0.0, "indicadores": {nome}, "niveis": [],
                "topicos_origem": {prioridade["topico"]}, "ancora_deterministica": True,
            }

    ranking = []
    for chunk_id, reg in candidatos.items():
        cobertura_extra = max(0, len(reg["indicadores"]) - 1)
        bonus_cobertura = min(0.08, BONUS_COBERTURA_MULTIPLA_RAG * cobertura_extra)
        reg["score_final"] = reg["score_base_max"] + bonus_cobertura
        ranking.append((chunk_id, reg))
    ranking.sort(key=lambda item: (-item[1]["score_final"], item[0]))
    selecionados = ranking[:MAX_CHUNKS_CARTA_COMPLETA]

    if AUDITAR_RAG_CARTA_COMPLETA:
        print(f"\n📚 RAG Carta completa — {dimensao}: {len(selecionados)} chunk(s) selecionado(s).")
        for pos, (chunk_id, reg) in enumerate(selecionados, 1):
            meta = reg["doc"].metadata
            rec_id = meta.get("recomendacao_id") or "-"
            paginas = (
                f"{meta.get('pagina_inicio')}"
                if meta.get("pagina_inicio") == meta.get("pagina_fim")
                else f"{meta.get('pagina_inicio')}-{meta.get('pagina_fim')}"
            )
            indicadores_txt = "; ".join(sorted(reg["indicadores"]))
            print(
                f"  {pos}. {chunk_id} | rec={rec_id} | pág={paginas} | "
                f"score={reg['score_final']:.4f} | semântico={reg['score_semantico_max']:.4f}"
            )
            print(f"     ↳ {meta.get('topico')}")
            print(f"     ↳ recuperado por: {indicadores_txt}")

    chunks_saida = []
    for chunk_id, reg in selecionados:
        original = CARTA_COMPLETA_POR_CHUNK_ID.get(chunk_id)
        if original:
            # Cópia de compatibilidade: preserva "dimensoes" do corpus granular
            # e registra a dimensão do diagnóstico que motivou esta recuperação.
            chunk_saida = dict(original)
            chunk_saida["dimensao_contexto"] = dimensao
            chunks_saida.append(chunk_saida)
    return chunks_saida


def selecionar_chunks_dimensao(
    dimensao,
    indicadores_dimensao,
    linha_planilha=None,
    usar_fallback_semantico=True,
):
    """Compatibilidade: alterna entre o RAG completo e o método antigo."""
    if MODO_RAG_CARTA == "completa":
        novos = selecionar_chunks_carta_completa(
            dimensao, indicadores_dimensao, linha_planilha=linha_planilha
        )
        if novos:
            return novos
        print(
            f"⚠️  RAG da Carta completa não retornou chunks para '{dimensao}'. "
            "Usando seleção antiga como fallback de segurança."
        )
    return _selecionar_chunks_dimensao_antigo(
        dimensao,
        indicadores_dimensao,
        linha_planilha=linha_planilha,
        usar_fallback_semantico=usar_fallback_semantico,
    )


def obter_dados_contextuais(dimensao, linha_planilha):
    """Retorna {rotulo_original: valor} dos dados contextuais (cadastrais/
    socioeconômicos e institucionais, ex.: PIB, IDH-M, GINI, CAPAG,
    empregos/empresas de TIC, universidades federais, PD&I) associados a
    uma dimensão específica, a partir de DADOS_CONTEXTUAIS_MAPA.

    Uso exclusivo: complementar a Análise Geral e a análise da própria
    dimensão. Nunca usado para calcular nível de maturidade, ordenar
    prioridades, gerar sugestões ou como indicador da metodologia — esses
    papéis continuam restritos aos indicadores classificados via
    classificar_indicador / MAPA_INDICADORES.
    """
    dados = {}
    for chave_normalizada, info in DADOS_CONTEXTUAIS_MAPA.items():
        if info["dimensao"] != dimensao:
            continue
        if chave_normalizada in linha_planilha.index:
            valor = linha_planilha[chave_normalizada]
            if not pd.isna(valor):
                dados[info["rotulo"]] = valor
    return dados


def obter_todos_dados_contextuais(linha_planilha):
    """Retorna todos os dados contextuais disponíveis (de todas as
    dimensões), usados apenas na Análise Geral."""
    dados = {}
    for dimensao in DIMENSOES:
        dados.update(obter_dados_contextuais(dimensao, linha_planilha))
    return dados


In [10]:
# =====================================================
# MODELOS ESTRUTURADOS DE SAÍDA (validação automática do JSON)
# =====================================================
class AnaliseGeral(BaseModel):
    analise_geral: str = Field(description="Dois parágrafos: pontos positivos e depois desafios/limitações, sem citar números nem o porte do município.")


class AnaliseDimensao(BaseModel):
    paragrafo_1: str = Field(
        description=(
            "Primeiro parágrafo da análise da dimensão. Deve apresentar a situação "
            "atual, capacidades existentes e resultados favoráveis, sem incluir "
            "sugestões de melhoria."
        )
    )
    paragrafo_2: str = Field(
        description=(
            "Segundo parágrafo da análise da dimensão. Deve interpretar os principais "
            "desafios, contrastes e lacunas, sem repetir o primeiro parágrafo e sem "
            "transformar o texto em lista de recomendações."
        )
    )
    paragrafo_3: Optional[str] = Field(
        default=None,
        description=(
            "Terceiro parágrafo opcional. Use somente quando houver conteúdo suficiente "
            "para aprofundar uma frente relevante da dimensão sem repetir os dois "
            "parágrafos anteriores. Caso não seja necessário, retorne null."
        )
    )
    sugestoes: List[str] = Field(
        description=(
            "Exatamente quatro sugestões de melhoria, distintas entre si, "
            "realistas, diretamente relacionadas aos indicadores e compatíveis "
            "com o porte do município."
        )
    )


class RevisaoDimensao(BaseModel):
    paragrafo_1: str = Field(
        description=(
            "Primeiro parágrafo da análise revisada. Deve preservar a função do "
            "primeiro parágrafo do rascunho: situação atual, capacidades existentes "
            "e resultados favoráveis. Pode incorporar atualização web apenas quando "
            "ela realmente qualificar esse conteúdo."
        )
    )
    paragrafo_2: str = Field(
        description=(
            "Segundo parágrafo da análise revisada. Deve preservar a função do "
            "segundo parágrafo do rascunho: desafios, contrastes e lacunas. "
            "Pode incorporar atualização web apenas quando ela realmente qualificar "
            "esse conteúdo."
        )
    )
    paragrafo_3: Optional[str] = Field(
        default=None,
        description=(
            "Terceiro parágrafo opcional da análise revisada. Preserve-o quando já "
            "existir no rascunho ou use-o quando uma atualização relevante exigir "
            "mais espaço sem sobrecarregar os dois primeiros. Caso contrário, null."
        )
    )
    sugestoes: List[str] = Field(
        description=(
            "Exatamente quatro sugestões de melhoria. Preserve as sugestões do "
            "rascunho sempre que possível e ajuste somente as que forem afetadas "
            "pela atualização web."
        )
    )


llm_estruturado_geral = llm.with_structured_output(AnaliseGeral)
llm_estruturado_dimensao = llm.with_structured_output(AnaliseDimensao)
llm_estruturado_revisao_dimensao = llm.with_structured_output(RevisaoDimensao)


In [11]:
# =====================================================
# PROMPTS: ANÁLISE GERAL E ANÁLISE POR DIMENSÃO
# =====================================================
INSTRUCAO_COM_BASE = (
    "BASE CONCEITUAL RECUPERADA DA CARTA BRASILEIRA PARA CIDADES INTELIGENTES:\n"
    "{base_conceitual}\n\n"
    "Use esta base para QUALIFICAR a interpretação dos indicadores e orientar as "
    "sugestões, respeitando a realidade e a maturidade observadas no município. "
    "NÃO copie trechos literalmente, NÃO cite números de recomendações, páginas "
    "ou o nome da Carta no texto final, e NÃO use a base para inventar fatos locais. "
    "Os indicadores da planilha continuam sendo a fonte principal sobre a situação "
    "real do município."
)
INSTRUCAO_SEM_BASE = (
    "Não há trecho específico da Carta Brasileira para Cidades Inteligentes "
    "disponível para este tópico. Baseie-se exclusivamente nos indicadores "
    "fornecidos e em boas práticas gerais de gestão pública municipal, sem "
    "inventar diretrizes ou citar qualquer documento de referência."
)

TEMPLATE_ANALISE_GERAL = """\
VOCÊ É UM ANALISTA SÊNIOR EM POLÍTICAS PÚBLICAS E PLANEJAMENTO URBANO,
COM EXPERIÊNCIA EM DESENVOLVIMENTO URBANO E MODERNIZAÇÃO DA GESTÃO MUNICIPAL
NO CONTEXTO BRASILEIRO.

MUNICÍPIO: {municipio}
PORTE POPULACIONAL: {porte}

A ANÁLISE GERAL É A ETAPA FINAL DO DIAGNÓSTICO.

As quatro análises abaixo JÁ FORAM:
1. geradas a partir dos indicadores, dados contextuais e base conceitual;
2. submetidas à validação web pontual;
3. revisadas quando a validação encontrou atualização relevante.

Portanto, use SOMENTE os textos finais abaixo como base da Análise Geral.
NÃO faça nova pesquisa web e NÃO introduza fatos externos que não estejam
presentes nessas análises.

ANÁLISES FINAIS DAS DIMENSÕES:
{dimensoes_validadas_txt}

TAREFA:
Produza EXATAMENTE DOIS PARÁGRAFOS de síntese institucional geral.

PARÁGRAFO 1 — BASES E CAPACIDADES:
- Sintetize as principais capacidades, serviços, estruturas e práticas já existentes.
- Procure padrões positivos que apareçam ou se relacionem em mais de uma dimensão.
- Mostre como essas capacidades se combinam no funcionamento geral do município.
- Não resuma cada dimensão separadamente.
- Evite enumeração de indicadores ou uma sequência de frases desconectadas.

PARÁGRAFO 2 — DESAFIOS TRANSVERSAIS E PRÓXIMO ESTÁGIO:
- Identifique os principais desafios estruturais que atravessam mais de uma dimensão.
- Relacione problemas que tenham uma origem ou consequência institucional semelhante,
  como falta de integração, planejamento, capacidade de gestão, digitalização,
  coordenação entre áreas ou modernização de serviços.
- Indique, de forma descritiva e sem criar uma lista de recomendações, qual parece
  ser o principal estágio seguinte de desenvolvimento do município.
- Dê prioridade aos padrões recorrentes nas quatro análises, e não a um problema
  isolado de uma única dimensão.

REGRAS OBRIGATÓRIAS:
- A Análise Geral deve ser uma SÍNTESE TRANSVERSAL, não uma quinta análise independente.
- Não contradiga nenhuma das quatro análises finais.
- Quando houver aparente tensão entre dimensões, use formulação que preserve as duas
  situações em vez de escolher arbitrariamente uma delas.
- Não introduza deficiência, programa, obra, projeto ou solução que não apareça nas
  análises finais.
- Não repita as sugestões de melhoria em formato de lista.
- Não mencionar níveis, notas, metodologia, pesquisa web, fontes, documentos de
  referência ou inteligência artificial.
- Não citar, mencionar ou usar como evidência o programa "Cidades do Futuro", mesmo que ele apareça nas análises das dimensões ou em fatos externos.
- Não usar travessão (—) em nenhuma frase do relatório. Prefira ponto, vírgula, dois-pontos ou parênteses, conforme o sentido. O hífen comum continua permitido quando fizer parte da grafia correta de uma palavra ou sigla.
- Não usar números, índices ou valores.
- Considerar o porte apenas para calibrar a complexidade da leitura; não citar o
  porte no texto final.
- Escreva para gestores municipais e leitores que NÃO são especialistas no tema.
- Linguagem institucional, simples, direta e natural.
- Prefira palavras de uso comum. Quando um termo técnico for indispensável, explique-o no próprio contexto de forma breve.
- Evite substantivos abstratos e construções burocráticas quando houver uma forma mais simples de dizer a mesma coisa.
- Prefira sujeito + verbo + complemento e frases curtas ou médias.
- Troque formulações como "sob a ótica institucional", "encontra suporte", "restringe a consolidação" e "demanda fortalecimento" por construções diretas como "na gestão municipal", "é apoiada por", "dificulta" e "precisa ser ampliado", quando o sentido for equivalente.
- Evitar frases genéricas que poderiam ser aplicadas a qualquer município.
- Priorizar exemplos concretos já presentes nos textos.
- Não exagerar aspectos positivos ou negativos.
- Não inventar relações de causa e efeito.
- Não usar o termo "incipiente".

EVITE EXPRESSÕES TÍPICAS DE TEXTOS GERADOS POR IA, como:
"pilares robustos", "ambiente fértil", "potencial significativo",
"caminho promissor", "desafios significativos", "oportunidades estratégicas",
"ativo valioso", "efervescência", "impulsionar a inovação",
"elevar a eficiência", "fortalecer a visão de futuro",
"representa uma oportunidade", "demonstra sólido compromisso",
"se destaca por", "é fundamental para", "desempenha papel fundamental".

Também evite iniciar repetidamente com:
"O município demonstra...", "O município apresenta...", "Destaca-se...",
"Observa-se que..." ou "Percebe-se que...".

Escreva como um diagnóstico institucional real, não como texto promocional
ou acadêmico.
"""

TEMPLATE_ANALISE_DIMENSAO = """\
VOCÊ É UM ANALISTA SÊNIOR EM POLÍTICAS PÚBLICAS, TRANSFORMAÇÃO DIGITAL
E PLANEJAMENTO URBANO MUNICIPAL NO CONTEXTO BRASILEIRO.

{instrucao_base}

MUNICÍPIO: {municipio}
PORTE POPULACIONAL: {porte}
DIMENSÃO ANALISADA: {dimensao}

INDICADORES DA DIMENSÃO:
{indicadores_txt}

DADOS CONTEXTUAIS DA DIMENSÃO
{dados_contextuais_txt}

TAREFA:
Produza DOIS OU TRÊS PARÁGRAFOS de análise institucional sobre a dimensão
{dimensao}, seguidos de EXATAMENTE QUATRO sugestões de melhoria. Use o terceiro
parágrafo somente quando ele acrescentar uma frente analítica realmente distinta.

ESTRUTURA OBRIGATÓRIA DA ANÁLISE:
- Primeiro parágrafo: descreva a situação atual e os principais resultados
  favoráveis efetivamente sustentados pelos indicadores. Relacione evidências
  complementares quando isso ajudar a formar uma leitura coerente da dimensão.
- Segundo parágrafo: apresente os principais desafios, contrastes e lacunas e
  interprete o que eles significam para a gestão municipal. Não apenas repita
  os indicadores já mencionados no primeiro parágrafo.
- Terceiro parágrafo (OPCIONAL): use apenas quando houver uma terceira frente
  relevante que mereça desenvolvimento próprio — por exemplo, integração entre
  sistemas/áreas, capacidade institucional ou um contraste importante que ficaria
  comprimido nos dois primeiros. Não crie um terceiro parágrafo apenas para alongar.
- Cada parágrafo deve ser desenvolvido o suficiente para formar uma análise,
  preferencialmente com 3 a 6 frases curtas ou médias.
- Quando houver resultados contrastantes, explique claramente a diferença.
- Evite transformar a análise em uma enumeração de indicadores.
- Os parágrafos devem ter funções diferentes e complementares.
- A análise deve ter no mínimo 2 e no máximo 3 parágrafos.

SUGESTÕES DE MELHORIA:
- Produza exatamente 4 sugestões.
- Cada sugestão deve responder a uma lacuna ou necessidade identificada nos
  INDICADORES DA DIMENSÃO.
- As quatro sugestões devem ser distintas entre si e evitar reformulações da
  mesma recomendação.
- Priorize ações concretas e executáveis pela gestão municipal.
- Quando os indicadores permitirem, varie o tipo de ação entre gestão e
  planejamento, infraestrutura ou tecnologia, integração de processos/dados e
  melhoria do serviço oferecido à população.
- Não crie uma sugestão apenas para completar a quantidade. Toda sugestão deve
  ter justificativa clara nos indicadores fornecidos.
- Os DADOS CONTEXTUAIS podem ajudar a caracterizar a análise, mas não devem ser
  a única origem de uma sugestão.

BASE DA PRIMEIRA GERAÇÃO — REGRA OBRIGATÓRIA:
- Produza o rascunho usando somente os indicadores, os dados contextuais e a base conceitual fornecida.
- NÃO use pesquisa web nesta etapa e NÃO suponha obras, programas ou mudanças recentes que não estejam nos dados fornecidos.

REGRAS ABSOLUTAS:
- NÃO encerrar o texto antes de concluir TODAS as partes
- NÃO usar números, índices ou valores
- NÃO mencionar inteligência artificial
- NÃO mencionar a base conceitual ou documentos de referência
- NÃO citar, mencionar, recomendar ou usar como evidência o programa "Cidades do Futuro", mesmo que esse nome apareça nos dados contextuais. Ignore essa referência para fins do relatório.
- NÃO usar travessão (—) no texto da análise nem nas sugestões. Reescreva usando ponto, vírgula, dois-pontos ou parênteses. O hífen comum é permitido quando fizer parte da grafia correta.
- Escreva para gestores municipais e leitores que NÃO são especialistas no tema
- Linguagem acessível, institucional e clara
- Evitar linguagem excessivamente técnica, abstrata, acadêmica ou burocrática
- Preferir palavras comuns e frases mais diretas e naturais
- Quando um termo técnico ou uma sigla forem necessários, explique o significado de forma breve na primeira menção
- Não simplifique a ponto de perder precisão: preserve o conteúdo técnico, mas explique-o em linguagem comum
- Escrever como um diagnóstico institucional real,
  e não como texto acadêmico ou promocional

Regras obrigatórias:

Utilize linguagem simples e direta, preservando a precisão técnica necessária.
Escreva como se o leitor fosse um gestor municipal que conhece a realidade local, mas não é especialista em tecnologia, urbanismo, saneamento ou gestão de dados.
Priorize frases de tamanho curto ou médio e uma ideia principal por frase.
Prefira verbos diretos a expressões abstratas ou burocráticas.
Evite construções como "sob a ótica de", "no que se refere a", "nesse sentido", "encontra suporte em", "restringe a consolidação de" e "demanda o fortalecimento de" quando uma formulação mais simples transmitir o mesmo sentido.
Descreva os resultados encontrados antes de fazer interpretações.
Evite elogios excessivos ao município.
Evite transformar todo resultado positivo em uma "potencialidade" ou todo resultado negativo em uma "oportunidade".
Não utilize linguagem de consultoria, marketing ou textos promocionais.
Evite conclusões genéricas que não estejam diretamente relacionadas aos indicadores analisados.
Não exagere a importância dos resultados.
Quando houver aspectos positivos e negativos, apresente-os de maneira equilibrada e factual.
As recomendações devem ser práticas e compatíveis com os problemas identificados.

EVITE EXPRESSÕES TÍPICAS DE TEXTOS GERADOS POR IA, como:

"pilares robustos"
"ambiente fértil"
"potencial significativo"
"caminho promissor"
"desafios significativos"
"oportunidades estratégicas"
"ativo valioso"
"efervescência"
"impulsionar a inovação"
"elevar a eficiência"
"fortalecer a visão de futuro"
"representa uma oportunidade"
"demonstra sólido compromisso"
"se destaca por"
"é fundamental para"
"desempenha papel fundamental"

Também evite iniciar repetidamente os parágrafos com construções como:

"O município demonstra..."
"O município apresenta..."
"Viçosa demonstra..."
"Destaca-se..."
"Observa-se que..."
"Percebe-se que..."

Varie a construção das frases de forma natural.

PREFIRA formulações mais concretas.

Pense sempre no seguinte teste de leitura: uma pessoa da prefeitura que não trabalha diretamente com o tema deve entender a frase na primeira leitura. Se houver duas formas igualmente corretas, escolha a mais simples.

Em vez de:
"O município demonstra um sólido compromisso com a transformação digital."

Escreva:
"O município já oferece parte dos serviços públicos pela internet e utiliza sistemas digitais em algumas áreas da administração."

Em vez de:
"Esse cenário representa uma oportunidade estratégica para ampliar a inovação."

Escreva:
"Esses serviços ainda podem ser ampliados e integrados."

Em vez de:
"A presença das universidades cria um ambiente fértil para a inovação."

Escreva:
"A presença das universidades facilita a aproximação entre a prefeitura, pesquisadores e empresas locais."

Não invente benefícios, causas ou relações que não estejam sustentadas pelos indicadores fornecidos.
"""




TEMPLATE_REVISAO_DIMENSAO = """\
VOCÊ É UM REVISOR SÊNIOR DE DIAGNÓSTICOS MUNICIPAIS.

MUNICÍPIO: {municipio}
PORTE POPULACIONAL: {porte}
DIMENSÃO: {dimensao}

RASCUNHO GERADO SEM PESQUISA WEB:
ANÁLISE:
{rascunho_analise}

SUGESTÕES:
{rascunho_sugestoes_txt}

INDICADORES DA PLANILHA:
{indicadores_txt}

DADOS CONTEXTUAIS DA DIMENSÃO:
{dados_contextuais_txt}

VALIDAÇÃO WEB POSTERIOR AO RASCUNHO:
{validacao_web_txt}

TAREFA:
Revise o rascunho SOMENTE quando a validação web trouxer:
- uma atualização confiável que qualifique a interpretação ou torne alguma sugestão
  desatualizada; OU
- um ENRIQUECIMENTO NOMINAL confiável de uma entidade que já aparecia genericamente
  no próprio rascunho.

O enriquecimento nominal serve apenas para tornar o texto mais concreto.
Ele não deve criar um novo argumento ou alterar a avaliação do município por si só.

FORMATO DE SAÍDA OBRIGATÓRIO:
- paragrafo_1: primeiro parágrafo completo da análise revisada;
- paragrafo_2: segundo parágrafo completo da análise revisada;
- paragrafo_3: terceiro parágrafo completo, apenas se necessário; caso contrário, null;
- sugestoes: EXATAMENTE quatro sugestões de melhoria.

IMPORTANTE:
- Nunca una parágrafos diferentes em um único campo.
- Não devolva quebras de parágrafo dentro de paragrafo_1, paragrafo_2 ou paragrafo_3.
- Cada campo deve conter apenas o texto daquele parágrafo.
- Preserve 2 parágrafos quando isso for suficiente; use no máximo 3.

REGRAS:
- Os indicadores da planilha continuam sendo a fonte principal do diagnóstico e da maturidade.
- A web não recalcula e não substitui indicadores.
- Se um indicador mostrar deficiência e a web mostrar ação em andamento, mantenha a deficiência como situação medida e acrescente apenas que existe iniciativa em curso, se isso for relevante.
- Diferencie rigorosamente: planejado; licitado/contratado; em implantação/construção; concluído/em operação.
- Não trate ação em andamento como resultado alcançado.
- Se uma sugestão disser para implantar algo que a web comprova já estar em execução, adapte-a para concluir, ampliar, integrar, monitorar ou avaliar, conforme o caso.
- Nenhuma sugestão pode nascer somente da web; ela deve continuar ligada a uma lacuna dos indicadores.
- Use no máximo CINCO fatos externos em toda a análise, correspondentes aos pontos realmente validados.
- Não use números, índices ou valores no texto final.
- Não mencione pesquisa web, fontes, documentos de referência ou inteligência artificial.
- Remova qualquer menção ao programa "Cidades do Futuro". Ele não deve aparecer na análise nem nas sugestões e não pode ser usado como evidência, exemplo, marco institucional ou justificativa.
- Remova todos os travessões (—) da análise e das sugestões. Substitua-os por ponto, vírgula, dois-pontos ou parênteses, escolhendo a opção mais natural para a frase. Não substitua hifens ortográficos legítimos.
- Preserve linguagem simples, institucional, direta e natural, adequada a gestores municipais não especialistas.
- A revisão também deve simplificar frases excessivamente técnicas, abstratas ou burocráticas, mesmo quando o conteúdo factual não precisar mudar.
- Prefira palavras comuns e verbos diretos; explique termos técnicos e siglas na primeira menção quando forem indispensáveis.
- Não retire precisão técnica nem informações relevantes apenas para encurtar o texto.
- Se a validação não exigir mudança, mantenha o conteúdo do rascunho.
- Quando houver ENRIQUECIMENTO NOMINAL confiável, substitua a referência genérica
  pelo nome oficial confirmado, preservando o sentido original da frase.
- Se houver sigla oficial confirmada e útil, escreva o nome por extenso na primeira
  menção e a sigla entre parênteses.
- Não introduza uma entidade nova que não corresponda a algo já citado no rascunho.
- Enriquecimento nominal não conta como um novo fato analítico: ele apenas especifica
  o nome de algo que já estava presente no texto.
"""

prompt_geral = ChatPromptTemplate.from_template(TEMPLATE_ANALISE_GERAL)
prompt_dimensao = ChatPromptTemplate.from_template(TEMPLATE_ANALISE_DIMENSAO)
prompt_revisao_dimensao = ChatPromptTemplate.from_template(TEMPLATE_REVISAO_DIMENSAO)

chain_geral = prompt_geral | llm_estruturado_geral
chain_dimensao = prompt_dimensao | llm_estruturado_dimensao
chain_revisao_dimensao = prompt_revisao_dimensao | llm_estruturado_revisao_dimensao


In [12]:
# =====================================================
# CHAMADAS AO MODELO — GERAÇÃO -> VALIDAÇÃO WEB -> REVISÃO
# =====================================================
def _formatar_indicadores(indicadores: dict) -> str:
    return "\n".join(f"- {k}: {v}" for k, v in indicadores.items())


def _formatar_chunks(chunks: list) -> str:
    """Formata chunks antigos e granulares sem depender de um esquema único."""
    blocos = []
    for c in chunks:
        topico = (
            c.get("topico")
            or c.get("titulo")
            or c.get("recomendacao_id")
            or c.get("chunk_id")
            or "Trecho da Carta"
        )
        texto = str(c.get("texto", "")).strip()
        if texto:
            blocos.append(f"[{topico}] {texto}")
    return "\n\n".join(blocos)


def _formatar_dados_contextuais(dados_contextuais: dict) -> str:
    """Formata dados contextuais sem tratá-los como indicadores."""
    if not dados_contextuais:
        return "Nenhum dado contextual disponível para este recorte."
    return "\n".join(f"- {k}: {v}" for k, v in dados_contextuais.items())


def _extrair_texto_resposta(resposta) -> str:
    """Extrai texto de AIMessage, inclusive em respostas com grounding."""
    texto = getattr(resposta, "text", None)
    if isinstance(texto, str) and texto.strip():
        return texto.strip()

    conteudo = getattr(resposta, "content", "")
    if isinstance(conteudo, str):
        return conteudo.strip()

    partes = []
    if isinstance(conteudo, list):
        for bloco in conteudo:
            if isinstance(bloco, str):
                partes.append(bloco)
            elif isinstance(bloco, dict) and bloco.get("type") == "text":
                partes.append(str(bloco.get("text", "")))
    return "\n".join(p for p in partes if p.strip()).strip()


def _normalizar_codigo_ibge(codigo_ibge):
    """Normaliza o código IBGE vindo da planilha sem inventar valor."""
    if codigo_ibge is None or pd.isna(codigo_ibge):
        return None
    try:
        if isinstance(codigo_ibge, (int, float)):
            return str(int(codigo_ibge))
    except Exception:
        pass
    texto = str(codigo_ibge).strip()
    if texto.endswith('.0') and texto[:-2].isdigit():
        texto = texto[:-2]
    return texto or None


def _identificacao_municipio(municipio, estado=None, codigo_ibge=None):
    codigo = _normalizar_codigo_ibge(codigo_ibge)
    uf_txt = str(estado).strip() if estado is not None and not pd.isna(estado) else "UF não informada"
    codigo_txt = codigo if codigo else "não informado"
    return uf_txt, codigo_txt


def _espera_backoff(numero_falha: int) -> int:
    """Backoff 5s, 15s, 45s... entre tentativas de grounding."""
    return 5 * (3 ** numero_falha)


def _validacao_sem_atualizacao(texto: str, tipo="dimensao") -> bool:
    """True somente quando a resposta inteira indica ausência de atualização.

    Isso evita descartar atualizações válidas quando apenas parte dos pontos
    checados trouxer novidade.
    """
    texto_norm = re.sub(r"\s+", " ", (texto or "").strip().lower())
    marcadores_exatos = {
        "pesquisa web sem atualização relevante para esta dimensão.",
        "pesquisa web sem atualizacao relevante para esta dimensao.",
        "pesquisa web sem atualização relevante para a análise geral.",
        "pesquisa web sem atualizacao relevante para a analise geral.",
        "pesquisa web sem contexto adicional confiável.",
        "pesquisa web sem contexto adicional confiavel.",
    }
    return (not texto_norm) or texto_norm in marcadores_exatos

def _invocar_grounding_com_backoff(prompt_pesquisa: str, rotulo: str, tentativas=2) -> str:
    """Executa Claude Web Search e espera real antes de repetir após falha.

    Com tentativas=2: 1 chamada inicial + 1 nova tentativa após 5s.
    Se tentativas for aumentado para 3: a segunda espera será 15s.
    """
    ultimo_erro = None
    for tentativa in range(tentativas):
        try:
            resposta = llm_pesquisa_web.invoke(prompt_pesquisa)
            texto = _extrair_texto_resposta(resposta)
            if texto:
                return texto
            raise ValueError("Resposta de grounding vazia.")
        except Exception as e:
            ultimo_erro = e
            if tentativa < tentativas - 1:
                espera = _espera_backoff(tentativa)
                print(f"⚠️  Falha em {rotulo}. Nova tentativa em {espera}s. ({e})")
                time.sleep(espera)
            else:
                print(f"⚠️  {rotulo} indisponível após {tentativas} tentativa(s). ({e})")
    return ""




def pesquisar_validacao_dimensao(
    dimensao,
    municipio,
    indicadores_dimensao,
    rascunho_dimensao,
    estado=None,
    codigo_ibge=None,
    tentativas=2,
):
    """Valida até CINCO pontos prioritários da dimensão após o rascunho.

    Mantém uma única chamada de Web Search por dimensão, com:
    - até cinco consultas específicas para validação;
    - no máximo uma consulta por ponto;
    - menos de cinco pontos quando não houver temas suficientes sujeitos a atualização.

    Além disso, faz enriquecimento nominal complementar quando houver entidades
    genéricas claramente identificáveis no próprio rascunho.
    """
    uf_txt, codigo_txt = _identificacao_municipio(municipio, estado, codigo_ibge)
    indicadores_txt = _formatar_indicadores(indicadores_dimensao)
    sugestoes_txt = "\n".join(f"- {s}" for s in rascunho_dimensao["sugestoes"])

    prompt_pesquisa = f"""
Valide de forma PONTUAL o rascunho abaixo da dimensão "{dimensao}" de um
diagnóstico municipal. A pesquisa ocorre DEPOIS da geração do texto.

Faça UMA ÚNICA ETAPA DE GROUNDING.

IDENTIFICAÇÃO TERRITORIAL OBRIGATÓRIA:
- Município: {municipio}
- UF: {uf_txt}
- Código IBGE: {codigo_txt}

INDICADORES DA PLANILHA — FONTE PRINCIPAL:
{indicadores_txt}

RASCUNHO DA ANÁLISE:
{rascunho_dimensao['analise']}

SUGESTÕES DO RASCUNHO:
{sugestoes_txt}

============================================================
PARTE A — VALIDAÇÃO PONTUAL
============================================================

MANTENHA A MESMA LÓGICA DE VALIDAÇÃO:

Escolha ATÉ CINCO pontos prioritários e DISTINTOS do rascunho cuja situação
possa ter mudado desde a coleta dos dados. Não force cinco pontos: selecione apenas
os que realmente sejam relevantes e sujeitos a atualização. Se houver somente um,
valide apenas um; se houver dois, três ou quatro, limite-se a eles.

PRIORIZE:
- deficiência que possa ter recebido obra, programa ou implantação recente;
- sugestão que possa já estar em execução;
- fato institucional sujeito a mudança recente;
- serviço ou sistema cuja situação atual possa alterar a interpretação;
- projeto, equipamento ou infraestrutura que possa estar planejado, contratado,
  em implantação ou já concluído.

NÃO escolha pontos que tratem essencialmente do mesmo assunto.
Exemplo: "tratamento de esgoto" e "construção da ETE" contam como um único tema.

REGRAS DA VALIDAÇÃO:
- Não faça busca genérica sobre toda a dimensão.
- Faça no máximo UMA consulta específica por ponto escolhido.
- Não recalcule nem substitua indicadores da planilha.
- Não declare que a planilha está errada por diferença de data/metodologia.
- Confirme que o fato pertence especificamente a {municipio}/{uf_txt};
  descarte município homônimo ou localização duvidosa.
- Priorize fontes oficiais e institucionais.
- Diferencie rigorosamente: planejado; licitado/contratado; em implantação/
  construção; concluído/em operação.
- Não trate anúncio ou obra em andamento como resultado já alcançado.
- Só considere uma atualização quando ela tiver impacto real na interpretação
  ou em uma das sugestões.
- Não desperdice consulta tentando apenas confirmar um fato estável já registrado
  na planilha.
- NÃO selecione, pesquise, valide, cite ou use como atualização o programa "Cidades do Futuro". Se ele aparecer nos resultados de busca, ignore-o completamente.
- O programa "Cidades do Futuro" também NÃO pode ser usado como enriquecimento nominal, exemplo de parceria, marco institucional ou justificativa para sugestão.

FORMATO DA PARTE A:
Para cada ponto com atualização útil, use:

PONTO 1
- ponto checado:
- fato atual encontrado:
- estágio da ação:
- fonte/instituição e data/ano, quando disponível:
- impacto na análise:
- impacto em uma sugestão, se houver:

PONTO 2
- ponto checado:
- fato atual encontrado:
- estágio da ação:
- fonte/instituição e data/ano, quando disponível:
- impacto na análise:
- impacto em uma sugestão, se houver:

PONTO 3
- ponto checado:
- fato atual encontrado:
- estágio da ação:
- fonte/instituição e data/ano, quando disponível:
- impacto na análise:
- impacto em uma sugestão, se houver:

PONTO 4
- ponto checado:
- fato atual encontrado:
- estágio da ação:
- fonte/instituição e data/ano, quando disponível:
- impacto na análise:
- impacto em uma sugestão, se houver:

PONTO 5
- ponto checado:
- fato atual encontrado:
- estágio da ação:
- fonte/instituição e data/ano, quando disponível:
- impacto na análise:
- impacto em uma sugestão, se houver:

Retorne somente os pontos que trouxerem atualização relevante.

============================================================
PARTE B — ENRIQUECIMENTO NOMINAL
============================================================

Depois da validação, procure referências GENÉRICAS já presentes no rascunho
que possam ser substituídas por nomes oficiais confirmados, por exemplo:
- "universidade federal" ou "universidade pública";
- "parque tecnológico";
- "hospital";
- "estação de tratamento";
- "conselho", "órgão", "secretaria" ou "autarquia";
- "programa municipal";
- "equipamento público";
- "sistema", "aplicativo" ou "plataforma";
- instituição de pesquisa, ensino ou inovação.

REGRAS DO ENRIQUECIMENTO NOMINAL:
- O enriquecimento NÃO substitui nem ocupa um dos pontos da Parte A.
- Só enriqueça uma entidade que JÁ esteja mencionada no rascunho.
- Não introduza instituição, programa ou equipamento novo apenas porque apareceu
  na pesquisa.
- Confirme o nome oficial em fonte oficial ou institucional sempre que possível.
- Confirme que a entidade pertence ou se relaciona inequivocamente a
  {municipio}/{uf_txt}.
- Aproveite preferencialmente as mesmas consultas feitas na Parte A.
- Evite buscas adicionais quando o nome não for importante para tornar o texto
  mais concreto.
- Não invente nomes nem complete siglas por inferência.
- Retorne no máximo 5 enriquecimentos nominais nesta dimensão.

FORMATO DA PARTE B:

ENRIQUECIMENTO NOMINAL 1
- referência genérica do rascunho:
- nome oficial confirmado:
- sigla, se houver:
- fonte/instituição de confirmação:

(repita apenas quando houver outros nomes realmente confiáveis)

============================================================
REGRA FINAL
============================================================

Se houver atualização em apenas um ponto, retorne esse ponto normalmente.
Se não houver atualização relevante nos pontos, MAS houver enriquecimento nominal
confiável, retorne somente o enriquecimento nominal.

Somente se não houver NENHUMA atualização confiável E NENHUM enriquecimento
nominal confiável, responda EXATAMENTE:
"Pesquisa web sem atualização relevante para esta dimensão."

IMPORTANTE:
Se apenas parte dos pontos trouxer novidade, NÃO use a frase
"Pesquisa web sem atualização relevante para esta dimensão."; retorne somente
os pontos que trouxeram atualização, junto com os enriquecimentos nominais que houver.
"""

    texto = _invocar_grounding_com_backoff(
        prompt_pesquisa,
        rotulo=f"validação web pós-geração da dimensão '{dimensao}'",
        tentativas=tentativas,
    )
    return texto or "Pesquisa web sem atualização relevante para esta dimensão."


def _formatar_dimensoes_validadas(dimensoes: dict) -> str:
    """Formata as quatro análises FINAIS que servirão de base à síntese geral."""
    nomes = {
        "economica": "Dimensão Econômica",
        "sociocultural": "Dimensão Sociocultural",
        "meio_ambiente": "Dimensão Meio Ambiente",
        "capacidades_institucionais": "Dimensão Capacidades Institucionais",
    }

    blocos = []
    for chave, titulo in nomes.items():
        bloco = dimensoes.get(chave)
        if not bloco or not bloco.get("analise"):
            raise ValueError(
                f"Análise final ausente para '{titulo}'. "
                "A Análise Geral só pode ser gerada após as quatro dimensões."
            )
        blocos.append(f"{titulo}:\n{bloco['analise']}")

    return "\n\n".join(blocos)


def gerar_analise_geral(
    municipio,
    porte,
    dimensoes_validadas,
    tentativas=2,
):
    """Gera a Análise Geral no FINAL como síntese das dimensões já validadas."""
    dimensoes_validadas_txt = _formatar_dimensoes_validadas(dimensoes_validadas)

    ultimo_erro = None
    for _ in range(tentativas):
        try:
            resultado = chain_geral.invoke({
                "municipio": municipio,
                "porte": porte,
                "dimensoes_validadas_txt": dimensoes_validadas_txt,
            })

            if not resultado.analise_geral or len(resultado.analise_geral.strip()) < 80:
                raise ValueError("Análise Geral final vazia ou curta demais.")

            paragrafos = _separar_paragrafos(resultado.analise_geral)
            if len(paragrafos) != 2:
                raise ValueError(
                    f"Análise Geral retornou {len(paragrafos)} parágrafo(s) "
                    "(esperado: 2)."
                )

            _validar_regras_editoriais(paragrafos, "Análise Geral")
            return "\n\n".join(paragrafos)

        except Exception as e:
            ultimo_erro = e
            print(
                "⚠️  Falha ao sintetizar a Análise Geral a partir das dimensões "
                f"validadas; tentando novamente... ({e})"
            )

    raise RuntimeError(
        "Não foi possível gerar a Análise Geral final após "
        f"{tentativas} tentativas: {ultimo_erro}"
    )

def _separar_paragrafos(texto: str) -> list:
    """Separa parágrafos reais, tolerando espaços nas linhas em branco."""
    return [p.strip() for p in re.split(r"\n\s*\n", texto.strip()) if p.strip()]


def _validar_regras_editoriais(textos, rotulo="texto"):
    """Impõe regras editoriais que não podem depender apenas do prompt."""
    if isinstance(textos, str):
        textos = [textos]
    combinado = "\n".join(str(t or "") for t in textos)
    normalizado = unicodedata.normalize("NFKD", combinado)
    normalizado = "".join(c for c in normalizado if not unicodedata.combining(c)).lower()

    if "cidades do futuro" in normalizado:
        raise ValueError(
            f"{rotulo} contém referência proibida ao programa 'Cidades do Futuro'."
        )
    if "—" in combinado:
        raise ValueError(
            f"{rotulo} contém travessão (—), proibido pelas regras editoriais."
        )
    return True


def gerar_analise_dimensao(
    dimensao,
    municipio,
    porte,
    indicadores_dimensao,
    dados_contextuais_dimensao=None,
    linha_planilha=None,
    tentativas=2,
):
    """Gera rascunho da dimensão SEM pesquisa web."""
    if not indicadores_dimensao:
        raise ValueError(f"A dimensão '{dimensao}' não possui indicadores associados.")

    chunks_selecionados = selecionar_chunks_dimensao(
        dimensao, indicadores_dimensao, linha_planilha
    )
    # Compatibilidade entre os dois esquemas de chunks:
    # - índice antigo: chave única "dimensao";
    # - índice novo: lista "dimensoes", pois um trecho pode ser transversal.
    #
    # Apenas os chunks antigos explicitamente marcados como "Geral" são
    # tratados como gerais. Um chunk novo, sem a chave "dimensao", não gera
    # KeyError e é considerado conteúdo setorial recuperado para esta análise.
    chunks_setoriais = [
        c for c in chunks_selecionados
        if c.get("dimensao") != "Geral"
    ]

    if chunks_selecionados:
        instrucao_base = INSTRUCAO_COM_BASE.format(
            base_conceitual=_formatar_chunks(chunks_selecionados)
        )
    else:
        instrucao_base = INSTRUCAO_SEM_BASE

    if not chunks_setoriais:
        print(
            f"⚠️  Dimensão '{dimensao}': nenhum chunk setorial específico na Carta "
            f"(usando apenas chunks gerais e os indicadores como base)."
        )

    indicadores_txt = _formatar_indicadores(indicadores_dimensao)
    dados_contextuais_txt = _formatar_dados_contextuais(dados_contextuais_dimensao or {})

    ultimo_erro = None

    for _ in range(tentativas):
        try:
            resultado = chain_dimensao.invoke({
                "municipio": municipio,
                "porte": porte,
                "dimensao": dimensao,
                "indicadores_txt": indicadores_txt,
                "instrucao_base": instrucao_base,
                "dados_contextuais_txt": dados_contextuais_txt,
            })

            if len(resultado.sugestoes) != 4:
                raise ValueError(
                    f"Modelo retornou {len(resultado.sugestoes)} sugestões (esperado: 4)."
                )

            paragrafo_1 = (resultado.paragrafo_1 or "").strip()
            paragrafo_2 = (resultado.paragrafo_2 or "").strip()
            paragrafo_3 = (getattr(resultado, "paragrafo_3", None) or "").strip()

            if len(paragrafo_1) < 40:
                raise ValueError("Primeiro parágrafo do rascunho vazio ou curto demais.")

            if len(paragrafo_2) < 40:
                raise ValueError("Segundo parágrafo do rascunho vazio ou curto demais.")

            paragrafos = [paragrafo_1, paragrafo_2]
            if paragrafo_3:
                if len(paragrafo_3) < 40:
                    raise ValueError("Terceiro parágrafo do rascunho curto demais.")
                paragrafos.append(paragrafo_3)

            _validar_regras_editoriais(
                paragrafos + list(resultado.sugestoes),
                f"Rascunho da dimensão {dimensao}",
            )

            return {
                "analise": "\n\n".join(paragrafos),
                "sugestoes": resultado.sugestoes,
            }

        except Exception as e:
            ultimo_erro = e
            print(
                f"⚠️  Falha ao gerar rascunho de '{dimensao}', "
                f"tentando novamente... ({e})"
            )

    raise RuntimeError(
        f"Não foi possível gerar a análise de '{dimensao}' "
        f"após {tentativas} tentativas: {ultimo_erro}"
    )


def revisar_analise_dimensao(
    dimensao,
    municipio,
    porte,
    indicadores_dimensao,
    rascunho_dimensao,
    dados_contextuais_dimensao=None,
    validacao_web=None,
    tentativas=2,
):
    """Revisa análise/sugestões somente se a validação pós-geração trouxer atualização."""
    if _validacao_sem_atualizacao(validacao_web):
        return rascunho_dimensao

    sugestoes_txt = "\n".join(f"- {s}" for s in rascunho_dimensao["sugestoes"])
    ultimo_erro = None
    for _ in range(tentativas):
        try:
            resultado = chain_revisao_dimensao.invoke({
                "municipio": municipio,
                "porte": porte,
                "dimensao": dimensao,
                "rascunho_analise": rascunho_dimensao["analise"],
                "rascunho_sugestoes_txt": sugestoes_txt,
                "indicadores_txt": _formatar_indicadores(indicadores_dimensao),
                "dados_contextuais_txt": _formatar_dados_contextuais(
                    dados_contextuais_dimensao or {}
                ),
                "validacao_web_txt": validacao_web,
            })
            if len(resultado.sugestoes) != 4:
                raise ValueError(
                    f"Revisão retornou {len(resultado.sugestoes)} sugestões (esperado: 4)."
                )

            paragrafo_1 = (resultado.paragrafo_1 or "").strip()
            paragrafo_2 = (resultado.paragrafo_2 or "").strip()
            paragrafo_3 = (getattr(resultado, "paragrafo_3", None) or "").strip()

            if len(paragrafo_1) < 20:
                raise ValueError("Primeiro parágrafo da revisão vazio ou curto demais.")
            if len(paragrafo_2) < 20:
                raise ValueError("Segundo parágrafo da revisão vazio ou curto demais.")

            paragrafos = [paragrafo_1, paragrafo_2]
            if paragrafo_3:
                if len(paragrafo_3) < 20:
                    raise ValueError("Terceiro parágrafo da revisão curto demais.")
                paragrafos.append(paragrafo_3)

            _validar_regras_editoriais(
                paragrafos + list(resultado.sugestoes),
                f"Revisão da dimensão {dimensao}",
            )

            return {
                "analise": "\n\n".join(paragrafos),
                "sugestoes": resultado.sugestoes,
            }
        except Exception as e:
            ultimo_erro = e
            print(f"⚠️  Falha ao revisar '{dimensao}', tentando novamente... ({e})")

    print(f"⚠️  Revisão de '{dimensao}' falhou; mantendo rascunho original. ({ultimo_erro})")
    return rascunho_dimensao


In [13]:
# =====================================================
# GERAÇÃO DO RELATÓRIO WORD (adaptado à saída estruturada)
# =====================================================
def gerar_relatorio_word(municipio, relatorio: dict):
    doc = Document()
    doc.add_heading(f"Relatório Institucional – {municipio}", level=1)

    doc.add_heading("Análise Geral", level=2)
    for paragrafo in _separar_paragrafos(relatorio["analise_geral"]):
        doc.add_paragraph(paragrafo)

    nomes_exibicao = {
        "economica": "Dimensão Econômica",
        "sociocultural": "Dimensão Sociocultural",
        "meio_ambiente": "Dimensão Meio Ambiente",
        "capacidades_institucionais": "Dimensão Capacidades Institucionais",
    }

    for chave, titulo in nomes_exibicao.items():
        bloco = relatorio["dimensoes"][chave]
        doc.add_heading(titulo, level=2)
        for paragrafo in _separar_paragrafos(bloco["analise"]):
            doc.add_paragraph(paragrafo)
        doc.add_paragraph("Sugestões de melhoria:")
        for sugestao in bloco["sugestoes"]:
            doc.add_paragraph(sugestao, style="List Bullet")

    nome_arquivo = f"{BASE_PATH}/Relatorio_{municipio.replace(' ', '_')}.docx"
    doc.save(nome_arquivo)
    print(f"📄 Relatório gerado: {nome_arquivo}")


In [14]:
# =====================================================
# EXECUÇÃO PRINCIPAL
# =====================================================
DIMENSAO_PARA_CHAVE = {
    "Econômica": "economica",
    "Sociocultural": "sociocultural",
    "Meio Ambiente": "meio_ambiente",
    "Capacidades Institucionais": "capacidades_institucionais",
}


def main():
    print(f"\n📚 Modo RAG da Carta: {MODO_RAG_CARTA}")
    if MODO_RAG_CARTA == "completa":
        print(f"📚 Corpus granular: {len(DOCUMENTOS_CARTA_COMPLETA)} documentos.")

    # --- Planilha ---
    try:
        df = pd.read_excel(ARQUIVO_PLANILHA)
    except FileNotFoundError:
        raise FileNotFoundError(f"Planilha de indicadores não encontrada: {ARQUIVO_PLANILHA}")

    df.columns = (
        df.columns.str.strip().str.lower().str.normalize("NFKD")
        .str.encode("ascii", errors="ignore").str.decode("utf-8")
    )

    municipios = df[COLUNA_MUNICIPIO].unique().tolist()
    print("\nMunicípios disponíveis:")
    for i, m in enumerate(municipios, 1):
        print(f"{i} - {m}")

    escolha = int(input("\nDigite o número do município: "))
    municipio = municipios[escolha - 1]

    dados = df[df[COLUNA_MUNICIPIO] == municipio].iloc[0]
    populacao = int(dados[COLUNA_POPULACAO])
    porte = classificar_porte(populacao)
    estado = dados.get("estado", None)
    codigo_ibge = _normalizar_codigo_ibge(dados.get("cod municipio", None))
    print(f"\n🔹 Porte identificado: {porte}")
    if codigo_ibge:
        print(f"🔹 Código IBGE usado para validação territorial: {codigo_ibge}")

    colunas_ignoradas = [COLUNA_MUNICIPIO, COLUNA_POPULACAO, "cod municipio", "estado", "avaliada"]
    candidatos_indicadores = dados.drop(
        labels=[c for c in colunas_ignoradas if c in dados.index]
    ).to_dict()

    # --- Classifica cada coluna candidata por dimensão/tópico ---
    # (colunas de metadado/agregado em COLUNAS_NAO_INDICADORES são
    # descartadas aqui mesmo, sem entrar em INDICADORES_NAO_CLASSIFICADOS)
    indicadores_por_dimensao = {d: {} for d in DIMENSOES}
    for nome, valor in candidatos_indicadores.items():
        classificacao = classificar_indicador(nome)
        if classificacao:
            indicadores_por_dimensao[classificacao["dimensao"]][nome] = valor

    # --- Dados contextuais (cadastrais/socioeconômicos e institucionais) ---
    # Usados apenas para complementar a Análise Geral e a análise da
    # dimensão correspondente — nunca como indicador, nunca para nível de
    # maturidade/priorização e nunca como origem direta de sugestões.
    dados_contextuais_por_dimensao = {
        dimensao: obter_dados_contextuais(dimensao, dados) for dimensao in DIMENSOES
    }

    for dimensao in DIMENSOES:
        if not indicadores_por_dimensao[dimensao]:
            print(f"⚠️  Dimensão '{dimensao}' ficou sem indicadores classificados.")

    if INDICADORES_NAO_CLASSIFICADOS:
        print(f"⚠️  Indicadores não classificados (revisar MAPA_INDICADORES): "
              f"{sorted(set(INDICADORES_NAO_CLASSIFICADOS))}")

    # --- Fluxo de geração e validação ---
    # 1) gera cada dimensão SEM web;
    # 2) valida até CINCO pontos prioritários em uma única etapa de grounding e faz enriquecimento nominal;
    # 3) revisa a dimensão quando houver atualização relevante;
    # 4) somente depois das quatro dimensões finais, gera a Análise Geral
    #    como síntese transversal — sem nova pesquisa web.
    #
    # Resultado: 4 chamadas grounded por município (uma por dimensão),
    # cada uma mantendo a lógica de até 5 pontos prioritários + enriquecimento nominal complementar.
    relatorio = {"analise_geral": "", "dimensoes": {}}

    for indice_dimensao, (dimensao, chave) in enumerate(DIMENSAO_PARA_CHAVE.items()):
        print(f"\n🔹 Gerando rascunho da dimensão {dimensao} sem pesquisa web...")

        rascunho_dimensao = gerar_analise_dimensao(
            dimensao=dimensao,
            municipio=municipio,
            porte=porte,
            indicadores_dimensao=indicadores_por_dimensao[dimensao],
            dados_contextuais_dimensao=dados_contextuais_por_dimensao[dimensao],
            linha_planilha=dados,
        )

        print(
            f"🔎 Validando até 5 pontos prioritários de {dimensao} "
            "+ enriquecimento nominal em uma única etapa de grounding..."
        )

        validacao_web_dimensao = pesquisar_validacao_dimensao(
            dimensao=dimensao,
            municipio=municipio,
            indicadores_dimensao=indicadores_por_dimensao[dimensao],
            rascunho_dimensao=rascunho_dimensao,
            estado=estado,
            codigo_ibge=codigo_ibge,
        )

        print(f"🔹 Validação web pós-geração — {dimensao}:")
        print(validacao_web_dimensao)

        relatorio["dimensoes"][chave] = revisar_analise_dimensao(
            dimensao=dimensao,
            municipio=municipio,
            porte=porte,
            indicadores_dimensao=indicadores_por_dimensao[dimensao],
            rascunho_dimensao=rascunho_dimensao,
            dados_contextuais_dimensao=dados_contextuais_por_dimensao[dimensao],
            validacao_web=validacao_web_dimensao,
        )

        print(f"✅ Dimensão {dimensao} finalizada e validada.")

        if indice_dimensao < len(DIMENSAO_PARA_CHAVE) - 1:
            time.sleep(PAUSA_ENTRE_PESQUISAS_WEB)

    # A Análise Geral só é criada depois das quatro dimensões finais.
    print(
        "\n🔹 As quatro dimensões estão finalizadas. "
        "Gerando a Análise Geral como síntese das dimensões já validadas..."
    )

    relatorio["analise_geral"] = gerar_analise_geral(
        municipio=municipio,
        porte=porte,
        dimensoes_validadas=relatorio["dimensoes"],
    )

    print("\n" + "=" * 72)
    print("ANÁLISE GERAL FINAL — SÍNTESE DAS 4 DIMENSÕES VALIDADAS")
    print("=" * 72)
    print(relatorio["analise_geral"])
    print("=" * 72 + "\n")

    print("\n" + json.dumps(relatorio, ensure_ascii=False, indent=2))

    gerar_relatorio_word(municipio, relatorio)
    print("✅ Processo finalizado com sucesso.")


main()



📚 Modo RAG da Carta: completa
📚 Corpus granular: 185 documentos.

Municípios disponíveis:
1 - Abadia dos Dourados
2 - Abaeté
3 - Abre Campo
4 - Acaiaca
5 - Açucena
6 - Água Boa
7 - Água Comprida
8 - Aguanil
9 - Águas Formosas
10 - Águas Vermelhas
11 - Aimorés
12 - Aiuruoca
13 - Alagoa
14 - Albertina
15 - Além Paraíba
16 - Alfenas
17 - Alfredo Vasconcelos
18 - Almenara
19 - Alpercata
20 - Alpinópolis
21 - Alterosa
22 - Alto Caparaó
23 - Alto Jequitibá
24 - Alto Rio Doce
25 - Alvarenga
26 - Alvinópolis
27 - Alvorada de Minas
28 - Amparo do Serra
29 - Andradas
30 - Andrelândia
31 - Angelândia
32 - Antônio Carlos
33 - Antônio Dias
34 - Antônio Prado de Minas
35 - Araçaí
36 - Aracitaba
37 - Araçuaí
38 - Araguari
39 - Arantina
40 - Araponga
41 - Araporã
42 - Arapuá
43 - Araújos
44 - Araxá
45 - Arceburgo
46 - Arcos
47 - Areado
48 - Argirita
49 - Aricanduva
50 - Arinos
51 - Astolfo Dutra
52 - Ataléia
53 - Augusto de Lima
54 - Baependi
55 - Baldim
56 - Bambuí
57 - Bandeira
58 - Bandeira do Sul